# CREMA-D + EmoT — Colab definitivo da dissertação

Execução integral visual-only, resumível no Google Drive, com SQLite, holdout por ator, validação cruzada de cinco folds por ator, fairness, XAI, Captum, evidências, logs, prints e pacote final. Os notebooks 06 e 07 permanecem preservados; este notebook não escreve no Overleaf.


## Fluxo detalhado do experimento

```mermaid
flowchart TD
    A[Colab Pro com GPU] --> A1[Montar Google Drive]
    A1 --> A2[Criar raw/cache/outputs no Drive]
    A2 --> B[Instalar dependências]
    B --> C[Baixar e cachear metadados oficiais CREMA-D]
    C --> D[Manifesto experimental]
    D --> D1[Label original do arquivo]
    D --> D2[Race, Ethnicity, Sex, Age]
    D --> D3[Votos faciais e agreement]
    D --> E[Estatísticas descritivas]
    E --> F[Baixar vídeos FLV]
    F --> G[Amostrar 100 frames por video]
    G --> H[MediaPipe Face Mesh]
    H --> I[Landmarks normalizados]
    I --> J[Controle de qualidade por grupo]
    J --> K[Tensor X: video x tempo x features]
    K --> L[Derivadas temporais: pos, vel, acc, jerk]
    L --> M[Split por ator sem vazamento]
    M --> N[FlatMLP baseline]
    M --> O[TemporalCNN1D baseline]
    M --> P[EmoT principal]
    M --> Q[EmoT com reweighting]
    N --> R[Avaliação preditiva]
    O --> R
    P --> R
    Q --> R
    R --> S[Fairness preditivo por Race]
    S --> S1[Worst-group accuracy]
    S --> S2[Equal opportunity]
    S --> S3[Equalized odds]
    S --> S4[Calibration/ECE]
    P --> T[XAI temporal e espacial]
    T --> T1[Attention Rollout]
    T --> T2[Gradient x Input]
    T --> T3[Integrated Gradients]
    T --> T4[Oclusao por regioes faciais]
    T --> T5[Deletion Test]
    T --> U[Fairness explicativo]
    U --> U1[ADI]
    U --> U2[Temporal Bias Drift]
    U --> U3[Deletion AUC por grupo]
    U --> U4[Correlacao XAI-Fairness]
    U --> V[Robustez: agreement >= 0.70]
    V --> W[Tabelas, figuras e modelos]
    W --> X[Zip final no Drive e download local]
```


## 1. Instalação e verificação do ambiente

Esta célula prepara o Colab. O treinamento usa GPU via PyTorch; a extracao de landmarks
com MediaPipe roda principalmente em CPU e costuma ser a etapa mais longa.


In [ ]:

%pip install -q --upgrade mediapipe opencv-python-headless seaborn scikit-learn tqdm requests captum


## 2. Imports, semente e configuração

`FULL_RUN=True` faz o notebook usar a base inteira. Para testar rapidamente o fluxo, mude para
`False`; o experimento final deve permanecer com `True`.

Os atributos sensíveis são usados apenas para auditoria. Eles não entram como entrada do modelo.

No Colab, esta célula também monta o Google Drive e cria a estrutura:
`raw/`, `cache/` e `outputs/`. O objetivo é persistir os artefatos fora da VM temporária
do Colab.


## Como executar e recuperar

1. Selecione **Ambiente de execução > Alterar tipo > GPU (A100)**.
2. Use **Executar tudo**. Autorize a montagem do Google Drive.
3. Após desconexão ou erro, abra o mesmo notebook e use **Executar tudo** novamente. O ledger `state/experiment_state.sqlite` escolhe a execução incompleta; downloads, shards, épocas e XAI já persistidos são reaproveitados.
4. Para uma execução nova, defina `FORCE_NEW_RUN=True`. Para refazer só uma etapa, adicione seu identificador a `FORCE_STAGES`.

Cada execução fica isolada em `cremad_fairness_xai/runs/<run_id>/`; `raw/`, `cache/` e `state/` são duráveis e compartilhados. Nenhuma célula escreve no Overleaf.


In [ ]:
# Módulo operacional embutido: também é testado em scripts/colab_resilience.py
exec(compile('"""Durable recovery primitives for the CREMA-D Colab experiment.\n\nThe module intentionally uses SQLite only for operational state and evidence\nmetadata. Large arrays, model checkpoints, figures, and reports remain files\non Google Drive and are referenced by path, size, and SHA-256 when practical.\n\nThe code is standard-library first so the same implementation can be tested\nlocally and embedded verbatim in a generated Colab notebook.\n"""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport hashlib\nimport json\nimport os\nimport re\nimport sqlite3\nimport sys\nimport threading\nimport time\nimport traceback as traceback_module\nimport uuid\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable, Iterator, Mapping, Optional\n\n\nSCHEMA_VERSION = 1\n\n\ndef utc_now() -> str:\n    """Return a sortable UTC timestamp without requiring third-party modules."""\n\n    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())\n\n\ndef json_dumps(value: Any) -> str:\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)\n\n\ndef safe_stage_name(value: str) -> str:\n    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")\n    return value or "stage"\n\n\ndef sha256_file(path: os.PathLike[str] | str, chunk_size: int = 1024 * 1024) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        while True:\n            chunk = handle.read(chunk_size)\n            if not chunk:\n                break\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef file_is_valid(path: os.PathLike[str] | str, min_bytes: int = 1) -> bool:\n    candidate = Path(path)\n    return candidate.is_file() and candidate.stat().st_size >= min_bytes\n\n\ndef _temporary_sibling(path: Path) -> Path:\n    return path.with_name(f"{path.name}.tmp-{os.getpid()}-{uuid.uuid4().hex}")\n\n\ndef atomic_write_bytes(path: os.PathLike[str] | str, data: bytes) -> Path:\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = _temporary_sibling(target)\n    try:\n        with temporary.open("wb") as handle:\n            handle.write(data)\n            handle.flush()\n            os.fsync(handle.fileno())\n        os.replace(temporary, target)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    return target\n\n\ndef atomic_write_text(\n    path: os.PathLike[str] | str, text: str, encoding: str = "utf-8"\n) -> Path:\n    return atomic_write_bytes(path, text.encode(encoding))\n\n\ndef atomic_write_json(path: os.PathLike[str] | str, value: Any) -> Path:\n    return atomic_write_text(path, json.dumps(value, ensure_ascii=False, indent=2))\n\n\ndef atomic_npz_save(path: os.PathLike[str] | str, **arrays: Any) -> Path:\n    """Save a compressed NumPy archive through an atomic sibling rename."""\n\n    import numpy as np\n\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(\n        f"{target.stem}.tmp-{os.getpid()}-{uuid.uuid4().hex}.npz"\n    )\n    try:\n        np.savez_compressed(temporary, **arrays)\n        os.replace(temporary, target)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    return target\n\n\ndef atomic_torch_save(payload: Any, path: os.PathLike[str] | str) -> Path:\n    """Save a torch payload atomically without importing torch at module import."""\n\n    import torch\n\n    target = Path(path)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = _temporary_sibling(target)\n    try:\n        torch.save(payload, temporary)\n        os.replace(temporary, target)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    return target\n\n\nclass SQLiteLedger:\n    """Small transactional ledger designed for a Google Drive-backed Colab run."""\n\n    def __init__(self, path: os.PathLike[str] | str, busy_timeout_ms: int = 30_000):\n        self.path = Path(path)\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.busy_timeout_ms = int(busy_timeout_ms)\n        self._initialize()\n\n    def _connect(self) -> sqlite3.Connection:\n        connection = sqlite3.connect(\n            self.path,\n            timeout=max(1.0, self.busy_timeout_ms / 1000),\n        )\n        connection.row_factory = sqlite3.Row\n        connection.execute(f"PRAGMA busy_timeout={self.busy_timeout_ms}")\n        connection.execute("PRAGMA synchronous=FULL")\n        connection.execute("PRAGMA foreign_keys=ON")\n        return connection\n\n    @contextlib.contextmanager\n    def _transaction(self) -> Iterator[sqlite3.Connection]:\n        connection: Optional[sqlite3.Connection] = None\n        last_error: Optional[sqlite3.OperationalError] = None\n        for attempt in range(6):\n            candidate = self._connect()\n            try:\n                candidate.execute("BEGIN IMMEDIATE")\n                connection = candidate\n                break\n            except sqlite3.OperationalError as exc:\n                candidate.close()\n                last_error = exc\n                if "locked" not in str(exc).lower() or attempt == 5:\n                    raise\n                time.sleep(0.2 * (2**attempt))\n        if connection is None:\n            if last_error is not None:\n                raise last_error\n            raise RuntimeError("Could not open SQLite transaction")\n        try:\n            yield connection\n            connection.commit()\n        except BaseException:\n            connection.rollback()\n            raise\n        finally:\n            connection.close()\n\n    def _initialize(self) -> None:\n        schema = """\n        CREATE TABLE IF NOT EXISTS schema_meta (\n            key TEXT PRIMARY KEY,\n            value TEXT NOT NULL\n        );\n\n        CREATE TABLE IF NOT EXISTS runs (\n            run_id TEXT PRIMARY KEY,\n            experiment_name TEXT NOT NULL,\n            config_hash TEXT,\n            config_json TEXT,\n            status TEXT NOT NULL,\n            started_at TEXT NOT NULL,\n            updated_at TEXT NOT NULL,\n            completed_at TEXT,\n            error TEXT\n        );\n\n        CREATE TABLE IF NOT EXISTS stages (\n            run_id TEXT NOT NULL,\n            stage_id TEXT NOT NULL,\n            status TEXT NOT NULL,\n            attempts INTEGER NOT NULL DEFAULT 0,\n            started_at TEXT,\n            completed_at TEXT,\n            duration_sec REAL,\n            error_type TEXT,\n            error_message TEXT,\n            traceback TEXT,\n            required_artifacts_json TEXT,\n            metadata_json TEXT,\n            PRIMARY KEY (run_id, stage_id),\n            FOREIGN KEY (run_id) REFERENCES runs(run_id) ON DELETE CASCADE\n        );\n\n        CREATE TABLE IF NOT EXISTS events (\n            event_id INTEGER PRIMARY KEY AUTOINCREMENT,\n            run_id TEXT NOT NULL,\n            timestamp TEXT NOT NULL,\n            stage_id TEXT,\n            level TEXT NOT NULL,\n            message TEXT NOT NULL,\n            payload_json TEXT,\n            FOREIGN KEY (run_id) REFERENCES runs(run_id) ON DELETE CASCADE\n        );\n\n        CREATE TABLE IF NOT EXISTS items (\n            run_id TEXT NOT NULL,\n            domain TEXT NOT NULL,\n            item_key TEXT NOT NULL,\n            status TEXT NOT NULL,\n            attempts INTEGER NOT NULL DEFAULT 1,\n            path TEXT,\n            bytes INTEGER,\n            sha256 TEXT,\n            error TEXT,\n            metadata_json TEXT,\n            updated_at TEXT NOT NULL,\n            PRIMARY KEY (run_id, domain, item_key),\n            FOREIGN KEY (run_id) REFERENCES runs(run_id) ON DELETE CASCADE\n        );\n\n        CREATE TABLE IF NOT EXISTS training_epochs (\n            run_id TEXT NOT NULL,\n            model_name TEXT NOT NULL,\n            epoch INTEGER NOT NULL,\n            metrics_json TEXT NOT NULL,\n            checkpoint_path TEXT,\n            created_at TEXT NOT NULL,\n            PRIMARY KEY (run_id, model_name, epoch),\n            FOREIGN KEY (run_id) REFERENCES runs(run_id) ON DELETE CASCADE\n        );\n\n        CREATE TABLE IF NOT EXISTS artifacts (\n            run_id TEXT NOT NULL,\n            stage_id TEXT NOT NULL,\n            path TEXT NOT NULL,\n            kind TEXT,\n            bytes INTEGER NOT NULL,\n            sha256 TEXT,\n            metadata_json TEXT,\n            created_at TEXT NOT NULL,\n            PRIMARY KEY (run_id, stage_id, path),\n            FOREIGN KEY (run_id) REFERENCES runs(run_id) ON DELETE CASCADE\n        );\n\n        CREATE INDEX IF NOT EXISTS idx_runs_experiment_status\n            ON runs(experiment_name, status, updated_at);\n        CREATE INDEX IF NOT EXISTS idx_events_run_stage\n            ON events(run_id, stage_id, timestamp);\n        CREATE INDEX IF NOT EXISTS idx_items_run_domain_status\n            ON items(run_id, domain, status);\n        """\n        with contextlib.closing(self._connect()) as connection:\n            # DELETE is more conservative than WAL on a mounted Google Drive\n            # filesystem. Set it once during initialization to avoid requiring\n            # an exclusive lock for every heartbeat connection.\n            connection.execute("PRAGMA journal_mode=DELETE")\n            connection.executescript(schema)\n            connection.execute(\n                "INSERT OR REPLACE INTO schema_meta(key, value) VALUES (?, ?)",\n                ("schema_version", str(SCHEMA_VERSION)),\n            )\n            connection.commit()\n\n    def close(self) -> None:\n        """Connections are short-lived; the method exists for a stable test API."""\n\n    @staticmethod\n    def _row_to_dict(row: Optional[sqlite3.Row]) -> Optional[dict[str, Any]]:\n        return dict(row) if row is not None else None\n\n    def start_run(\n        self,\n        run_id: str,\n        experiment_name: str,\n        config_hash: str = "",\n        config: Optional[Mapping[str, Any]] = None,\n    ) -> str:\n        now = utc_now()\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                INSERT INTO runs(\n                    run_id, experiment_name, config_hash, config_json,\n                    status, started_at, updated_at\n                ) VALUES (?, ?, ?, ?, \'running\', ?, ?)\n                ON CONFLICT(run_id) DO UPDATE SET\n                    experiment_name=excluded.experiment_name,\n                    config_hash=excluded.config_hash,\n                    config_json=excluded.config_json,\n                    status=\'running\',\n                    updated_at=excluded.updated_at,\n                    error=NULL\n                """,\n                (\n                    str(run_id),\n                    str(experiment_name),\n                    str(config_hash or ""),\n                    json_dumps(dict(config or {})),\n                    now,\n                    now,\n                ),\n            )\n        return str(run_id)\n\n    def start_or_resume_run(\n        self,\n        experiment_name: str,\n        preferred_run_id: Optional[str] = None,\n        config_hash: str = "",\n        config: Optional[Mapping[str, Any]] = None,\n        force_new: bool = False,\n    ) -> tuple[str, bool]:\n        if preferred_run_id and not force_new:\n            with contextlib.closing(self._connect()) as connection:\n                row = connection.execute(\n                    "SELECT * FROM runs WHERE run_id=?", (str(preferred_run_id),)\n                ).fetchone()\n            if row is not None and row["status"] != "completed":\n                self.start_run(\n                    str(preferred_run_id), experiment_name, config_hash, config\n                )\n                return str(preferred_run_id), True\n            if row is not None and row["status"] == "completed":\n                # The active pointer is authoritative. Once it is complete, a\n                # fresh Run all must not resurrect an older failed execution.\n                force_new = True\n\n        if not force_new:\n            query = """\n                SELECT * FROM runs\n                WHERE experiment_name=? AND status IN (\'running\',\'failed\',\'interrupted\')\n            """\n            params: list[Any] = [str(experiment_name)]\n            if config_hash:\n                query += " AND config_hash=?"\n                params.append(str(config_hash))\n            query += " ORDER BY updated_at DESC LIMIT 1"\n            with contextlib.closing(self._connect()) as connection:\n                row = connection.execute(query, params).fetchone()\n            if row is not None:\n                self.start_run(row["run_id"], experiment_name, config_hash, config)\n                return str(row["run_id"]), True\n\n        new_run_id = preferred_run_id or time.strftime(\n            "%Y%m%d_%H%M%S", time.localtime()\n        )\n        if force_new or preferred_run_id is None:\n            new_run_id = f"{new_run_id}_{uuid.uuid4().hex[:8]}"\n        self.start_run(new_run_id, experiment_name, config_hash, config)\n        return str(new_run_id), False\n\n    def mark_run(self, run_id: str, status: str, error: Optional[str] = None) -> None:\n        now = utc_now()\n        completed_at = now if status == "completed" else None\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                UPDATE runs\n                SET status=?, updated_at=?, completed_at=?, error=?\n                WHERE run_id=?\n                """,\n                (str(status), now, completed_at, error, str(run_id)),\n            )\n\n    def begin_stage(\n        self,\n        run_id: str,\n        stage_id: str,\n        required_artifacts: Iterable[os.PathLike[str] | str] = (),\n        metadata: Optional[Mapping[str, Any]] = None,\n    ) -> None:\n        now = utc_now()\n        required = [str(Path(path)) for path in required_artifacts]\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                INSERT INTO stages(\n                    run_id, stage_id, status, attempts, started_at,\n                    required_artifacts_json, metadata_json\n                ) VALUES (?, ?, \'running\', 1, ?, ?, ?)\n                ON CONFLICT(run_id, stage_id) DO UPDATE SET\n                    status=\'running\',\n                    attempts=stages.attempts + 1,\n                    started_at=excluded.started_at,\n                    completed_at=NULL,\n                    duration_sec=NULL,\n                    error_type=NULL,\n                    error_message=NULL,\n                    traceback=NULL,\n                    required_artifacts_json=excluded.required_artifacts_json,\n                    metadata_json=excluded.metadata_json\n                """,\n                (\n                    str(run_id),\n                    str(stage_id),\n                    now,\n                    json_dumps(required),\n                    json_dumps(dict(metadata or {})),\n                ),\n            )\n\n    def complete_stage(\n        self,\n        run_id: str,\n        stage_id: str,\n        duration_sec: Optional[float] = None,\n        metadata: Optional[Mapping[str, Any]] = None,\n    ) -> None:\n        now = utc_now()\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                UPDATE stages\n                SET status=\'completed\', completed_at=?, duration_sec=?,\n                    error_type=NULL, error_message=NULL, traceback=NULL,\n                    metadata_json=COALESCE(?, metadata_json)\n                WHERE run_id=? AND stage_id=?\n                """,\n                (\n                    now,\n                    duration_sec,\n                    json_dumps(dict(metadata)) if metadata is not None else None,\n                    str(run_id),\n                    str(stage_id),\n                ),\n            )\n            connection.execute(\n                "UPDATE runs SET status=\'running\', updated_at=?, error=NULL WHERE run_id=?",\n                (now, str(run_id)),\n            )\n\n    def fail_stage(\n        self,\n        run_id: str,\n        stage_id: str,\n        error: BaseException,\n        traceback_text: str,\n        duration_sec: Optional[float] = None,\n    ) -> None:\n        now = utc_now()\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                UPDATE stages\n                SET status=\'failed\', completed_at=?, duration_sec=?,\n                    error_type=?, error_message=?, traceback=?\n                WHERE run_id=? AND stage_id=?\n                """,\n                (\n                    now,\n                    duration_sec,\n                    type(error).__name__,\n                    str(error)[:4000],\n                    traceback_text,\n                    str(run_id),\n                    str(stage_id),\n                ),\n            )\n            connection.execute(\n                "UPDATE runs SET status=\'failed\', updated_at=?, error=? WHERE run_id=?",\n                (now, f"{stage_id}: {type(error).__name__}: {error}"[:4000], str(run_id)),\n            )\n\n    def get_stage(self, run_id: str, stage_id: str) -> Optional[dict[str, Any]]:\n        with contextlib.closing(self._connect()) as connection:\n            row = connection.execute(\n                "SELECT * FROM stages WHERE run_id=? AND stage_id=?",\n                (str(run_id), str(stage_id)),\n            ).fetchone()\n        return self._row_to_dict(row)\n\n    def stage_is_reusable(\n        self,\n        run_id: str,\n        stage_id: str,\n        required_artifacts: Iterable[os.PathLike[str] | str] = (),\n        min_bytes: int = 1,\n    ) -> bool:\n        row = self.get_stage(run_id, stage_id)\n        if row is None or row["status"] != "completed":\n            return False\n        return all(file_is_valid(path, min_bytes=min_bytes) for path in required_artifacts)\n\n    def reset_stage(self, run_id: str, stage_id: str) -> None:\n        """Reset ledger state only. Artifact deletion is deliberately not implicit."""\n\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                UPDATE stages\n                SET status=\'pending\', started_at=NULL, completed_at=NULL,\n                    duration_sec=NULL, error_type=NULL, error_message=NULL, traceback=NULL\n                WHERE run_id=? AND stage_id=?\n                """,\n                (str(run_id), str(stage_id)),\n            )\n\n    def event(\n        self,\n        run_id: str,\n        level: str,\n        message: str,\n        stage_id: Optional[str] = None,\n        payload: Optional[Mapping[str, Any]] = None,\n    ) -> None:\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                INSERT INTO events(run_id, timestamp, stage_id, level, message, payload_json)\n                VALUES (?, ?, ?, ?, ?, ?)\n                """,\n                (\n                    str(run_id),\n                    utc_now(),\n                    stage_id,\n                    str(level),\n                    str(message),\n                    json_dumps(dict(payload or {})),\n                ),\n            )\n\n    def upsert_item(\n        self,\n        run_id: str,\n        domain: str,\n        item_key: str,\n        status: str,\n        path: Optional[os.PathLike[str] | str] = None,\n        byte_count: Optional[int] = None,\n        error: Optional[str] = None,\n        metadata: Optional[Mapping[str, Any]] = None,\n        compute_hash: bool = True,\n    ) -> None:\n        path_text = str(Path(path)) if path is not None else None\n        digest = None\n        if path is not None and file_is_valid(path):\n            byte_count = int(Path(path).stat().st_size)\n            if compute_hash:\n                digest = sha256_file(path)\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                INSERT INTO items(\n                    run_id, domain, item_key, status, attempts, path, bytes,\n                    sha256, error, metadata_json, updated_at\n                ) VALUES (?, ?, ?, ?, 1, ?, ?, ?, ?, ?, ?)\n                ON CONFLICT(run_id, domain, item_key) DO UPDATE SET\n                    status=excluded.status,\n                    attempts=items.attempts + 1,\n                    path=excluded.path,\n                    bytes=excluded.bytes,\n                    sha256=COALESCE(excluded.sha256, items.sha256),\n                    error=excluded.error,\n                    metadata_json=excluded.metadata_json,\n                    updated_at=excluded.updated_at\n                """,\n                (\n                    str(run_id),\n                    str(domain),\n                    str(item_key),\n                    str(status),\n                    path_text,\n                    byte_count,\n                    digest,\n                    error,\n                    json_dumps(dict(metadata or {})),\n                    utc_now(),\n                ),\n            )\n\n    def get_item(\n        self, run_id: str, domain: str, item_key: str\n    ) -> Optional[dict[str, Any]]:\n        with contextlib.closing(self._connect()) as connection:\n            row = connection.execute(\n                "SELECT * FROM items WHERE run_id=? AND domain=? AND item_key=?",\n                (str(run_id), str(domain), str(item_key)),\n            ).fetchone()\n        return self._row_to_dict(row)\n\n    def record_training_epoch(\n        self,\n        run_id: str,\n        model_name: str,\n        epoch: int,\n        metrics: Mapping[str, Any],\n        checkpoint_path: Optional[os.PathLike[str] | str] = None,\n    ) -> None:\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                INSERT OR REPLACE INTO training_epochs(\n                    run_id, model_name, epoch, metrics_json, checkpoint_path, created_at\n                ) VALUES (?, ?, ?, ?, ?, ?)\n                """,\n                (\n                    str(run_id),\n                    str(model_name),\n                    int(epoch),\n                    json_dumps(dict(metrics)),\n                    str(Path(checkpoint_path)) if checkpoint_path is not None else None,\n                    utc_now(),\n                ),\n            )\n\n    def record_training_history(\n        self,\n        run_id: str,\n        model_name: str,\n        rows: Iterable[Mapping[str, Any]],\n        checkpoint_path: Optional[os.PathLike[str] | str] = None,\n    ) -> int:\n        """Backfill multiple epoch rows in one Drive-friendly transaction."""\n\n        timestamp = utc_now()\n        checkpoint_text = (\n            str(Path(checkpoint_path)) if checkpoint_path is not None else None\n        )\n        records = [\n            (\n                str(run_id),\n                str(model_name),\n                int(row["epoch"]),\n                json_dumps(dict(row)),\n                checkpoint_text,\n                timestamp,\n            )\n            for row in rows\n        ]\n        if not records:\n            return 0\n        with self._transaction() as connection:\n            connection.executemany(\n                """\n                INSERT OR REPLACE INTO training_epochs(\n                    run_id, model_name, epoch, metrics_json, checkpoint_path, created_at\n                ) VALUES (?, ?, ?, ?, ?, ?)\n                """,\n                records,\n            )\n        return len(records)\n\n    def training_history(self, run_id: str, model_name: str) -> list[dict[str, Any]]:\n        with contextlib.closing(self._connect()) as connection:\n            rows = connection.execute(\n                """\n                SELECT * FROM training_epochs\n                WHERE run_id=? AND model_name=? ORDER BY epoch\n                """,\n                (str(run_id), str(model_name)),\n            ).fetchall()\n        return [dict(row) for row in rows]\n\n    def register_artifact(\n        self,\n        run_id: str,\n        stage_id: str,\n        path: os.PathLike[str] | str,\n        kind: str = "file",\n        metadata: Optional[Mapping[str, Any]] = None,\n        compute_hash: bool = True,\n    ) -> dict[str, Any]:\n        candidate = Path(path)\n        if not file_is_valid(candidate):\n            raise FileNotFoundError(f"Artifact missing or empty: {candidate}")\n        record = {\n            "path": str(candidate),\n            "bytes": int(candidate.stat().st_size),\n            "sha256": sha256_file(candidate) if compute_hash else None,\n        }\n        with self._transaction() as connection:\n            connection.execute(\n                """\n                INSERT OR REPLACE INTO artifacts(\n                    run_id, stage_id, path, kind, bytes, sha256, metadata_json, created_at\n                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)\n                """,\n                (\n                    str(run_id),\n                    str(stage_id),\n                    record["path"],\n                    str(kind),\n                    record["bytes"],\n                    record["sha256"],\n                    json_dumps(dict(metadata or {})),\n                    utc_now(),\n                ),\n            )\n        return record\n\n    def list_artifacts(self, run_id: str) -> list[dict[str, Any]]:\n        with contextlib.closing(self._connect()) as connection:\n            rows = connection.execute(\n                "SELECT * FROM artifacts WHERE run_id=? ORDER BY stage_id, path",\n                (str(run_id),),\n            ).fetchall()\n        return [dict(row) for row in rows]\n\n    def stage_rows(self, run_id: str) -> list[dict[str, Any]]:\n        with contextlib.closing(self._connect()) as connection:\n            rows = connection.execute(\n                """\n                SELECT * FROM stages\n                WHERE run_id=?\n                ORDER BY COALESCE(started_at, \'\'), stage_id\n                """,\n                (str(run_id),),\n            ).fetchall()\n        return [dict(row) for row in rows]\n\n\nclass TeeStream:\n    def __init__(self, original: Any, log_handle: Any):\n        self.original = original\n        self.log_handle = log_handle\n\n    def write(self, text: str) -> int:\n        written = self.original.write(text)\n        self.log_handle.write(text)\n        self.log_handle.flush()\n        return written\n\n    def flush(self) -> None:\n        self.original.flush()\n        self.log_handle.flush()\n\n    def isatty(self) -> bool:\n        return bool(getattr(self.original, "isatty", lambda: False)())\n\n\nclass Heartbeat:\n    def __init__(\n        self,\n        ledger: SQLiteLedger,\n        run_id: str,\n        stage_id: str,\n        interval_seconds: float,\n        resource_probe: Optional[Callable[[], Mapping[str, Any]]] = None,\n        progress_dir: Optional[os.PathLike[str] | str] = None,\n    ):\n        self.ledger = ledger\n        self.run_id = run_id\n        self.stage_id = stage_id\n        self.interval_seconds = max(1.0, float(interval_seconds))\n        self.resource_probe = resource_probe\n        self.progress_dir = Path(progress_dir) if progress_dir is not None else None\n        self._stop = threading.Event()\n        self._thread = threading.Thread(target=self._loop, daemon=True)\n\n    def start(self) -> None:\n        self._thread.start()\n\n    def stop(self) -> None:\n        self._stop.set()\n        self._thread.join(timeout=min(5.0, self.interval_seconds))\n\n    def _loop(self) -> None:\n        while not self._stop.wait(self.interval_seconds):\n            payload: dict[str, Any] = {}\n            if self.resource_probe is not None:\n                try:\n                    payload.update(dict(self.resource_probe()))\n                except Exception as exc:  # heartbeat must never stop the experiment\n                    payload["resource_probe_error"] = str(exc)[:500]\n            self.ledger.event(\n                self.run_id,\n                level="heartbeat",\n                message=f"Stage {self.stage_id} is still running",\n                stage_id=self.stage_id,\n                payload=payload,\n            )\n            print(f"[HEARTBEAT] {self.stage_id} ainda em execução | {utc_now()}")\n            if self.progress_dir is not None:\n                save_progress_snapshot(\n                    self.ledger,\n                    self.run_id,\n                    self.progress_dir,\n                    self.stage_id,\n                    "running",\n                    keep_history=False,\n                )\n\n\ndef save_progress_snapshot(\n    ledger: SQLiteLedger,\n    run_id: str,\n    progress_dir: os.PathLike[str] | str,\n    stage_id: str,\n    status: str,\n    keep_history: bool = True,\n) -> Optional[Path]:\n    """Save a compact PNG dashboard. Failure to render never fails the run."""\n\n    try:\n        import matplotlib.pyplot as plt\n\n        rows = ledger.stage_rows(run_id)\n        progress_path = Path(progress_dir)\n        progress_path.mkdir(parents=True, exist_ok=True)\n        completed = sum(row["status"] == "completed" for row in rows)\n        failed = sum(row["status"] == "failed" for row in rows)\n        running = sum(row["status"] == "running" for row in rows)\n        labels = [row["stage_id"] for row in rows[-18:]] or [stage_id]\n        colors = {\n            "completed": "#2E8B57",\n            "failed": "#C0392B",\n            "running": "#F39C12",\n            "pending": "#95A5A6",\n        }\n        row_statuses = [row["status"] for row in rows[-18:]] or [status]\n        fig_height = max(4.5, 0.34 * len(labels) + 2.5)\n        fig, ax = plt.subplots(figsize=(12, fig_height))\n        y_positions = list(range(len(labels)))\n        ax.barh(\n            y_positions,\n            [1] * len(labels),\n            color=[colors.get(value, "#7F8C8D") for value in row_statuses],\n        )\n        ax.set_yticks(y_positions, labels)\n        ax.set_xlim(0, 1)\n        ax.set_xticks([])\n        ax.set_title(\n            f"CREMA-D pipeline | run={run_id}\\n"\n            f"último estágio={stage_id} ({status}) | concluídos={completed} "\n            f"falhos={failed} em execução={running}"\n        )\n        for index, value in enumerate(row_statuses):\n            ax.text(0.02, index, value, va="center", color="white", weight="bold")\n        fig.tight_layout()\n        timestamp = time.strftime("%Y%m%d_%H%M%S")\n        def save_figure_atomic(path: Path) -> None:\n            temporary = path.with_name(\n                f"{path.stem}.tmp-{os.getpid()}-{uuid.uuid4().hex}{path.suffix}"\n            )\n            try:\n                fig.savefig(temporary, dpi=140, bbox_inches="tight")\n                os.replace(temporary, path)\n            finally:\n                if temporary.exists():\n                    temporary.unlink()\n\n        latest = progress_path / "latest_progress.png"\n        save_figure_atomic(latest)\n        target = latest\n        if keep_history:\n            target = progress_path / (\n                f"{timestamp}_{safe_stage_name(stage_id)}_{safe_stage_name(status)}.png"\n            )\n            save_figure_atomic(target)\n        plt.close(fig)\n        return target\n    except Exception as exc:\n        try:\n            ledger.event(\n                run_id,\n                level="warning",\n                message="Could not render progress snapshot",\n                stage_id=stage_id,\n                payload={"error": str(exc)[:1000]},\n            )\n        except Exception:\n            pass\n        return None\n\n\n@contextlib.contextmanager\ndef tracked_stage(\n    ledger: SQLiteLedger,\n    run_id: str,\n    stage_id: str,\n    required_artifacts: Iterable[os.PathLike[str] | str] = (),\n    metadata: Optional[Mapping[str, Any]] = None,\n    log_dir: Optional[os.PathLike[str] | str] = None,\n    progress_dir: Optional[os.PathLike[str] | str] = None,\n    heartbeat_seconds: Optional[float] = None,\n    resource_probe: Optional[Callable[[], Mapping[str, Any]]] = None,\n) -> Iterator[None]:\n    """Record a stage transaction, tee its text log, and persist failures."""\n\n    stage_id = str(stage_id)\n    required_paths = [Path(path) for path in required_artifacts]\n    ledger.mark_run(run_id, "running")\n    ledger.begin_stage(run_id, stage_id, required_paths, metadata)\n    ledger.event(\n        run_id,\n        level="info",\n        message="Stage started",\n        stage_id=stage_id,\n        payload=dict(metadata or {}),\n    )\n    started = time.time()\n    log_handle = None\n    stdout_before, stderr_before = sys.stdout, sys.stderr\n    heartbeat = None\n    try:\n        if log_dir is not None:\n            log_path = Path(log_dir) / "stage_logs" / f"{safe_stage_name(stage_id)}.log"\n            log_path.parent.mkdir(parents=True, exist_ok=True)\n            log_handle = log_path.open("a", encoding="utf-8")\n            log_handle.write(f"\\n===== {utc_now()} | run={run_id} | start =====\\n")\n            log_handle.flush()\n            sys.stdout = TeeStream(stdout_before, log_handle)\n            sys.stderr = TeeStream(stderr_before, log_handle)\n        if heartbeat_seconds:\n            heartbeat = Heartbeat(\n                ledger,\n                run_id,\n                stage_id,\n                heartbeat_seconds,\n                resource_probe=resource_probe,\n                progress_dir=progress_dir,\n            )\n            heartbeat.start()\n        print(f"[STAGE] {stage_id}: início")\n        yield\n        if heartbeat is not None:\n            heartbeat.stop()\n            heartbeat = None\n        missing = [str(path) for path in required_paths if not file_is_valid(path)]\n        if missing:\n            raise RuntimeError(\n                "Stage finished without required artifacts: " + ", ".join(missing)\n            )\n        for path in required_paths:\n            ledger.register_artifact(run_id, stage_id, path)\n        duration = time.time() - started\n        ledger.complete_stage(run_id, stage_id, duration_sec=duration, metadata=metadata)\n        ledger.event(\n            run_id,\n            level="info",\n            message="Stage completed",\n            stage_id=stage_id,\n            payload={"duration_sec": round(duration, 3)},\n        )\n        print(f"[STAGE] {stage_id}: concluído em {duration:.1f}s")\n        if progress_dir is not None:\n            save_progress_snapshot(\n                ledger, run_id, progress_dir, stage_id, "completed"\n            )\n    except BaseException as exc:\n        if heartbeat is not None:\n            heartbeat.stop()\n            heartbeat = None\n        duration = time.time() - started\n        traceback_text = "".join(\n            traceback_module.format_exception(type(exc), exc, exc.__traceback__)\n        )\n        ledger.fail_stage(\n            run_id,\n            stage_id,\n            error=exc,\n            traceback_text=traceback_text,\n            duration_sec=duration,\n        )\n        ledger.mark_run(\n            run_id,\n            "failed",\n            error=f"{stage_id}: {type(exc).__name__}: {exc}",\n        )\n        ledger.event(\n            run_id,\n            level="error",\n            message=f"Stage failed: {type(exc).__name__}: {exc}",\n            stage_id=stage_id,\n            payload={"duration_sec": round(duration, 3)},\n        )\n        print(f"[STAGE] {stage_id}: FALHOU após {duration:.1f}s")\n        print(traceback_text)\n        if progress_dir is not None:\n            save_progress_snapshot(ledger, run_id, progress_dir, stage_id, "failed")\n        raise\n    finally:\n        if heartbeat is not None:\n            heartbeat.stop()\n        sys.stdout, sys.stderr = stdout_before, stderr_before\n        if log_handle is not None:\n            log_handle.flush()\n            log_handle.close()\n\n\ndef print_stage_dashboard(ledger: SQLiteLedger, run_id: str) -> None:\n    rows = ledger.stage_rows(run_id)\n    if not rows:\n        print("Nenhuma etapa registrada ainda.")\n        return\n    columns = ["stage_id", "status", "attempts", "duration_sec", "error_message"]\n    try:\n        import pandas as pd\n\n        display(pd.DataFrame(rows)[columns])  # type: ignore[name-defined]\n    except Exception:\n        print(" | ".join(columns))\n        for row in rows:\n            print(" | ".join(str(row.get(column, "")) for column in columns))\n', 'colab_resilience.py', 'exec'), globals())
print(f'Recovery runtime carregado | schema={SCHEMA_VERSION}')


In [ ]:
# Módulo científico testável embutido em scripts/definitive_analysis.py
exec(compile('"""Scientific analysis primitives for the definitive CREMA-D Colab.\n\nThe functions in this module are intentionally independent from Colab, CUDA,\nGoogle Drive and the training implementation.  This keeps the actor-level\nsplit contract and the quantitative post-processing directly unit-testable.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom collections.abc import Iterable, Sequence\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import (\n    accuracy_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n)\nfrom sklearn.model_selection import KFold, StratifiedKFold, train_test_split\n\n\nDEFAULT_GROUP_ATTRIBUTES = ("Race", "Ethnicity", "Sex", "age_bin")\n\n\ndef _require_columns(frame: pd.DataFrame, columns: Iterable[str]) -> None:\n    missing = sorted(set(columns).difference(frame.columns))\n    if missing:\n        raise ValueError(f"Missing required columns: {missing}")\n\n\ndef _candidate_strata(actors: pd.DataFrame, n_required: int) -> tuple[pd.Series | None, str]:\n    candidates: list[tuple[str, pd.Series]] = []\n    if {"Race", "Sex"}.issubset(actors.columns):\n        candidates.append(\n            (\n                "Race__Sex",\n                actors["Race"].fillna("Unknown").astype(str)\n                + "__"\n                + actors["Sex"].fillna("Unknown").astype(str),\n            )\n        )\n    for column in ("Race", "Sex"):\n        if column in actors.columns:\n            candidates.append((column, actors[column].fillna("Unknown").astype(str)))\n\n    for name, strata in candidates:\n        counts = strata.value_counts(dropna=False)\n        if len(counts) >= 2 and int(counts.min()) >= n_required:\n            return strata, name\n    return None, "unstratified"\n\n\ndef _split_train_validation_actors(\n    actors: pd.DataFrame,\n    validation_fraction: float,\n    seed: int,\n) -> tuple[np.ndarray, np.ndarray]:\n    if not 0 < validation_fraction < 0.5:\n        raise ValueError("validation_fraction must be between 0 and 0.5")\n    if len(actors) < 3:\n        raise ValueError("At least three training actors are required")\n\n    strata, _ = _candidate_strata(actors, n_required=2)\n    stratify = strata\n    if stratify is not None:\n        n_classes = int(stratify.nunique())\n        n_validation = int(np.ceil(len(actors) * validation_fraction))\n        n_training = len(actors) - n_validation\n        if n_validation < n_classes or n_training < n_classes:\n            stratify = None\n\n    try:\n        train_positions, validation_positions = train_test_split(\n            np.arange(len(actors)),\n            test_size=validation_fraction,\n            random_state=seed,\n            shuffle=True,\n            stratify=stratify,\n        )\n    except ValueError:\n        train_positions, validation_positions = train_test_split(\n            np.arange(len(actors)),\n            test_size=validation_fraction,\n            random_state=seed,\n            shuffle=True,\n            stratify=None,\n        )\n    return np.sort(train_positions), np.sort(validation_positions)\n\n\ndef build_actor_folds(\n    manifest: pd.DataFrame,\n    n_splits: int = 5,\n    seed: int = 42,\n    validation_fraction: float = 0.15,\n    actor_column: str = "ActorID",\n) -> tuple[list[dict[str, object]], pd.DataFrame, str]:\n    """Build outer test folds and inner validation splits without actor leakage.\n\n    The outer assignment is stratified by ``Race x Sex`` when the actor counts\n    allow it, then progressively falls back to Race, Sex and finally KFold.\n    Every sample is held out exactly once across the outer folds.\n    """\n\n    _require_columns(manifest, [actor_column])\n    if n_splits < 2:\n        raise ValueError("n_splits must be at least two")\n\n    demographic_columns = [c for c in ("Race", "Sex", "Ethnicity", "age_bin", "Age") if c in manifest.columns]\n    actor_frame = manifest[[actor_column, *demographic_columns]].copy()\n    for column in demographic_columns:\n        inconsistent = actor_frame.groupby(actor_column, dropna=False)[column].nunique(dropna=False)\n        if (inconsistent > 1).any():\n            offenders = inconsistent[inconsistent > 1].index.tolist()\n            raise ValueError(f"Demographic value changes within actor for {column}: {offenders[:5]}")\n    actors = actor_frame.drop_duplicates(actor_column).reset_index(drop=True)\n    if len(actors) < n_splits:\n        raise ValueError(f"Need at least {n_splits} actors; received {len(actors)}")\n\n    strata, strategy = _candidate_strata(actors, n_required=n_splits)\n    if strata is None:\n        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        outer_splits = splitter.split(actors)\n    else:\n        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)\n        outer_splits = splitter.split(actors, strata)\n\n    sample_actors = manifest[actor_column].to_numpy()\n    actor_test_fold: dict[object, int] = {}\n    folds: list[dict[str, object]] = []\n    for fold_number, (train_validation_positions, test_positions) in enumerate(outer_splits, start=1):\n        train_validation_actors = actors.iloc[train_validation_positions].reset_index(drop=True)\n        inner_train_positions, validation_positions = _split_train_validation_actors(\n            train_validation_actors,\n            validation_fraction=validation_fraction,\n            seed=seed + fold_number,\n        )\n\n        train_actor_values = train_validation_actors.iloc[inner_train_positions][actor_column].to_numpy()\n        validation_actor_values = train_validation_actors.iloc[validation_positions][actor_column].to_numpy()\n        test_actor_values = actors.iloc[test_positions][actor_column].to_numpy()\n        for actor in test_actor_values:\n            actor_test_fold[actor] = fold_number\n\n        folds.append(\n            {\n                "fold": fold_number,\n                "train_idx": np.flatnonzero(np.isin(sample_actors, train_actor_values)),\n                "val_idx": np.flatnonzero(np.isin(sample_actors, validation_actor_values)),\n                "test_idx": np.flatnonzero(np.isin(sample_actors, test_actor_values)),\n                "train_actors": train_actor_values.tolist(),\n                "val_actors": validation_actor_values.tolist(),\n                "test_actors": test_actor_values.tolist(),\n                "outer_stratification": strategy,\n            }\n        )\n\n    assignments = actors.copy()\n    assignments["outer_test_fold"] = assignments[actor_column].map(actor_test_fold).astype(int)\n    assignments["outer_stratification"] = strategy\n    validate_actor_folds(manifest, folds, actor_column=actor_column)\n    return folds, assignments, strategy\n\n\ndef validate_actor_folds(\n    manifest: pd.DataFrame,\n    folds: Sequence[dict[str, object]],\n    actor_column: str = "ActorID",\n) -> dict[str, object]:\n    """Validate index bounds, actor disjointness and exact outer-fold coverage."""\n\n    _require_columns(manifest, [actor_column])\n    appearances = np.zeros(len(manifest), dtype=np.int64)\n    leakage_rows: list[dict[str, object]] = []\n    for fold in folds:\n        fold_number = int(fold["fold"])\n        split_indices: dict[str, np.ndarray] = {}\n        split_actors: dict[str, set[object]] = {}\n        for role in ("train", "val", "test"):\n            index = np.asarray(fold[f"{role}_idx"], dtype=np.int64)\n            if len(index) == 0:\n                raise ValueError(f"Fold {fold_number} has an empty {role} split")\n            if index.min() < 0 or index.max() >= len(manifest):\n                raise ValueError(f"Fold {fold_number} contains an out-of-range index")\n            split_indices[role] = index\n            split_actors[role] = set(manifest.iloc[index][actor_column].tolist())\n\n        for left, right in (("train", "val"), ("train", "test"), ("val", "test")):\n            overlap = split_actors[left].intersection(split_actors[right])\n            if overlap:\n                leakage_rows.append(\n                    {\n                        "fold": fold_number,\n                        "left": left,\n                        "right": right,\n                        "actors": sorted(map(str, overlap)),\n                    }\n                )\n        appearances[split_indices["test"]] += 1\n\n    valid = not leakage_rows and bool(np.all(appearances == 1))\n    if not valid:\n        raise ValueError(\n            "Invalid actor folds: "\n            f"leakage={leakage_rows[:3]}, test appearances={np.unique(appearances, return_counts=True)}"\n        )\n    return {\n        "valid": True,\n        "n_folds": len(folds),\n        "n_samples": len(manifest),\n        "n_actors": int(manifest[actor_column].nunique()),\n        "test_appearances_min": int(appearances.min()),\n        "test_appearances_max": int(appearances.max()),\n        "actor_leakage_count": 0,\n    }\n\n\ndef _normalized_confusion(y_true: np.ndarray, y_pred: np.ndarray, labels: Sequence[int]) -> np.ndarray:\n    raw = confusion_matrix(y_true, y_pred, labels=list(labels)).astype(np.float64)\n    return np.divide(raw, raw.sum(axis=1, keepdims=True), out=np.zeros_like(raw), where=raw.sum(axis=1, keepdims=True) > 0)\n\n\ndef aggregate_cv_results(\n    records: Sequence[dict[str, object]],\n    labels: Sequence[int],\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, dict[str, np.ndarray]]]:\n    """Aggregate fold metrics and row-normalized confusion matrices by model."""\n\n    if not records:\n        raise ValueError("At least one cross-validation record is required")\n    metric_rows: list[dict[str, object]] = []\n    confusions: dict[str, dict[str, np.ndarray]] = {}\n    model_names = sorted({str(record["model"]) for record in records})\n    for record in records:\n        y_true = np.asarray(record["y_true"], dtype=np.int64)\n        y_pred = np.asarray(record["y_pred"], dtype=np.int64)\n        if y_true.shape != y_pred.shape or y_true.ndim != 1:\n            raise ValueError("y_true and y_pred must be aligned one-dimensional arrays")\n        metric_rows.append(\n            {\n                "model": str(record["model"]),\n                "fold": int(record["fold"]),\n                "n_test": int(len(y_true)),\n                "accuracy": float(accuracy_score(y_true, y_pred)),\n                "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n                "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n                "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),\n            }\n        )\n\n    fold_metrics = pd.DataFrame(metric_rows).sort_values(["model", "fold"]).reset_index(drop=True)\n    summary_rows: list[dict[str, object]] = []\n    for model in model_names:\n        selected_records = [record for record in records if str(record["model"]) == model]\n        model_metrics = fold_metrics[fold_metrics["model"] == model]\n        row: dict[str, object] = {"model": model, "n_folds": int(len(model_metrics))}\n        for metric in ("accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"):\n            row[f"{metric}_mean"] = float(model_metrics[metric].mean())\n            row[f"{metric}_std"] = float(model_metrics[metric].std(ddof=1)) if len(model_metrics) > 1 else 0.0\n            row[f"{metric}_min"] = float(model_metrics[metric].min())\n            row[f"{metric}_max"] = float(model_metrics[metric].max())\n        summary_rows.append(row)\n\n        matrices = np.stack(\n            [\n                _normalized_confusion(\n                    np.asarray(record["y_true"], dtype=np.int64),\n                    np.asarray(record["y_pred"], dtype=np.int64),\n                    labels,\n                )\n                for record in selected_records\n            ]\n        )\n        pooled_true = np.concatenate([np.asarray(record["y_true"], dtype=np.int64) for record in selected_records])\n        pooled_pred = np.concatenate([np.asarray(record["y_pred"], dtype=np.int64) for record in selected_records])\n        pooled_raw = confusion_matrix(pooled_true, pooled_pred, labels=list(labels))\n        confusions[model] = {\n            "mean": matrices.mean(axis=0),\n            "std": matrices.std(axis=0, ddof=1) if len(matrices) > 1 else np.zeros_like(matrices[0]),\n            "pooled_raw": pooled_raw,\n            "pooled_normalized": _normalized_confusion(pooled_true, pooled_pred, labels),\n        }\n\n    summary = pd.DataFrame(summary_rows).sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)\n    return fold_metrics, summary, confusions\n\n\ndef group_emotion_metrics(\n    metadata: pd.DataFrame,\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    group_attributes: Sequence[str] = DEFAULT_GROUP_ATTRIBUTES,\n    class_names: Sequence[str] | None = None,\n    min_support: int = 10,\n) -> pd.DataFrame:\n    """Return recall, precision, F1 and false-positive rate per group/emotion."""\n\n    _require_columns(metadata, group_attributes)\n    y_true = np.asarray(y_true, dtype=np.int64)\n    y_pred = np.asarray(y_pred, dtype=np.int64)\n    if len(metadata) != len(y_true) or y_true.shape != y_pred.shape:\n        raise ValueError("metadata, y_true and y_pred must be aligned")\n    if class_names is None:\n        n_classes = int(max(y_true.max(initial=0), y_pred.max(initial=0)) + 1)\n        class_names = [str(index) for index in range(n_classes)]\n\n    rows: list[dict[str, object]] = []\n    for attribute in group_attributes:\n        group_values = metadata[attribute].fillna("Unknown").astype(str).to_numpy()\n        for group in pd.unique(group_values):\n            group_mask = group_values == group\n            for class_id, emotion in enumerate(class_names):\n                true_mask = group_mask & (y_true == class_id)\n                predicted_mask = group_mask & (y_pred == class_id)\n                support_true = int(true_mask.sum())\n                support_predicted = int(predicted_mask.sum())\n                if support_true < min_support:\n                    continue\n                true_positive = int((true_mask & (y_pred == class_id)).sum())\n                false_positive = int((predicted_mask & (y_true != class_id)).sum())\n                negative_support = int((group_mask & (y_true != class_id)).sum())\n                recall = true_positive / support_true\n                precision = true_positive / support_predicted if support_predicted else np.nan\n                f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0\n                rows.append(\n                    {\n                        "protected_attribute": attribute,\n                        "group": group,\n                        "emotion_id": class_id,\n                        "emotion": str(emotion),\n                        "support_true": support_true,\n                        "support_predicted": support_predicted,\n                        "true_positive": true_positive,\n                        "false_positive": false_positive,\n                        "recall": float(recall),\n                        "precision": float(precision) if not np.isnan(precision) else np.nan,\n                        "f1": float(f1),\n                        "false_positive_rate": float(false_positive / negative_support) if negative_support else np.nan,\n                    }\n                )\n    return pd.DataFrame(rows)\n\n\ndef confusion_pairs_by_group(\n    metadata: pd.DataFrame,\n    y_true: np.ndarray,\n    y_pred: np.ndarray,\n    group_attributes: Sequence[str] = DEFAULT_GROUP_ATTRIBUTES,\n    class_names: Sequence[str] | None = None,\n    min_true_support: int = 10,\n) -> pd.DataFrame:\n    """Quantify every non-diagonal true-to-predicted confusion within groups."""\n\n    _require_columns(metadata, group_attributes)\n    y_true = np.asarray(y_true, dtype=np.int64)\n    y_pred = np.asarray(y_pred, dtype=np.int64)\n    if len(metadata) != len(y_true) or y_true.shape != y_pred.shape:\n        raise ValueError("metadata, y_true and y_pred must be aligned")\n    if class_names is None:\n        n_classes = int(max(y_true.max(initial=0), y_pred.max(initial=0)) + 1)\n        class_names = [str(index) for index in range(n_classes)]\n\n    rows: list[dict[str, object]] = []\n    for attribute in group_attributes:\n        groups = metadata[attribute].fillna("Unknown").astype(str).to_numpy()\n        for group in pd.unique(groups):\n            group_mask = groups == group\n            group_error_count = int((group_mask & (y_true != y_pred)).sum())\n            if not group_error_count:\n                continue\n            for true_id, true_emotion in enumerate(class_names):\n                true_mask = group_mask & (y_true == true_id)\n                true_support = int(true_mask.sum())\n                if true_support < min_true_support:\n                    continue\n                for predicted_id, predicted_emotion in enumerate(class_names):\n                    if true_id == predicted_id:\n                        continue\n                    count = int((true_mask & (y_pred == predicted_id)).sum())\n                    if count == 0:\n                        continue\n                    rows.append(\n                        {\n                            "protected_attribute": attribute,\n                            "group": group,\n                            "true_emotion_id": true_id,\n                            "true_emotion": str(true_emotion),\n                            "predicted_emotion_id": predicted_id,\n                            "predicted_emotion": str(predicted_emotion),\n                            "count": count,\n                            "true_emotion_support": true_support,\n                            "group_error_count": group_error_count,\n                            "error_rate_within_true": float(count / true_support),\n                            "share_of_group_errors": float(count / group_error_count),\n                        }\n                    )\n    if not rows:\n        return pd.DataFrame()\n    return pd.DataFrame(rows).sort_values(\n        ["protected_attribute", "group", "count"],\n        ascending=[True, True, False],\n    ).reset_index(drop=True)\n\n\ndef sample_distribution_tables(\n    metadata: pd.DataFrame,\n    group_attributes: Sequence[str] = DEFAULT_GROUP_ATTRIBUTES,\n    emotion_column: str = "label_name",\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Create long-form demographic counts and emotion-by-demographic counts."""\n\n    _require_columns(metadata, [emotion_column, *group_attributes])\n    distribution_rows: list[dict[str, object]] = []\n    cross_rows: list[pd.DataFrame] = []\n    for attribute in group_attributes:\n        values = metadata[attribute].fillna("Unknown").astype(str)\n        counts = values.value_counts(dropna=False)\n        for group, count in counts.items():\n            distribution_rows.append(\n                {\n                    "protected_attribute": attribute,\n                    "group": group,\n                    "n": int(count),\n                    "percent": float(100 * count / len(metadata)),\n                }\n            )\n        cross = pd.crosstab(metadata[emotion_column], values, dropna=False).stack().rename("n").reset_index()\n        cross.columns = ["emotion", "group", "n"]\n        cross.insert(0, "protected_attribute", attribute)\n        cross["percent_within_emotion"] = cross["n"] / cross.groupby("emotion")["n"].transform("sum") * 100\n        cross_rows.append(cross)\n    return pd.DataFrame(distribution_rows), pd.concat(cross_rows, ignore_index=True)\n\n\ndef balance_strategy_table(\n    metadata: pd.DataFrame,\n    label_column: str = "label_code",\n    race_column: str = "Race",\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Document class weights and label-by-race sampling weights."""\n\n    _require_columns(metadata, [label_column, race_column])\n    label_counts = metadata[label_column].fillna("Unknown").astype(str).value_counts().sort_index()\n    class_rows = pd.DataFrame({"label": label_counts.index, "n": label_counts.values})\n    raw_weights = len(metadata) / np.maximum(class_rows["n"].to_numpy(dtype=float), 1.0)\n    class_rows["class_weight_normalized"] = raw_weights / raw_weights.mean()\n    class_rows["strategy"] = "weighted_cross_entropy"\n\n    keys = (\n        metadata[label_column].fillna("Unknown").astype(str)\n        + "__"\n        + metadata[race_column].fillna("Unknown").astype(str)\n    )\n    key_counts = keys.value_counts().sort_index()\n    sampler_rows = []\n    raw_sample_weights = keys.map(lambda key: 1.0 / key_counts[key]).to_numpy(dtype=float)\n    normalized_sample_weights = raw_sample_weights / raw_sample_weights.mean()\n    for key, count in key_counts.items():\n        label, race = key.split("__", 1)\n        mask = keys == key\n        sampler_rows.append(\n            {\n                "label": label,\n                "Race": race,\n                "n": int(count),\n                "mean_sampling_weight_normalized": float(normalized_sample_weights[mask].mean()),\n                "expected_share_after_resampling": float(1 / len(key_counts)),\n                "strategy": "weighted_random_sampler_label_x_race",\n            }\n        )\n    return class_rows, pd.DataFrame(sampler_rows)\n\n\ndef safe_slug(value: object) -> str:\n    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")\n    return slug or "unknown"\n', 'definitive_analysis.py', 'exec'), globals())
print('Módulo definitivo de análise carregado.')


In [ ]:

import os, re, gc, json, time, math, random, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

FULL_RUN = True
DEFINITIVE_RUN = True
ALLOW_DEV_MODE = False
FAST_DEV_SAMPLES = 360

# Contrato científico pós-qualificação.
CV_N_SPLITS = 5
CV_VALIDATION_FRACTION = 0.15
CV_MODELS = ("FlatMLP", "TemporalCNN1D", "EmoT", "EmoT_Reweighted")
PROTECTED_ATTRIBUTES_DEFINITIVE = ("Race", "Ethnicity", "Sex", "age_bin")
CV_MIN_GROUP_N = 30
CV_MIN_EMOTION_SUPPORT = 10
TOP_CONFUSIONS_PER_GROUP = 5
CAPTUM_VERIFY_SAMPLES = 64
CAPTUM_BATCH_SIZE = 8
CAPTUM_INTERNAL_BATCH_SIZE = 64

if DEFINITIVE_RUN and not FULL_RUN and not ALLOW_DEV_MODE:
    raise RuntimeError("A execução definitiva exige FULL_RUN=True. Use ALLOW_DEV_MODE=True apenas para depuração explícita.")
T_FRAMES = 100
MIN_FACE_DETECTION_RATE = 0.60
HIGH_CONSENSUS_THRESHOLD = 0.70

# "semantic" usa regiões expressivas da face; "full" usa os 468 landmarks.
LANDMARK_SET = "semantic"
ADD_TEMPORAL_DERIVATIVES = True

# Configuração Pedro/AcT: mantemos a base CREMA-D + MediaPipe, mas aproximamos a
# camada Transformer da implementação original usada pelo Pedro.
EMOT_REFERENCE = "Pedro/AcT base adaptado para CREMA-D+MediaPipe"
EMOT_MODEL_SIZE = "base"
PEDRO_ACT_SPECS = {
    "micro": {"N_HEADS": 1, "N_LAYERS": 4, "DROPOUT": 0.30, "MLP": 256},
    "small": {"N_HEADS": 2, "N_LAYERS": 5, "DROPOUT": 0.30, "MLP": 256},
    "base": {"N_HEADS": 3, "N_LAYERS": 6, "DROPOUT": 0.30, "MLP": 256},
    "large": {"N_HEADS": 4, "N_LAYERS": 6, "DROPOUT": 0.40, "MLP": 512},
    "enormous": {"N_HEADS": 8, "N_LAYERS": 10, "DROPOUT": 0.40, "MLP": 512},
}
EMOT_SPEC = PEDRO_ACT_SPECS[EMOT_MODEL_SIZE]
N_HEADS = EMOT_SPEC["N_HEADS"]
N_LAYERS = EMOT_SPEC["N_LAYERS"]
D_MODEL = 64 * N_HEADS
DIM_FF = 4 * D_MODEL
DROPOUT = EMOT_SPEC["DROPOUT"]
MLP_HEAD_SIZE = EMOT_SPEC["MLP"]

BATCH_SIZE = 256
EPOCHS = 3000
# Pedro treina até N_EPOCHS salvando o melhor checkpoint. No Colab, manter
# early stopping evita desperdício quando a validação estabiliza; use None para
# forçar as 3000 épocas completas.
PATIENCE = 50
USE_PEDRO_LR_SCHEDULE = True
WARMUP_PERC = 0.10
LR = 1.0
WEIGHT_DECAY = 3e-5
LABEL_SMOOTHING = 0.10
GRAD_CLIP_VALUE = 0.5
GRAD_CLIP_NORM = 1.0

# XAI: None usa todo o teste; se faltar tempo/memória, use por exemplo 600.
MAX_XAI_SAMPLES = None
XAI_BATCH_SIZE = 32
DELETION_FRACTIONS = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50, 0.75, 1.00]
TOP_K_LANDMARKS = 20
MAX_IG_SAMPLES = 600
IG_STEPS = 16
REGION_OCCLUSION_BATCH_SIZE = 64

DOWNLOAD_WORKERS = 16
DOWNLOAD_TIMEOUT = 60

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "cremad_fairness_xai"

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    print("Montando Google Drive em /content/drive ...")
    drive.mount("/content/drive", force_remount=False)
    BASE_DIR = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
else:
    BASE_DIR = Path("/content/cremad_fairness_xai") if IN_COLAB else Path.cwd().parent / "data" / "cremad_fairness_xai"

# Contrato de recuperação. Para começar do zero, altere apenas
# FORCE_NEW_RUN=True. FORCE_STAGES permite invalidar seletivamente uma etapa.
EXPERIMENT_NAME = "cremad_fairness_xai_emot_definitive"
RESUME_MODE = True
FORCE_NEW_RUN = False
FORCE_STAGES = set()
HEARTBEAT_SECONDS = 120
LANDMARK_SHARD_SIZE = 50
CHECKPOINT_EVERY_EPOCHS = 1
AUTO_DOWNLOAD_FINAL_ZIP = False

RAW_DIR = BASE_DIR / "raw"
VIDEO_DIR = RAW_DIR / "videos_flv"
METADATA_DIR = RAW_DIR / "metadata"
CACHE_DIR = BASE_DIR / "cache"
STATE_DIR = BASE_DIR / "state"
RUNS_DIR = BASE_DIR / "runs"
STATE_DB = STATE_DIR / "experiment_state.sqlite"
ACTIVE_RUN_JSON = STATE_DIR / "active_run.json"

for d in [RAW_DIR, VIDEO_DIR, METADATA_DIR, CACHE_DIR, STATE_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

recovery_config = {
    "seed": SEED,
    "definitive_run": DEFINITIVE_RUN,
    "cv_n_splits": CV_N_SPLITS,
    "cv_models": list(CV_MODELS),
    "captum_verify_samples": CAPTUM_VERIFY_SAMPLES,
    "full_run": FULL_RUN,
    "fast_dev_samples": FAST_DEV_SAMPLES,
    "t_frames": T_FRAMES,
    "landmark_set": LANDMARK_SET,
    "temporal_derivatives": ADD_TEMPORAL_DERIVATIVES,
    "emot_model_size": EMOT_MODEL_SIZE,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "d_model": D_MODEL,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "label_smoothing": LABEL_SMOOTHING,
    "max_xai_samples": MAX_XAI_SAMPLES,
    "max_ig_samples": MAX_IG_SAMPLES,
    "ig_steps": IG_STEPS,
    "deletion_fractions": DELETION_FRACTIONS,
}
CONFIG_HASH = hashlib.sha256(json_dumps(recovery_config).encode("utf-8")).hexdigest()
preferred_run_id = None
if RESUME_MODE and ACTIVE_RUN_JSON.exists():
    try:
        active_state = json.loads(ACTIVE_RUN_JSON.read_text(encoding="utf-8"))
        if active_state.get("config_hash") == CONFIG_HASH:
            preferred_run_id = active_state.get("run_id")
    except Exception as exc:
        print(f"Aviso: ponteiro de execução ignorado: {exc}")

LEDGER = SQLiteLedger(STATE_DB)
RUN_ID, RUN_WAS_RESUMED = LEDGER.start_or_resume_run(
    experiment_name=EXPERIMENT_NAME,
    preferred_run_id=preferred_run_id,
    config_hash=CONFIG_HASH,
    config=recovery_config,
    force_new=FORCE_NEW_RUN or not RESUME_MODE,
)

RUN_DIR = RUNS_DIR / RUN_ID
OUTPUT_DIR = RUN_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CKPT_DIR = OUTPUT_DIR / "models"
LOG_DIR = OUTPUT_DIR / "logs"
REPORT_DIR = OUTPUT_DIR / "reports"
XAI_DIR = OUTPUT_DIR / "xai"
FAIRNESS_DIR = OUTPUT_DIR / "fairness"
ZIP_DIR = OUTPUT_DIR / "zips"
PROGRESS_DIR = OUTPUT_DIR / "progress_snapshots"

for d in [RUN_DIR, OUTPUT_DIR, FIG_DIR, TABLE_DIR, CKPT_DIR, LOG_DIR, REPORT_DIR, XAI_DIR, FAIRNESS_DIR, ZIP_DIR, PROGRESS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

atomic_write_json(
    ACTIVE_RUN_JSON,
    {"run_id": RUN_ID, "config_hash": CONFIG_HASH, "updated_at": utc_now()},
)
print(f"Run: {RUN_ID} | retomada={RUN_WAS_RESUMED} | SQLite={STATE_DB}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print(f"IN_COLAB: {IN_COLAB}")
print(f"USE_GOOGLE_DRIVE: {USE_GOOGLE_DRIVE}")
print(f"Device: {DEVICE}")
print(f"Base dir: {BASE_DIR}")
print(f"Raw videos: {VIDEO_DIR}")
print(f"Cache: {CACHE_DIR}")
print(f"Outputs: {OUTPUT_DIR}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    if "A100" not in gpu_name.upper():
        print("Aviso: o experimento roda em outras GPUs, mas a estimativa principal assume Colab Pro com A100.")
else:
    gpu_name = "CPU"

run_config = {
    "run_id": RUN_ID,
    "definitive_run": DEFINITIVE_RUN,
    "cv_n_splits": CV_N_SPLITS,
    "cv_models": list(CV_MODELS),
    "protected_attributes_definitive": list(PROTECTED_ATTRIBUTES_DEFINITIVE),
    "captum_verify_samples": CAPTUM_VERIFY_SAMPLES,
    "full_run": FULL_RUN,
    "fast_dev_samples": FAST_DEV_SAMPLES,
    "t_frames": T_FRAMES,
    "landmark_set": LANDMARK_SET,
    "add_temporal_derivatives": ADD_TEMPORAL_DERIVATIVES,
    "emot_reference": EMOT_REFERENCE,
    "emot_model_size": EMOT_MODEL_SIZE,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "d_model": D_MODEL,
    "dim_ff": DIM_FF,
    "dropout": DROPOUT,
    "mlp_head_size": MLP_HEAD_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "use_pedro_lr_schedule": USE_PEDRO_LR_SCHEDULE,
    "warmup_perc": WARMUP_PERC,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "high_consensus_threshold": HIGH_CONSENSUS_THRESHOLD,
    "max_xai_samples": MAX_XAI_SAMPLES,
    "base_dir": str(BASE_DIR),
    "device": str(DEVICE),
    "gpu_name": gpu_name,
}
run_config["config_hash"] = CONFIG_HASH
run_config["run_dir"] = str(RUN_DIR)
run_config["resumed"] = RUN_WAS_RESUMED
atomic_write_json(LOG_DIR / f"run_config_{RUN_ID}.json", run_config)
print(f"Configuração da execução salva em: {LOG_DIR / f'run_config_{RUN_ID}.json'}")

RUN_EVENTS = []

def current_gpu_stats():
    stats = {
        "cuda_available": bool(torch.cuda.is_available()),
        "gpu_name": gpu_name,
    }
    if torch.cuda.is_available():
        stats.update({
            "gpu_memory_allocated_mb": round(torch.cuda.memory_allocated() / (1024 ** 2), 2),
            "gpu_memory_reserved_mb": round(torch.cuda.memory_reserved() / (1024 ** 2), 2),
            "gpu_max_memory_allocated_mb": round(torch.cuda.max_memory_allocated() / (1024 ** 2), 2),
            "gpu_max_memory_reserved_mb": round(torch.cuda.max_memory_reserved() / (1024 ** 2), 2),
        })
        try:
            import subprocess as _subprocess
            query = "utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu"
            out = _subprocess.check_output(
                ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
                text=True,
                stderr=_subprocess.STDOUT,
            ).strip().splitlines()[0]
            util, mem_used, mem_total, power, temp = [x.strip() for x in out.split(",")]
            stats.update({
                "nvidia_gpu_utilization_percent": float(util),
                "nvidia_memory_used_mb": float(mem_used),
                "nvidia_memory_total_mb": float(mem_total),
                "nvidia_power_draw_w": float(power),
                "nvidia_temperature_c": float(temp),
            })
        except Exception as exc:
            stats["nvidia_smi_error"] = str(exc)[:200]
    return stats

def log_event(stage, status="done", start_time=None, **extra):
    now = time.time()
    row = {
        "run_id": RUN_ID,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "stage": stage,
        "status": status,
    }
    if start_time is not None:
        row["duration_sec"] = round(now - start_time, 3)
        row["duration_min"] = round((now - start_time) / 60, 3)
    row.update(current_gpu_stats())
    row.update(extra)
    RUN_EVENTS.append(row)
    try:
        LEDGER.event(RUN_ID, level="info", message=f"{stage}: {status}", stage_id=stage, payload=row)
    except Exception as exc:
        print(f"[WARN] evento SQLite não persistido: {exc}")
    log_df = pd.DataFrame(RUN_EVENTS)
    log_df.to_csv(LOG_DIR / f"resource_usage_log_{RUN_ID}.csv", index=False)
    log_df.to_csv(LOG_DIR / "resource_usage_log_latest.csv", index=False)
    msg = f"[LOG] {stage}: {status}"
    if "duration_sec" in row:
        msg += f" | {row['duration_sec']:.1f}s"
    if row.get("cuda_available"):
        msg += f" | VRAM usada={row.get('nvidia_memory_used_mb', row.get('gpu_memory_allocated_mb', 'n/a'))} MB"
    print(msg)
    return row

def show_gpu_snapshot(label):
    row = log_event(f"gpu_snapshot_{label}", status="snapshot")
    display(pd.DataFrame([row]).T.rename(columns={0: label}))
    return row

log_event("setup_colab_drive_and_dirs", status="ready")


## 3. Diagnóstico da GPU

O print da GPU ajuda a documentar o ambiente computacional usado no experimento.


In [ ]:

import subprocess

if torch.cuda.is_available():
    subprocess.run(["nvidia-smi"], check=False)
    show_gpu_snapshot("inicio")
else:
    print("Nenhuma GPU foi detectada. No Colab, ative Runtime > Change runtime type > GPU.")
    show_gpu_snapshot("sem_gpu")


## 4. Fontes oficiais e classes do CREMA-D

O label principal vem do nome do arquivo. Exemplo: `1001_IEO_HAP_HI.flv` tem emoção
original `HAP`. Os votos dos avaliadores do CREMA-D entram apenas na analise de robustez.


In [ ]:

CREMAD_RAW_BASE = "https://raw.githubusercontent.com/CheyneyComputerScience/CREMA-D/master"
CREMAD_MEDIA_BASE = "https://media.githubusercontent.com/media/CheyneyComputerScience/CREMA-D/master"

URL_DEMOGRAPHICS = f"{CREMAD_RAW_BASE}/VideoDemographics.csv"
URL_SUMMARY = f"{CREMAD_RAW_BASE}/processedResults/summaryTable.csv"
URL_TABULATED = f"{CREMAD_RAW_BASE}/processedResults/tabulatedVotes.csv"

LABEL_MAP = {
    "ANG": "angry",
    "DIS": "disgust",
    "FEA": "fear",
    "HAP": "happy",
    "NEU": "neutral",
    "SAD": "sad",
}
CLASS_CODES = list(LABEL_MAP.keys())
CLASS_NAMES = [LABEL_MAP[c] for c in CLASS_CODES]
CODE_TO_ID = {code: idx for idx, code in enumerate(CLASS_CODES)}
ID_TO_CODE = {idx: code for code, idx in CODE_TO_ID.items()}
N_CLASSES = len(CLASS_NAMES)

print("Classes usadas:")
for code, name in LABEL_MAP.items():
    print(f"  {code}: {name}")


## 5. Manifesto experimental

Esta célula junta metadados demográficos, labels originais e consenso facial. O campo
`face_agreement` vem dos avaliadores do próprio CREMA-D, não de avaliadores novos.


In [ ]:

def read_csv_cached(url, path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        print(f"Carregando metadado em cache: {path}")
        return pd.read_csv(path)
    print(f"Baixando metadado oficial: {url}")
    df = pd.read_csv(url)
    df.to_csv(path, index=False)
    print(f"Metadado salvo em: {path}")
    return df


def load_cremad_metadata():
    demographics = read_csv_cached(URL_DEMOGRAPHICS, METADATA_DIR / "VideoDemographics.csv")
    summary = read_csv_cached(URL_SUMMARY, METADATA_DIR / "summaryTable.csv").rename(columns={"FileName": "file_stem"})
    tabulated = read_csv_cached(URL_TABULATED, METADATA_DIR / "tabulatedVotes.csv")

    summary["ActorID"] = summary["file_stem"].str.extract(r"^(\d{4})").astype(int)

    # IDs 200001..299999 correspondem aos votos faciais no tabulatedVotes.
    face_votes = tabulated[(tabulated["Unnamed: 0"] >= 200000) & (tabulated["Unnamed: 0"] < 300000)].copy()
    face_votes = face_votes.rename(
        columns={
            "fileName": "file_stem",
            "agreement": "face_agreement",
            "emoVote": "face_emo_vote",
            "numResponses": "face_num_responses",
        }
    )
    face_votes = face_votes[["file_stem", "face_num_responses", "face_agreement", "face_emo_vote"]]

    meta = summary.merge(demographics, on="ActorID", how="left")
    meta = meta.merge(face_votes, on="file_stem", how="left")

    parts = meta["file_stem"].str.split("_", expand=True)
    meta["sentence_code"] = parts[1]
    meta["label_code"] = parts[2]
    meta["intensity_code"] = parts[3]
    meta["label_name"] = meta["label_code"].map(LABEL_MAP)
    meta["y"] = meta["label_code"].map(CODE_TO_ID)
    meta["video_url"] = meta["file_stem"].map(lambda s: f"{CREMAD_MEDIA_BASE}/VideoFlash/{s}.flv")
    meta["video_path"] = meta["file_stem"].map(lambda s: str(VIDEO_DIR / f"{s}.flv"))
    meta["age_bin"] = pd.cut(
        meta["Age"],
        bins=[0, 29, 39, 49, 59, 200],
        labels=["20-29", "30-39", "40-49", "50-59", "60+"],
        include_lowest=True,
    ).astype(str)

    meta = meta[meta["label_code"].isin(CLASS_CODES)].copy()
    return meta.sort_values("file_stem").reset_index(drop=True)

metadata_start = time.time()
meta_full = load_cremad_metadata()
meta_full.to_csv(TABLE_DIR / "manifest_full_cremad.csv", index=False)
if FULL_RUN:
    meta_run = meta_full.copy()
else:
    meta_run = (
        meta_full.groupby(["label_code", "Race"], group_keys=False)
        .apply(lambda x: x.sample(min(len(x), max(1, FAST_DEV_SAMPLES // 24)), random_state=SEED))
        .reset_index(drop=True)
    )

print(f"Manifesto completo: {len(meta_full):,} vídeos")
print(f"Execução atual: {len(meta_run):,} vídeos")
print(f"Atores unicos: {meta_run['ActorID'].nunique()}")
meta_run.to_csv(TABLE_DIR / "manifest_run_cremad.csv", index=False)
print(f"Manifesto completo salvo em: {TABLE_DIR / 'manifest_full_cremad.csv'}")
print(f"Manifesto desta execução salvo em: {TABLE_DIR / 'manifest_run_cremad.csv'}")
log_event(
    "metadata_manifest",
    start_time=metadata_start,
    n_full=int(len(meta_full)),
    n_run=int(len(meta_run)),
    n_actors=int(meta_run["ActorID"].nunique()),
)
display(meta_run.head())


## 5.1 Estimativa de tempo da execução completa

Esta estimativa ajuda a planejar o uso do Colab Pro. Ela não substitui os logs reais:
ao final do notebook, a seção de processamento salva os tempos medidos por etapa em
`outputs/tables/processing_time_by_stage.csv`.

O ponto crítico é que a A100 acelera treino e XAI, mas a extração de landmarks com
MediaPipe depende bastante de CPU, leitura dos vídeos e velocidade do Google Drive.


In [ ]:

estimated_sampled_frames = len(meta_run) * T_FRAMES

runtime_estimate_rows = [
    {
        "stage": "Instalação, Drive e metadados",
        "first_run_a100": "5-15 min",
        "cached_run_a100": "2-5 min",
        "main_bottleneck": "pip/Colab/Drive",
    },
    {
        "stage": "Download dos vídeos CREMA-D",
        "first_run_a100": "20-60 min",
        "cached_run_a100": "0-5 min",
        "main_bottleneck": "rede e GitHub media",
    },
    {
        "stage": "Estatísticas de vídeo e amostras visuais",
        "first_run_a100": "5-20 min",
        "cached_run_a100": "5-20 min",
        "main_bottleneck": "I/O de vídeo",
    },
    {
        "stage": "MediaPipe landmarks em 100 frames/vídeo",
        "first_run_a100": "3-7 h",
        "cached_run_a100": "0-10 min",
        "main_bottleneck": "CPU + leitura dos frames",
    },
    {
        "stage": "Treino dos modelos",
        "first_run_a100": "1.5-4 h",
        "cached_run_a100": "1.5-4 h",
        "main_bottleneck": "GPU/early stopping",
    },
    {
        "stage": "Fairness, robustez e matrizes",
        "first_run_a100": "10-30 min",
        "cached_run_a100": "10-30 min",
        "main_bottleneck": "CPU",
    },
    {
        "stage": "XAI: attention, gradientes, IG, oclusão e deletion",
        "first_run_a100": "1-3 h",
        "cached_run_a100": "1-3 h",
        "main_bottleneck": "GPU + número de amostras XAI",
    },
    {
        "stage": "Zip final e download dos artefatos",
        "first_run_a100": "10-30 min",
        "cached_run_a100": "10-30 min",
        "main_bottleneck": "tamanho do cache/outputs",
    },
]
runtime_estimate = pd.DataFrame(runtime_estimate_rows)
runtime_estimate_path = TABLE_DIR / "runtime_estimate_table.csv"
runtime_estimate.to_csv(runtime_estimate_path, index=False)

print(f"Vídeos na execução: {len(meta_run):,}")
print(f"Frames amostrados para MediaPipe: {estimated_sampled_frames:,} ({T_FRAMES} por vídeo)")
print("Estimativa A100 Colab Pro, primeira execução sem cache: 6-15 horas.")
print("Estimativa A100 com vídeos/landmarks já em cache: 3-8 horas.")
print("Se PATIENCE=None e as 3000 épocas forem forçadas, o treino pode passar de 24 horas.")
print(f"Tabela de estimativa salva em: {runtime_estimate_path}")
display(runtime_estimate)
log_event(
    "runtime_estimate",
    status="estimated",
    n_videos=int(len(meta_run)),
    sampled_frames=int(estimated_sampled_frames),
    first_run_a100_total="6-15 h",
    cached_run_a100_total="3-8 h",
)


## 6. Estatísticas descritivas

Antes de treinar, documentamos a composição da base. Isso evita discutir fairness sem
saber o tamanho dos grupos e o balanceamento das emoções.


In [ ]:

def show_counts(df, column, title):
    counts = df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="n")
    counts["percent"] = 100 * counts["n"] / counts["n"].sum()
    print(f"\n{title}")
    display(counts)
    return counts

descriptive_start = time.time()
descriptive_outputs = {
    "label_name": show_counts(meta_run, "label_name", "Distribuição por emoção"),
    "Race": show_counts(meta_run, "Race", "Distribuição por Race"),
    "Ethnicity": show_counts(meta_run, "Ethnicity", "Distribuição por Ethnicity"),
    "Sex": show_counts(meta_run, "Sex", "Distribuição por Sex"),
    "age_bin": show_counts(meta_run, "age_bin", "Distribuição por faixa etária"),
}
for name, table in descriptive_outputs.items():
    out_path = TABLE_DIR / f"descriptive_{name}.csv"
    table.to_csv(out_path, index=False)
    print(f"Tabela descritiva salva: {out_path}")

print("\nEmocao x Race")
crosstab_emotion_race = pd.crosstab(meta_run["label_name"], meta_run["Race"], margins=True)
display(crosstab_emotion_race)
crosstab_emotion_race.to_csv(TABLE_DIR / "crosstab_emotion_by_race.csv")

print("\nEmocao x Ethnicity")
crosstab_emotion_ethnicity = pd.crosstab(meta_run["label_name"], meta_run["Ethnicity"], margins=True)
display(crosstab_emotion_ethnicity)
crosstab_emotion_ethnicity.to_csv(TABLE_DIR / "crosstab_emotion_by_ethnicity.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.countplot(data=meta_run, y="label_name", order=CLASS_NAMES, ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set_title("Amostras por emoção original")
sns.countplot(data=meta_run, y="Race", order=meta_run["Race"].value_counts().index, ax=axes[0, 1], color="#F58518")
axes[0, 1].set_title("Amostras por Race")
sns.countplot(data=meta_run, y="Ethnicity", order=meta_run["Ethnicity"].value_counts().index, ax=axes[1, 0], color="#54A24B")
axes[1, 0].set_title("Amostras por Ethnicity")
sns.histplot(data=meta_run, x="face_agreement", bins=20, ax=axes[1, 1], color="#B279A2")
axes[1, 1].axvline(HIGH_CONSENSUS_THRESHOLD, color="black", linestyle="--")
axes[1, 1].set_title("Consenso facial dos avaliadores CREMA-D")
plt.tight_layout()
fig.savefig(FIG_DIR / "01_descriptive_distributions.png", bbox_inches="tight")
plt.show()
log_event("descriptive_statistics", start_time=descriptive_start, n_tables=len(descriptive_outputs), figure="01_descriptive_distributions.png")


## Escopo, composição da base e tempo estimado

O CREMA-D é áudio-visual, mas o experimento abaixo é estritamente visual-only. Esta seção materializa o escopo, as distribuições por raça/etnia, sexo, faixa etária e emoção, além das interseções e da estimativa para A100.


In [ ]:

scope_rows = [
    {
        "dataset": "CREMA-D",
        "dataset_modalities_available": "audio+video",
        "modality_used_in_this_experiment": "visual_only",
        "visual_representation": f"MediaPipe landmarks ({LANDMARK_SET}); temporal_derivatives={ADD_TEMPORAL_DERIVATIVES}",
        "audio_features_used": False,
        "unit_of_evaluation": "video",
        "identity_control": "actor-disjoint holdout and actor-disjoint five-fold CV",
        "claim_boundary": "Results cannot be compared directly with audio-only or audiovisual fusion systems.",
    }
]
modality_scope = pd.DataFrame(scope_rows)
atomic_write_text(TABLE_DIR / "modality_scope.csv", modality_scope.to_csv(index=False))
scope_text = """# Escopo experimental

- O CREMA-D disponibiliza áudio e vídeo.
- Esta execução usa exclusivamente a modalidade visual: frames de vídeo são convertidos em landmarks faciais; nenhuma feature de áudio entra nos modelos.
- Comparações de desempenho com a literatura devem usar métodos visual-only e protocolo de separação compatível. Resultados áudio-only ou multimodais devem aparecer apenas como contexto, nunca como comparação direta.
- A unidade de avaliação é o vídeo e todo split é feito por ator para impedir vazamento de identidade.
"""
atomic_write_text(REPORT_DIR / "experimental_scope.md", scope_text)

distribution_groups = ["Race", "Ethnicity", "Sex", "age_bin"]
sample_distribution, emotion_distribution = sample_distribution_tables(
    meta_run,
    group_attributes=distribution_groups,
    emotion_column="label_name",
)
atomic_write_text(TABLE_DIR / "sample_distribution_by_demographic.csv", sample_distribution.to_csv(index=False))
atomic_write_text(TABLE_DIR / "sample_distribution_emotion_by_demographic.csv", emotion_distribution.to_csv(index=False))

intersectional_specs = [
    ["Race", "Sex"],
    ["Race", "age_bin"],
    ["Sex", "age_bin"],
    ["Race", "Sex", "age_bin"],
]
intersectional_counts = []
for columns in intersectional_specs:
    table = meta_run.groupby(columns, dropna=False).size().rename("n").reset_index()
    table.insert(0, "intersection", " x ".join(columns))
    table["percent"] = 100 * table["n"] / len(meta_run)
    intersectional_counts.append(table)
sample_distribution_intersectional = pd.concat(intersectional_counts, ignore_index=True, sort=False)
atomic_write_text(TABLE_DIR / "sample_distribution_intersectional.csv", sample_distribution_intersectional.to_csv(index=False))

runtime_estimate_definitive_a100 = pd.DataFrame([
    {"scenario": "A100 sem cache de landmarks", "estimated_hours_low": 5.0, "estimated_hours_high": 8.0, "dominant_cost": "MediaPipe em CPU + 5-fold CV"},
    {"scenario": "A100 com landmarks íntegros em cache", "estimated_hours_low": 1.5, "estimated_hours_high": 3.5, "dominant_cost": "treino holdout + 20 treinos do 5-fold + XAI"},
    {"scenario": "retomada após falha", "estimated_hours_low": 0.1, "estimated_hours_high": 3.0, "dominant_cost": "somente shards/épocas/folds ainda incompletos"},
])
atomic_write_text(TABLE_DIR / "runtime_estimate_definitive_a100.csv", runtime_estimate_definitive_a100.to_csv(index=False))

display(modality_scope)
display(sample_distribution)
display(emotion_distribution.head(40))
display(runtime_estimate_definitive_a100)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, attribute in zip(axes.flat, distribution_groups):
    plot_data = sample_distribution[sample_distribution["protected_attribute"] == attribute]
    sns.barplot(data=plot_data, x="group", y="n", ax=ax, color="#4C78A8")
    ax.set_title(f"Distribuição por {attribute}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
fig.savefig(FIG_DIR / "02b_sample_distribution_demographics.png", bbox_inches="tight")
plt.show()
log_event("definitive_scope_and_sample_distribution", n_samples=int(len(meta_run)), n_distribution_rows=int(len(sample_distribution)))


## 7. Download dos vídeos

Os arquivos já existentes são pulados. Assim, se o Colab cair, basta executar novamente.


In [ ]:
# Etapa durável: download_videos
_stage_required = list([TABLE_DIR / "download_report.csv"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'download_videos',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'download_videos' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\ndef download_file(url, path, timeout=DOWNLOAD_TIMEOUT, retries=3):\n    path = Path(path)\n    if file_is_valid(path):\n        return {"path": str(path), "status": "exists", "bytes": path.stat().st_size, "error": None}\n    last_error = None\n    for attempt in range(1, retries + 1):\n        tmp = path.with_name(path.name + f".tmp-{os.getpid()}")\n        try:\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with requests.get(url, stream=True, timeout=timeout) as response:\n                response.raise_for_status()\n                with open(tmp, "wb") as handle:\n                    for chunk in response.iter_content(chunk_size=1024 * 256):\n                        if chunk:\n                            handle.write(chunk)\n                    handle.flush()\n                    os.fsync(handle.fileno())\n            os.replace(tmp, path)\n            return {"path": str(path), "status": "downloaded", "bytes": path.stat().st_size, "error": None}\n        except Exception as exc:\n            last_error = str(exc)\n            if tmp.exists():\n                tmp.unlink()\n            time.sleep(1.5 * attempt)\n    return {"path": str(path), "status": "failed", "bytes": 0, "error": last_error}\n\nrecords = meta_run[["video_url", "video_path", "file_stem"]].drop_duplicates().to_dict("records")\nstart = time.time()\nresults = []\nwith ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:\n    future_to_record = {\n        executor.submit(download_file, row["video_url"], row["video_path"]): row\n        for row in records\n    }\n    for future in tqdm(as_completed(future_to_record), total=len(future_to_record), desc="Baixando vídeos"):\n        row = future_to_record[future]\n        result = future.result()\n        results.append(result)\n        LEDGER.upsert_item(\n            RUN_ID,\n            domain="download",\n            item_key=row["file_stem"],\n            status="completed" if result["status"] in {"exists", "downloaded"} else "failed",\n            path=result["path"],\n            byte_count=result["bytes"],\n            error=result["error"],\n            metadata={"url": row["video_url"], "download_status": result["status"]},\n            compute_hash=False,\n        )\n\ndownload_report = pd.DataFrame(results).sort_values("path").reset_index(drop=True)\natomic_write_text(TABLE_DIR / "download_report.csv", download_report.to_csv(index=False))\nprint(f"Tempo de download: {(time.time() - start) / 60:.1f} min")\ndisplay(download_report["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="n"))\nfailed = download_report[download_report["status"] == "failed"]\nlog_event(\n    "download_videos",\n    start_time=start,\n    n_files=int(len(download_report)),\n    n_downloaded=int((download_report["status"] == "downloaded").sum()),\n    n_existing=int((download_report["status"] == "exists").sum()),\n    n_failed=int(len(failed)),\n)\nif len(failed):\n    display(failed.head(20))\n    raise RuntimeError(f"{len(failed)} download(s) falharam; execute novamente para retomar apenas os ausentes.")\n', "<cremad-stage-download_videos>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 8. Estatísticas dos vídeos e amostragem temporal

Antes dos landmarks, medimos duração, FPS e número de frames dos vídeos baixados. Essa tabela
responde à pergunta metodológica: quando usamos `T_FRAMES=100`, aproximando a configuração
temporal do EmoT do Pedro, estamos amostrando quantos quadros em vídeos de tamanhos diferentes?


In [ ]:

def inspect_video_file(path):
    path = Path(path)
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return {
            "video_path": str(path),
            "video_status": "open_failed",
            "frame_count": np.nan,
            "fps": np.nan,
            "duration_sec": np.nan,
            "width": np.nan,
            "height": np.nan,
            "sample_stride_frames": np.nan,
            "sample_stride_sec": np.nan,
        }
    frame_count = float(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    width = float(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = float(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    duration_sec = frame_count / fps if fps > 0 else np.nan
    sample_stride_frames = frame_count / max(T_FRAMES - 1, 1) if frame_count > 0 else np.nan
    sample_stride_sec = duration_sec / max(T_FRAMES - 1, 1) if np.isfinite(duration_sec) else np.nan
    return {
        "video_path": str(path),
        "video_status": "ok",
        "frame_count": frame_count,
        "fps": fps,
        "duration_sec": duration_sec,
        "width": width,
        "height": height,
        "sample_stride_frames": sample_stride_frames,
        "sample_stride_sec": sample_stride_sec,
    }

video_stats_start = time.time()
video_stats_rows = []
for _, row in tqdm(meta_run[["file_stem", "video_path"]].drop_duplicates().iterrows(), total=meta_run["file_stem"].nunique(), desc="Inspecionando vídeos"):
    stats = inspect_video_file(row["video_path"])
    stats["file_stem"] = row["file_stem"]
    video_stats_rows.append(stats)

video_stats = pd.DataFrame(video_stats_rows)
video_stats = video_stats.merge(meta_run[["file_stem", "label_name", "Race", "Ethnicity", "Sex", "age_bin"]], on="file_stem", how="left")
video_stats.to_csv(TABLE_DIR / "video_duration_stats.csv", index=False)

print(f"Estatísticas dos vídeos salvas em: {TABLE_DIR / 'video_duration_stats.csv'}")
display(video_stats[["frame_count", "fps", "duration_sec", "sample_stride_frames", "sample_stride_sec"]].describe().T)
display(video_stats.groupby("Race", dropna=False)[["duration_sec", "frame_count", "sample_stride_sec"]].agg(["count", "mean", "median", "min", "max"]))
log_event(
    "video_duration_statistics",
    start_time=video_stats_start,
    n_videos=int(len(video_stats)),
    mean_duration_sec=float(video_stats["duration_sec"].mean(skipna=True)),
    mean_frame_count=float(video_stats["frame_count"].mean(skipna=True)),
)


## 8. Landmarks faciais com MediaPipe

Usamos regiões expressivas para reduzir memória e melhorar interpretabilidade: labios,
olhos, sobrancelhas, nariz e contorno facial. O modelo recebe apenas geometria facial, nao
cor/textura diretamente.


In [ ]:

import mediapipe as mp

print(f"MediaPipe version: {getattr(mp, '__version__', 'unknown')}")

# O MediaPipe tem duas APIs Python em circulação:
# 1. API clássica: `mp.solutions.face_mesh.FaceMesh`
# 2. API moderna: `mp.tasks.vision.FaceLandmarker`
#
# Alguns ambientes recentes expõem apenas a API moderna. Por isso, esta célula
# detecta a API disponível e usa a alternativa correta automaticamente.
MEDIAPIPE_BACKEND = None
mp_face_mesh = None
BaseOptions = None
FaceLandmarker = None
FaceLandmarkerOptions = None
VisionRunningMode = None

if hasattr(mp, "solutions") and hasattr(mp.solutions, "face_mesh"):
    mp_face_mesh = mp.solutions.face_mesh
    MEDIAPIPE_BACKEND = "solutions"
elif hasattr(mp, "tasks") and hasattr(mp.tasks, "vision"):
    BaseOptions = mp.tasks.BaseOptions
    FaceLandmarker = mp.tasks.vision.FaceLandmarker
    FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode
    MEDIAPIPE_BACKEND = "tasks"
else:
    try:
        from mediapipe.tasks import python as mp_tasks_python
        from mediapipe.tasks.python import vision as mp_tasks_vision

        BaseOptions = mp_tasks_python.BaseOptions
        FaceLandmarker = mp_tasks_vision.FaceLandmarker
        FaceLandmarkerOptions = mp_tasks_vision.FaceLandmarkerOptions
        VisionRunningMode = mp_tasks_vision.RunningMode
        MEDIAPIPE_BACKEND = "tasks"
    except Exception as exc:
        raise ImportError(
            "A instalação de mediapipe não expõe `mp.solutions` nem a API "
            "`mediapipe.tasks`. Rode novamente a célula de instalação com "
            "`%pip install --upgrade mediapipe` e reinicie o runtime/kernel "
            "antes de continuar."
        ) from exc

print(f"MediaPipe backend selecionado: {MEDIAPIPE_BACKEND}")

# IDs semânticos baseados na topologia canônica do MediaPipe Face Mesh.
# Eles evitam depender de constantes `FACEMESH_*`, que só existem na API clássica.
REGION_IDS = {
    "lips": {
        0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95,
        146, 178, 181, 185, 191, 267, 269, 270, 291, 308, 310, 311, 312,
        314, 317, 318, 321, 324, 375, 402, 405, 409, 415,
    },
    "left_eye": {
        249, 263, 362, 373, 374, 380, 381, 382, 384, 385, 386, 387, 388,
        390, 398, 466,
    },
    "right_eye": {
        7, 33, 133, 144, 145, 153, 154, 155, 157, 158, 159, 160, 161, 163,
        173, 246,
    },
    "left_eyebrow": {276, 282, 283, 285, 293, 295, 296, 300, 334, 336},
    "right_eyebrow": {46, 52, 53, 55, 63, 65, 66, 70, 105, 107},
    "nose": {
        1, 2, 4, 5, 45, 48, 64, 94, 97, 98, 115, 168, 195, 197, 219,
        220, 235, 240, 275, 278, 290, 305, 326, 327, 344, 439, 440,
    },
    "face_oval": {
        10, 21, 54, 58, 67, 93, 103, 109, 127, 132, 136, 148, 149, 150,
        152, 162, 172, 176, 234, 251, 284, 288, 297, 323, 332, 338, 356,
        361, 365, 377, 378, 379, 389, 397, 400, 454,
    },
}

FACE_LANDMARKER_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
    "face_landmarker/float16/latest/face_landmarker.task"
)
FACE_LANDMARKER_MODEL_PATH = CACHE_DIR / "face_landmarker.task"

def ensure_face_landmarker_model():
    # Baixa o modelo da API MediaPipe Tasks quando necessário.
    if MEDIAPIPE_BACKEND != "tasks":
        return None
    if FACE_LANDMARKER_MODEL_PATH.exists() and FACE_LANDMARKER_MODEL_PATH.stat().st_size > 0:
        return FACE_LANDMARKER_MODEL_PATH
    print("Baixando modelo MediaPipe FaceLandmarker...")
    response = requests.get(FACE_LANDMARKER_MODEL_URL, timeout=120)
    response.raise_for_status()
    FACE_LANDMARKER_MODEL_PATH.write_bytes(response.content)
    print(f"Modelo salvo em: {FACE_LANDMARKER_MODEL_PATH}")
    return FACE_LANDMARKER_MODEL_PATH

def create_face_landmarker():
    # Cria o detector de landmarks compatível com a API disponível.
    if MEDIAPIPE_BACKEND == "solutions":
        return mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        )

    model_path = ensure_face_landmarker_model()
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(model_path)),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
    )
    return FaceLandmarker.create_from_options(options)

if LANDMARK_SET == "semantic":
    SELECTED_LANDMARK_IDS = sorted(set().union(*REGION_IDS.values()))
elif LANDMARK_SET == "full":
    SELECTED_LANDMARK_IDS = list(range(468))
else:
    raise ValueError("LANDMARK_SET deve ser 'semantic' ou 'full'.")

LANDMARK_TO_REGIONS = {}
for region, ids in REGION_IDS.items():
    for idx in ids:
        LANDMARK_TO_REGIONS.setdefault(idx, []).append(region)

N_LANDMARKS = len(SELECTED_LANDMARK_IDS)
BASE_FEATURE_DIM = N_LANDMARKS * 2
print(f"Landmarks selecionados: {N_LANDMARKS}")
print(f"Features base por frame: {BASE_FEATURE_DIM}")
for region, ids in REGION_IDS.items():
    print(region, len(set(ids).intersection(SELECTED_LANDMARK_IDS)))
log_event("mediapipe_landmark_setup", status=MEDIAPIPE_BACKEND, n_landmarks=int(N_LANDMARKS), feature_dim=int(BASE_FEATURE_DIM))


## 9. Extração e cache dos landmarks

Para cada video: amostramos frames uniformemente, detectamos a face, extraimos landmarks,
normalizamos centro/escala e interpolamos frames sem detecção quando a taxa de detecção
e suficiente.

O relatório de processamento permite avaliar se a própria detecção facial falha mais em
algum grupo demográfico.


In [ ]:
# Etapa durável: landmark_processing
_stage_required = list([CACHE_DIR / f"cremad_landmarks_T{T_FRAMES}_{LANDMARK_SET}.npz", TABLE_DIR / f"processing_report_T{T_FRAMES}_{LANDMARK_SET}.csv"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'landmark_processing',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'landmark_processing' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\nLANDMARK_SCOPE = "full" if FULL_RUN else f"dev{FAST_DEV_SAMPLES}"\nLANDMARK_CACHE = CACHE_DIR / (\n    f"cremad_landmarks_T{T_FRAMES}_{LANDMARK_SET}.npz"\n    if FULL_RUN else f"cremad_landmarks_T{T_FRAMES}_{LANDMARK_SET}_{LANDMARK_SCOPE}.npz"\n)\nPROCESSING_REPORT_CSV = TABLE_DIR / f"processing_report_T{T_FRAMES}_{LANDMARK_SET}.csv"\nPROCESSING_REPORT_CACHE = CACHE_DIR / f"processing_report_T{T_FRAMES}_{LANDMARK_SET}_{LANDMARK_SCOPE}.csv"\nLEGACY_PROCESSING_REPORT = BASE_DIR / "outputs" / "tables" / f"processing_report_T{T_FRAMES}_{LANDMARK_SET}.csv"\nLANDMARK_SHARD_DIR = CACHE_DIR / f"landmark_shards_T{T_FRAMES}_{LANDMARK_SET}_{LANDMARK_SCOPE}"\nLANDMARK_SHARD_DIR.mkdir(parents=True, exist_ok=True)\n\ndef sample_video_frames(video_path, t_frames=T_FRAMES):\n    cap = cv2.VideoCapture(str(video_path))\n    frames = []\n    while True:\n        ok, frame = cap.read()\n        if not ok:\n            break\n        frames.append(frame)\n    cap.release()\n    if not frames:\n        return []\n    idx = np.linspace(0, len(frames) - 1, t_frames).round().astype(int)\n    return [frames[i] for i in idx]\n\ndef normalize_landmarks_xy(coords_xy):\n    center = coords_xy.mean(axis=0, keepdims=True)\n    centered = coords_xy - center\n    span = coords_xy.max(axis=0) - coords_xy.min(axis=0)\n    return centered / max(float(span.max()), 1e-6)\n\ndef extract_frame_landmarks(frame_bgr, face_landmarker):\n    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)\n    if MEDIAPIPE_BACKEND == "solutions":\n        result = face_landmarker.process(rgb)\n        if not result.multi_face_landmarks:\n            return None\n        landmarks = result.multi_face_landmarks[0].landmark\n    else:\n        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))\n        result = face_landmarker.detect(mp_image)\n        if not result.face_landmarks:\n            return None\n        landmarks = result.face_landmarks[0]\n    coords = np.array([[landmarks[idx].x, landmarks[idx].y] for idx in SELECTED_LANDMARK_IDS], dtype=np.float32)\n    return normalize_landmarks_xy(coords).reshape(-1).astype(np.float32)\n\ndef interpolate_missing_frames(seq):\n    valid = ~np.isnan(seq).any(axis=1)\n    if valid.sum() == 0:\n        return None\n    if valid.all():\n        return seq.astype(np.float32)\n    x = np.arange(seq.shape[0])\n    out = seq.copy()\n    for d in range(seq.shape[1]):\n        out[~valid, d] = np.interp(x[~valid], x[valid], seq[:, d][valid])\n    return out.astype(np.float32)\n\ndef process_one_video(row, face_landmarker):\n    frames = sample_video_frames(row["video_path"], T_FRAMES)\n    if not frames:\n        return None, {"file_stem": row["file_stem"], "status": "no_frames", "n_frames_read": 0, "detected_frames": 0, "detection_rate": 0.0}\n    seq = np.full((T_FRAMES, BASE_FEATURE_DIM), np.nan, dtype=np.float32)\n    detected = 0\n    for i, frame in enumerate(frames):\n        landmarks = extract_frame_landmarks(frame, face_landmarker)\n        if landmarks is not None:\n            seq[i] = landmarks\n            detected += 1\n    rate = detected / T_FRAMES\n    if rate < MIN_FACE_DETECTION_RATE:\n        return None, {"file_stem": row["file_stem"], "status": "low_detection_rate", "n_frames_read": len(frames), "detected_frames": detected, "detection_rate": rate}\n    seq = interpolate_missing_frames(seq)\n    if seq is None:\n        return None, {"file_stem": row["file_stem"], "status": "interpolation_failed", "n_frames_read": len(frames), "detected_frames": detected, "detection_rate": rate}\n    return seq, {"file_stem": row["file_stem"], "status": "ok", "n_frames_read": len(frames), "detected_frames": detected, "detection_rate": rate}\n\ndef shard_path(start_index, end_index):\n    return LANDMARK_SHARD_DIR / f"shard_{start_index:05d}_{end_index:05d}.npz"\n\ndef load_landmark_shard(path):\n    data = np.load(path, allow_pickle=False)\n    reports = json.loads(str(data["report_json"].item()))\n    return data["X_base"], data["y"], data["file_stems"].astype(str), reports\n\nlandmark_start = time.time()\nreport_candidate = next(\n    (path for path in [PROCESSING_REPORT_CACHE, PROCESSING_REPORT_CSV, LEGACY_PROCESSING_REPORT] if file_is_valid(path)),\n    None,\n)\nif file_is_valid(LANDMARK_CACHE) and report_candidate is not None and "landmark_processing" not in FORCE_STAGES:\n    cached = np.load(LANDMARK_CACHE, allow_pickle=False)\n    X_base = cached["X_base"]\n    y = cached["y"]\n    valid_file_stems = cached["file_stems"].astype(str)\n    processing_report = pd.read_csv(report_candidate)\n    atomic_write_text(PROCESSING_REPORT_CSV, processing_report.to_csv(index=False))\n    atomic_write_text(PROCESSING_REPORT_CACHE, processing_report.to_csv(index=False))\n    landmark_status = "cache_loaded"\n    print(f"Cache consolidado carregado: {LANDMARK_CACHE}")\nelse:\n    rows = meta_run.to_dict("records")\n    ranges = [(start, min(start + LANDMARK_SHARD_SIZE, len(rows))) for start in range(0, len(rows), LANDMARK_SHARD_SIZE)]\n    missing_ranges = [(start, end) for start, end in ranges if not file_is_valid(shard_path(start, end)) or "landmark_processing" in FORCE_STAGES]\n    print(f"Shards totais={len(ranges)} | já válidos={len(ranges)-len(missing_ranges)} | pendentes={len(missing_ranges)}")\n    if missing_ranges:\n        with create_face_landmarker() as face_landmarker:\n            for start_index, end_index in tqdm(missing_ranges, desc="Shards de landmarks"):\n                shard_X, shard_y, shard_stems, shard_reports = [], [], [], []\n                for row in rows[start_index:end_index]:\n                    seq, report = process_one_video(row, face_landmarker)\n                    shard_reports.append(report)\n                    if seq is not None:\n                        shard_X.append(seq)\n                        shard_y.append(int(row["y"]))\n                        shard_stems.append(row["file_stem"])\n                X_array = np.stack(shard_X).astype(np.float32) if shard_X else np.empty((0, T_FRAMES, BASE_FEATURE_DIM), dtype=np.float32)\n                path = shard_path(start_index, end_index)\n                atomic_npz_save(\n                    path,\n                    X_base=X_array,\n                    y=np.asarray(shard_y, dtype=np.int64),\n                    file_stems=np.asarray(shard_stems, dtype=str),\n                    report_json=np.asarray(json.dumps(shard_reports, ensure_ascii=False)),\n                )\n                for report in shard_reports:\n                    LEDGER.upsert_item(\n                        RUN_ID,\n                        domain="landmark",\n                        item_key=report["file_stem"],\n                        status="completed" if report["status"] == "ok" else report["status"],\n                        path=path,\n                        metadata=report,\n                        compute_hash=False,\n                    )\n                print(f"Shard persistido: {path.name} | válidos={len(shard_X)}/{end_index-start_index}")\n\n    all_X, all_y, all_stems, all_reports = [], [], [], []\n    for start_index, end_index in ranges:\n        path = shard_path(start_index, end_index)\n        if not file_is_valid(path):\n            raise RuntimeError(f"Shard ausente ou vazio: {path}")\n        shard_X, shard_y, shard_stems, shard_reports = load_landmark_shard(path)\n        if len(shard_X):\n            all_X.append(shard_X)\n            all_y.append(shard_y)\n            all_stems.append(shard_stems)\n        all_reports.extend(shard_reports)\n    if not all_X:\n        raise RuntimeError("Nenhum vídeo atingiu o limiar mínimo de detecção facial.")\n    X_base = np.concatenate(all_X).astype(np.float32)\n    y = np.concatenate(all_y).astype(np.int64)\n    valid_file_stems = np.concatenate(all_stems).astype(str)\n    processing_report = pd.DataFrame(all_reports)\n    if len(processing_report) != len(rows):\n        raise RuntimeError(f"Consolidação incompleta: {len(processing_report)}/{len(rows)} relatórios")\n    atomic_write_text(PROCESSING_REPORT_CSV, processing_report.to_csv(index=False))\n    atomic_write_text(PROCESSING_REPORT_CACHE, processing_report.to_csv(index=False))\n    atomic_npz_save(\n        LANDMARK_CACHE,\n        X_base=X_base,\n        y=y,\n        file_stems=valid_file_stems,\n        selected_landmark_ids=np.asarray(SELECTED_LANDMARK_IDS, dtype=np.int64),\n    )\n    landmark_status = "shards_consolidated"\n    print(f"Cache consolidado salvo: {LANDMARK_CACHE}")\n\nprint(f"X_base: {X_base.shape} | y: {y.shape}")\ndisplay(processing_report["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="n"))\nlog_event(\n    "landmark_processing",\n    status=landmark_status,\n    start_time=landmark_start,\n    n_valid_videos=int(len(valid_file_stems)),\n    n_total_videos=int(len(processing_report)),\n    ok_rate=float((processing_report["status"] == "ok").mean()),\n    mean_detection_rate=float(processing_report["detection_rate"].mean()),\n)\n', "<cremad-stage-landmark_processing>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 9.1 Amostras visuais do pré-processamento

Esta célula gera prints visuais para a dissertação e para a reunião com os orientadores:

1. amostras de frames reais do CREMA-D por grupo;
2. sequência temporal amostrada;
3. frame com landmarks detectados;
4. landmarks normalizados, que são a entrada efetiva do modelo.


In [ ]:

VISUAL_SAMPLE_MAX_GROUPS = 6
VISUAL_STRIP_FRAMES = 8
VISUAL_FRAME_HEIGHT = 96

REGION_COLORS = {
    "lips": (214, 39, 40),
    "left_eye": (31, 119, 180),
    "right_eye": (31, 119, 180),
    "left_eyebrow": (148, 103, 189),
    "right_eyebrow": (148, 103, 189),
    "nose": (44, 160, 44),
    "face_oval": (127, 127, 127),
}

def extract_selected_landmarks_pixels(frame_bgr, face_landmarker):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    if MEDIAPIPE_BACKEND == "solutions":
        result = face_landmarker.process(rgb)
        if not result.multi_face_landmarks:
            return None
        landmarks = result.multi_face_landmarks[0].landmark
    else:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        result = face_landmarker.detect(mp_image)
        if not result.face_landmarks:
            return None
        landmarks = result.face_landmarks[0]
    return np.array([[landmarks[idx].x * w, landmarks[idx].y * h] for idx in SELECTED_LANDMARK_IDS], dtype=np.float32)

def draw_landmark_overlay(frame_bgr, pixel_coords):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB).copy()
    if pixel_coords is None:
        return rgb
    id_to_pos = {lm_id: pos for pos, lm_id in enumerate(SELECTED_LANDMARK_IDS)}
    for region, ids in REGION_IDS.items():
        color = REGION_COLORS.get(region, (255, 255, 255))
        for lm_id in ids:
            pos = id_to_pos.get(lm_id)
            if pos is None:
                continue
            x, y = pixel_coords[pos]
            cv2.circle(rgb, (int(round(x)), int(round(y))), 2, color, -1, lineType=cv2.LINE_AA)
    return rgb

def make_temporal_strip(frames, max_frames=VISUAL_STRIP_FRAMES, height=VISUAL_FRAME_HEIGHT):
    if not frames:
        return None
    idx = np.linspace(0, len(frames) - 1, min(max_frames, len(frames))).round().astype(int)
    imgs = []
    for i in idx:
        rgb = cv2.cvtColor(frames[i], cv2.COLOR_BGR2RGB)
        scale = height / max(1, rgb.shape[0])
        width = max(1, int(rgb.shape[1] * scale))
        imgs.append(cv2.resize(rgb, (width, height), interpolation=cv2.INTER_AREA))
    return np.concatenate(imgs, axis=1)

visual_start = time.time()
visual_candidates = processing_report.merge(
    meta_run[["file_stem", "video_path", "label_name", "Race", "Ethnicity", "Sex", "age_bin"]],
    on="file_stem",
    how="left",
)
visual_candidates = visual_candidates[visual_candidates["status"] == "ok"].drop_duplicates("file_stem")

if visual_candidates.empty:
    print("Nenhuma amostra processada com status ok para visualização.")
else:
    sample_parts = []
    for _, group in visual_candidates.groupby("Race", dropna=False):
        sample_parts.append(group.sample(1, random_state=SEED))
    visual_samples = pd.concat(sample_parts, ignore_index=True).head(VISUAL_SAMPLE_MAX_GROUPS)
    visual_samples.to_csv(TABLE_DIR / "visual_sample_manifest.csv", index=False)
    display(visual_samples[["file_stem", "label_name", "Race", "Ethnicity", "Sex", "age_bin", "detection_rate"]])

    n = len(visual_samples)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.6))
    axes = np.atleast_1d(axes)

    visual_payload = []
    with create_face_landmarker() as face_landmarker:
        for ax, (_, row) in zip(axes, visual_samples.iterrows()):
            frames = sample_video_frames(row["video_path"], T_FRAMES)
            frame = frames[len(frames) // 2] if frames else None
            if frame is None:
                ax.text(0.5, 0.5, "sem frame", ha="center", va="center")
                ax.axis("off")
                continue
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            ax.imshow(rgb)
            ax.set_title(f"{row['Race']}\n{row['label_name']}", fontsize=9)
            ax.axis("off")
            pixel_coords = extract_selected_landmarks_pixels(frame, face_landmarker)
            overlay = draw_landmark_overlay(frame, pixel_coords)
            norm_flat = extract_frame_landmarks(frame, face_landmarker)
            norm_xy = norm_flat.reshape(-1, 2) if norm_flat is not None else None
            strip = make_temporal_strip(frames)
            visual_payload.append((row, rgb, strip, overlay, norm_xy))

    plt.tight_layout()
    sample_fig_path = FIG_DIR / "02_visual_sample_frames_by_race.png"
    fig.savefig(sample_fig_path, bbox_inches="tight")
    plt.show()

    n_pipe = len(visual_payload)
    fig, axes = plt.subplots(n_pipe, 4, figsize=(15, 3.2 * n_pipe))
    axes = np.asarray(axes).reshape(n_pipe, 4)
    for i, (row, rgb, strip, overlay, norm_xy) in enumerate(visual_payload):
        title_prefix = f"{row['file_stem']} | {row['Race']} | {row['label_name']}"
        axes[i, 0].imshow(rgb)
        axes[i, 0].set_title("Frame bruto\n" + title_prefix, fontsize=8)
        if strip is not None:
            axes[i, 1].imshow(strip)
        else:
            axes[i, 1].text(0.5, 0.5, "sem strip", ha="center", va="center")
        axes[i, 1].set_title(f"Amostragem temporal\n{VISUAL_STRIP_FRAMES} de {T_FRAMES} frames", fontsize=8)
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Processamento\nlandmarks detectados", fontsize=8)
        if norm_xy is not None:
            axes[i, 3].scatter(norm_xy[:, 0], -norm_xy[:, 1], s=8, alpha=0.8)
            axes[i, 3].set_aspect("equal", adjustable="box")
        else:
            axes[i, 3].text(0.5, 0.5, "sem landmarks", ha="center", va="center")
        axes[i, 3].set_title("Entrada do modelo\nlandmarks normalizados", fontsize=8)
        for j in range(4):
            axes[i, j].axis("off")
    plt.tight_layout()
    pipeline_fig_path = FIG_DIR / "02_visual_preprocessing_pipeline.png"
    fig.savefig(pipeline_fig_path, bbox_inches="tight")
    plt.show()

    print(f"Figura de amostras salva em: {sample_fig_path}")
    print(f"Figura do pipeline visual salva em: {pipeline_fig_path}")
    log_event(
        "visual_preprocessing_samples",
        start_time=visual_start,
        n_samples=int(n),
        sample_figure=str(sample_fig_path),
        pipeline_figure=str(pipeline_fig_path),
    )


## 10. Fairness do pré-processamento

Antes do modelo, avaliamos a taxa de detecção facial por grupo. Se a detecção falha mais
em um grupo, isso pode contaminar as métricas posteriores.


In [ ]:

meta_valid = meta_run[meta_run["file_stem"].isin(valid_file_stems)].copy()
meta_valid = meta_valid.set_index("file_stem").loc[valid_file_stems].reset_index()
meta_valid["y"] = y
meta_valid["label_name"] = meta_valid["label_code"].map(LABEL_MAP)

proc_demo = processing_report.merge(meta_run[["file_stem", "Race", "Ethnicity", "Sex", "age_bin", "label_name"]], on="file_stem", how="left")
preprocessing_fairness_start = time.time()
processing_summaries = []
for group_col in ["Race", "Ethnicity", "Sex", "age_bin"]:
    print(f"\nProcessamento por {group_col}")
    summary = proc_demo.groupby(group_col, dropna=False).agg(
        n=("file_stem", "count"),
        ok_rate=("status", lambda s: float((s == "ok").mean())),
        mean_detection_rate=("detection_rate", "mean"),
        median_detection_rate=("detection_rate", "median"),
    ).reset_index()
    display(summary)
    summary.insert(0, "group_col", group_col)
    processing_summaries.append(summary)

processing_quality_by_group = pd.concat(processing_summaries, ignore_index=True)
processing_quality_by_group.to_csv(TABLE_DIR / "processing_quality_by_demographic_group.csv", index=False)
processing_report.to_csv(LOG_DIR / f"processing_report_{RUN_ID}.csv", index=False)
print(f"Resumo de qualidade por grupo salvo em: {TABLE_DIR / 'processing_quality_by_demographic_group.csv'}")
print(f"Log de processamento salvo em: {LOG_DIR / f'processing_report_{RUN_ID}.csv'}")

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=proc_demo, x="Race", y="detection_rate", ax=ax)
ax.set_title("Taxa de detecção facial por Race")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=20)
plt.tight_layout()
fig.savefig(FIG_DIR / "02_preprocessing_detection_rate_by_race.png", bbox_inches="tight")
plt.show()
log_event(
    "preprocessing_fairness_quality",
    start_time=preprocessing_fairness_start,
    n_groups=int(processing_quality_by_group["group_col"].nunique()),
    mean_detection_rate=float(proc_demo["detection_rate"].mean()),
)


## 11. Derivadas temporais

Adicionamos velocidade, aceleração e jerk aos landmarks. Assim, o EmoT recebe informação
de movimento facial, não apenas a geometria estática de cada frame.


In [ ]:

def compute_temporal_derivatives(X):
    vel = np.zeros_like(X, dtype=np.float32)
    vel[:, 1:, :] = X[:, 1:, :] - X[:, :-1, :]
    acc = np.zeros_like(X, dtype=np.float32)
    acc[:, 1:-1, :] = X[:, 2:, :] - 2 * X[:, 1:-1, :] + X[:, :-2, :]
    jerk = np.zeros_like(X, dtype=np.float32)
    jerk[:, 2:-2, :] = X[:, 4:, :] - 2 * X[:, 3:-1, :] + 2 * X[:, 1:-3, :] - X[:, :-4, :]
    return np.concatenate([X, vel, acc, jerk], axis=-1).astype(np.float32)

derivative_start = time.time()
if ADD_TEMPORAL_DERIVATIVES:
    X = compute_temporal_derivatives(X_base)
    FEATURE_BLOCKS = ["pos", "vel", "acc", "jerk"]
else:
    X = X_base.astype(np.float32)
    FEATURE_BLOCKS = ["pos"]

T, D = X.shape[1], X.shape[2]
print(f"Tensor final X: {X.shape}")
print(f"Blocos de feature: {FEATURE_BLOCKS}")
log_event("temporal_feature_engineering", start_time=derivative_start, x_shape=str(X.shape), feature_blocks=",".join(FEATURE_BLOCKS))


## 12. Split por ator

O split é feito por ator para evitar vazamento de identidade. Tentamos estratificar por
`Race`; se isso falhar por grupos pequenos, usamos split aleatório por ator como fallback.


In [ ]:

def make_actor_split(meta_df, seed=SEED):
    actors = meta_df[["ActorID", "Race", "Sex", "Ethnicity", "Age", "age_bin"]].drop_duplicates("ActorID").copy()
    known = actors[actors["Race"] != "Unknown"].copy()
    unknown = actors[actors["Race"] == "Unknown"].copy()
    try:
        train_actors, temp_actors = train_test_split(known, test_size=0.30, random_state=seed, stratify=known["Race"])
        val_actors, test_actors = train_test_split(temp_actors, test_size=0.50, random_state=seed, stratify=temp_actors["Race"])
    except ValueError as exc:
        print("Stratified split falhou; usando split aleatório por ator.")
        print(exc)
        train_actors, temp_actors = train_test_split(known, test_size=0.30, random_state=seed)
        val_actors, test_actors = train_test_split(temp_actors, test_size=0.50, random_state=seed)
    train_actors = pd.concat([train_actors, unknown], ignore_index=True)
    split_map = {int(a): "train" for a in train_actors["ActorID"]}
    split_map.update({int(a): "val" for a in val_actors["ActorID"]})
    split_map.update({int(a): "test" for a in test_actors["ActorID"]})
    return split_map

split_start = time.time()
split_map = make_actor_split(meta_valid)
meta_valid["split"] = meta_valid["ActorID"].map(split_map)
train_idx = np.where(meta_valid["split"].values == "train")[0]
val_idx = np.where(meta_valid["split"].values == "val")[0]
test_idx = np.where(meta_valid["split"].values == "test")[0]

print(f"Train: {len(train_idx):,} vídeos | atores: {meta_valid.loc[train_idx, 'ActorID'].nunique()}")
print(f"Val:   {len(val_idx):,} vídeos | atores: {meta_valid.loc[val_idx, 'ActorID'].nunique()}")
print(f"Test:  {len(test_idx):,} vídeos | atores: {meta_valid.loc[test_idx, 'ActorID'].nunique()}")
display(pd.crosstab(meta_valid["Race"], meta_valid["split"], margins=True))
display(pd.crosstab(meta_valid["label_name"], meta_valid["split"], margins=True))
log_event(
    "actor_holdout_split",
    start_time=split_start,
    n_train=int(len(train_idx)),
    n_val=int(len(val_idx)),
    n_test=int(len(test_idx)),
    train_actors=int(meta_valid.loc[train_idx, "ActorID"].nunique()),
    val_actors=int(meta_valid.loc[val_idx, "ActorID"].nunique()),
    test_actors=int(meta_valid.loc[test_idx, "ActorID"].nunique()),
)


## 13. Normalizacao e DataLoaders

O z-score e ajustado apenas no treino. Isso evita vazamento estatístico para validação e
teste.


In [ ]:

normalization_start = time.time()
X_train_raw, y_train = X[train_idx], y[train_idx]
X_val_raw, y_val = X[val_idx], y[val_idx]
X_test_raw, y_test = X[test_idx], y[test_idx]

flat_train = X_train_raw.reshape(-1, D)
mu = flat_train.mean(axis=0).astype(np.float32)
sigma = flat_train.std(axis=0).astype(np.float32)
sigma[sigma == 0] = 1.0

X_train = ((X_train_raw - mu) / sigma).astype(np.float32)
X_val = ((X_val_raw - mu) / sigma).astype(np.float32)
X_test = ((X_test_raw - mu) / sigma).astype(np.float32)

meta_train = meta_valid.iloc[train_idx].reset_index(drop=True)
meta_val = meta_valid.iloc[val_idx].reset_index(drop=True)
meta_test = meta_valid.iloc[test_idx].reset_index(drop=True)
meta_valid.to_csv(TABLE_DIR / "manifest_valid_with_splits.csv", index=False)
np.savez_compressed(CACHE_DIR / "normalization_split_cache.npz", mu=mu, sigma=sigma, train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Media treino normalizado: {X_train.mean():.4f}")
print(f"Desvio treino normalizado: {X_train.std():.4f}")

def make_loader(X_np, y_np, batch_size=BATCH_SIZE, shuffle=False, sampler=None):
    ds = TensorDataset(torch.tensor(X_np, dtype=torch.float32), torch.tensor(y_np, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle if sampler is None else False, sampler=sampler, num_workers=0, pin_memory=torch.cuda.is_available())

def class_weights_from_y(y_np):
    counts = np.bincount(y_np, minlength=N_CLASSES).astype(np.float32)
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def make_group_label_sampler(meta_df):
    keys = meta_df["label_code"].astype(str) + "__" + meta_df["Race"].astype(str)
    counts = keys.value_counts().to_dict()
    weights = keys.map(lambda k: 1.0 / counts[k]).values.astype(np.float64)
    weights = weights / weights.mean()
    return WeightedRandomSampler(torch.DoubleTensor(weights), num_samples=len(weights), replacement=True)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val)
fair_train_loader = make_loader(X_train, y_train, sampler=make_group_label_sampler(meta_train))
criterion_weighted = nn.CrossEntropyLoss(weight=class_weights_from_y(y_train), label_smoothing=LABEL_SMOOTHING)
log_event(
    "normalization_and_dataloaders",
    start_time=normalization_start,
    x_train_shape=str(X_train.shape),
    x_val_shape=str(X_val.shape),
    x_test_shape=str(X_test.shape),
)


## Documentação do balanceamento

A execução não descarta vídeos. Todos os modelos usam loss ponderado por emoção; a variante `EmoT_Reweighted` acrescenta amostragem por emoção × Race. As tabelas abaixo tornam os pesos e a decisão metodológica auditáveis.


In [ ]:

balance_class_weights, balance_sampler = balance_strategy_table(meta_train)
balance_documentation = pd.DataFrame([
    {
        "model_scope": "FlatMLP, TemporalCNN1D, EmoT",
        "label_imbalance_control": "CrossEntropy ponderada pelo inverso da frequência da emoção",
        "demographic_imbalance_control": "nenhum resampling demográfico",
        "downsampling": False,
        "reason": "Preserva todas as amostras e controla o desbalanceamento de classe no loss.",
    },
    {
        "model_scope": "EmoT_Reweighted",
        "label_imbalance_control": "CrossEntropy ponderada pelo inverso da frequência da emoção",
        "demographic_imbalance_control": "WeightedRandomSampler por emoção x Race",
        "downsampling": False,
        "reason": "Mitigação comparável que aumenta a exposição de combinações minoritárias sem descartar vídeos.",
    },
])
atomic_write_text(TABLE_DIR / "balance_class_weights.csv", balance_class_weights.to_csv(index=False))
atomic_write_text(TABLE_DIR / "balance_sampler_label_x_race.csv", balance_sampler.to_csv(index=False))
atomic_write_text(TABLE_DIR / "balance_strategy_documentation.csv", balance_documentation.to_csv(index=False))
display(balance_documentation)
display(balance_class_weights)
display(balance_sampler)
log_event("balance_strategy_documentation", n_train=int(len(meta_train)), n_label_race_cells=int(len(balance_sampler)))


## 14. Modelos

Os três modelos usam a mesma entrada `(B, T, D)`. O modelo principal é uma adaptação
PyTorch do EmoT/AcT usado pelo Pedro: projeção linear dos landmarks, token de classe,
embedding posicional aprendível, encoder Transformer pós-normalização e cabeça MLP.

Mantemos CREMA-D + MediaPipe porque o objetivo agora é fairness étnico-racial; portanto,
a fidelidade aqui está na camada Transformer e nos metaparâmetros, não no dataset
RAVDESS/OpenFace original.


In [ ]:

class FlatMLP(nn.Module):
    def __init__(self, T, D, n_classes, hidden=256, dropout=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(T * D, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x):
        return self.net(x)

class TemporalCNN1D(nn.Module):
    def __init__(self, T, D, n_classes, n_filters=96, dropout=0.30):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(D, n_filters, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(n_filters),
            nn.MaxPool1d(2),
            nn.Conv1d(n_filters, n_filters * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(n_filters * 2),
            nn.MaxPool1d(2),
            nn.Conv1d(n_filters * 2, n_filters * 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(n_filters * 2, n_classes))
    def forward(self, x):
        x = x.permute(0, 2, 1)
        return self.classifier(self.features(x).squeeze(-1))

class PedroStyleTransformerBlockWithAttention(nn.Module):
    def __init__(self, d_model, n_heads, dim_ff, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.ff = nn.Sequential(nn.Linear(d_model, dim_ff), nn.GELU(), nn.Linear(dim_ff, d_model))
        self.drop2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
    def forward(self, x, return_attention=False):
        attn_out, attn_w = self.attn(x, x, x, need_weights=return_attention, average_attn_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        x = self.norm2(x + self.drop2(self.ff(x)))
        return (x, attn_w) if return_attention else (x, None)

class EmoTransformer(nn.Module):
    def __init__(
        self,
        D,
        n_classes,
        d_model=192,
        n_heads=3,
        n_layers=6,
        dim_ff=768,
        dropout=0.30,
        mlp_head_size=256,
        max_len=100,
    ):
        super().__init__()
        self.input_proj = nn.Linear(D, d_model)
        self.cls_token = nn.Parameter(torch.empty(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.empty(1, max_len + 1, d_model))
        self.blocks = nn.ModuleList([PedroStyleTransformerBlockWithAttention(d_model, n_heads, dim_ff, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_model, mlp_head_size), nn.Linear(mlp_head_size, n_classes))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_normal_(self.cls_token)
        nn.init.normal_(self.pos_embed, mean=0.0, std=0.02)

    def forward(self, x, return_attention=False):
        B = x.size(0)
        x = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_embed[:, : x.size(1), :]
        attentions = []
        for block in self.blocks:
            x, attn = block(x, return_attention=return_attention)
            if return_attention:
                attentions.append(attn)
        logits = self.head(x[:, 0])
        return (logits, attentions) if return_attention else logits

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 15. Métricas e fairness preditivo

Definimos desempenho global, desempenho por grupo, calibration/ECE, equal opportunity,
equalized odds e teste de permutação para o gap de acurácia.


In [ ]:

def softmax_np(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)

def compute_basic_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

def multiclass_ece(y_true, y_prob, n_bins=10):
    conf = y_prob.max(axis=1)
    pred = y_prob.argmax(axis=1)
    ok = (pred == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(y_true)) * abs(ok[mask].mean() - conf[mask].mean())
    return float(ece)

def metrics_by_group(y_true, y_pred, y_prob, groups, min_n=30):
    rows = []
    groups = np.array(groups)
    for group in pd.Series(groups).dropna().unique():
        mask = groups == group
        if mask.sum() < min_n:
            continue
        m = compute_basic_metrics(y_true[mask], y_pred[mask])
        m.update({"group": group, "n": int(mask.sum()), "ece": multiclass_ece(y_true[mask], y_prob[mask])})
        rows.append(m)
    return pd.DataFrame(rows).sort_values("group") if rows else pd.DataFrame()

def recall_by_class_group(y_true, y_pred, groups, min_n=10):
    rows = []
    groups = np.array(groups)
    for group in pd.Series(groups).dropna().unique():
        for c, name in enumerate(CLASS_NAMES):
            mask = (groups == group) & (y_true == c)
            if mask.sum() >= min_n:
                rows.append({"group": group, "class_name": name, "recall": float((y_pred[mask] == c).mean()), "n": int(mask.sum())})
    return pd.DataFrame(rows)

def fpr_by_class_group(y_true, y_pred, groups, min_n=10):
    rows = []
    groups = np.array(groups)
    for group in pd.Series(groups).dropna().unique():
        for c, name in enumerate(CLASS_NAMES):
            mask = (groups == group) & (y_true != c)
            if mask.sum() >= min_n:
                rows.append({"group": group, "class_name": name, "fpr": float((y_pred[mask] == c).mean()), "n": int(mask.sum())})
    return pd.DataFrame(rows)

def summarize_fairness(y_true, y_pred, y_prob, groups, protected_name, min_n=30):
    group_df = metrics_by_group(y_true, y_pred, y_prob, groups, min_n=min_n)
    if group_df.empty:
        return group_df, pd.DataFrame()
    rec = recall_by_class_group(y_true, y_pred, groups)
    fpr = fpr_by_class_group(y_true, y_pred, groups)
    if len(rec):
        eo = rec.groupby("class_name")["recall"].agg(lambda s: s.max() - s.min())
        equal_opp_mean, equal_opp_max = eo.mean(), eo.max()
    else:
        equal_opp_mean = equal_opp_max = np.nan
    if len(rec) and len(fpr):
        tpr_gap = rec.groupby("class_name")["recall"].agg(lambda s: s.max() - s.min())
        fpr_gap = fpr.groupby("class_name")["fpr"].agg(lambda s: s.max() - s.min())
        eod = pd.concat([tpr_gap.rename("tpr"), fpr_gap.rename("fpr")], axis=1).max(axis=1)
        eod_mean, eod_max = eod.mean(), eod.max()
    else:
        eod_mean = eod_max = np.nan
    summary = pd.DataFrame([{
        "protected_attribute": protected_name,
        "n_groups_used": len(group_df),
        "worst_group_accuracy": group_df["accuracy"].min(),
        "performance_gap_accuracy": group_df["accuracy"].max() - group_df["accuracy"].min(),
        "equal_opportunity_gap_mean": equal_opp_mean,
        "equal_opportunity_gap_max": equal_opp_max,
        "equalized_odds_gap_mean": eod_mean,
        "equalized_odds_gap_max": eod_max,
        "calibration_ece_gap": group_df["ece"].max() - group_df["ece"].min(),
    }])
    return group_df, summary

def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=500, seed=SEED):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), size=len(y_true))
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    vals = np.array(vals)
    return {"mean": float(vals.mean()), "ci_low": float(np.percentile(vals, 2.5)), "ci_high": float(np.percentile(vals, 97.5))}

def permutation_gap_test(y_true, y_pred, groups, n_perm=500, min_n=30, seed=SEED):
    rng = np.random.default_rng(seed)
    groups = np.array(groups)
    def gap(g):
        vals = []
        for group in pd.Series(g).dropna().unique():
            mask = g == group
            if mask.sum() >= min_n:
                vals.append(accuracy_score(y_true[mask], y_pred[mask]))
        return np.nan if len(vals) < 2 else max(vals) - min(vals)
    obs = gap(groups)
    if np.isnan(obs):
        return {"observed_gap": np.nan, "p_value": np.nan}
    null = []
    for _ in range(n_perm):
        shuffled = groups.copy()
        rng.shuffle(shuffled)
        null.append(gap(shuffled))
    null = np.array(null)
    return {"observed_gap": float(obs), "p_value": float((np.sum(null >= obs) + 1) / (len(null) + 1))}


## 16. Treino e inferência

O loop usa mixed precision quando há GPU. Para aproximar o treino do Pedro, usamos AdamW,
label smoothing, clipping de gradiente e schedule Transformer com warmup. O early stopping
fica ativo por viabilidade no Colab; para uma réplica estrita do número de épocas, defina
`PATIENCE = None`.


In [ ]:

# Checkpoints concretos esperados incluem latest_EmoT.pt e best_EmoT.pt.
def clip_gradients(model):
    if GRAD_CLIP_VALUE is not None:
        torch.nn.utils.clip_grad_value_(model.parameters(), GRAD_CLIP_VALUE)
    if GRAD_CLIP_NORM is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)

def make_pedro_lr_scheduler(optimizer, total_steps):
    warmup_steps = max(1, int(WARMUP_PERC * total_steps))
    def lr_lambda(step):
        step = max(1, step)
        return (D_MODEL ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda), warmup_steps

def train_one_epoch(model, loader, optimizer, criterion, scaler=None, scheduler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if USE_AMP:
            with torch.amp.autocast("cuda"):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            clip_gradients(model)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            clip_gradients(model)
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def predict_model(model, X_np, batch_size=BATCH_SIZE):
    model.eval()
    logits_all = []
    dummy = np.zeros(len(X_np), dtype=np.int64)
    for xb, _ in make_loader(X_np, dummy, batch_size=batch_size):
        logits_all.append(model(xb.to(DEVICE, non_blocking=True)).detach().cpu().numpy())
    logits_all = np.concatenate(logits_all)
    prob = softmax_np(logits_all)
    return prob.argmax(axis=1), prob, logits_all

def evaluate_on_arrays(model, X_np, y_np):
    pred, prob, logits = predict_model(model, X_np)
    return compute_basic_metrics(y_np, pred), pred, prob, logits

def load_torch_payload(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def fit_model(model_name, model, loader, val_X, val_y, criterion, epochs=EPOCHS):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler, warmup_steps = (None, None)
    if USE_PEDRO_LR_SCHEDULE:
        scheduler, warmup_steps = make_pedro_lr_scheduler(optimizer, max(1, len(loader) * epochs))
    scaler = torch.amp.GradScaler("cuda") if USE_AMP else None
    latest_path = CKPT_DIR / f"latest_{model_name}.pt"
    best_path = CKPT_DIR / f"best_{model_name}.pt"
    history_path = LOG_DIR / f"training_history_{model_name}.csv"
    best_f1, best_epoch, patience_left = -1.0, -1, PATIENCE
    history, start_epoch = [], 1
    start = time.time()

    force_model_training = bool({f"train_{model_name}", "training_all_models"}.intersection(FORCE_STAGES))
    if file_is_valid(latest_path) and not force_model_training:
        payload = load_torch_payload(latest_path, DEVICE)
        model.load_state_dict(payload["model_state_dict"])
        optimizer.load_state_dict(payload["optimizer_state_dict"])
        if scheduler is not None and payload.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(payload["scheduler_state_dict"])
        if scaler is not None and payload.get("scaler_state_dict") is not None:
            scaler.load_state_dict(payload["scaler_state_dict"])
        history = payload.get("history", [])
        atomic_write_text(history_path, pd.DataFrame(history).to_csv(index=False))
        recorded_epochs = {int(row["epoch"]) for row in LEDGER.training_history(RUN_ID, model_name)}
        missing_history = [row for row in history if int(row["epoch"]) not in recorded_epochs]
        LEDGER.record_training_history(RUN_ID, model_name, missing_history, checkpoint_path=latest_path)
        best_f1 = float(payload.get("best_f1", -1.0))
        best_epoch = int(payload.get("best_epoch", -1))
        patience_left = payload.get("patience_left", PATIENCE)
        start_epoch = int(payload.get("epoch", 0)) + 1
        if payload.get("python_random_state") is not None:
            random.setstate(payload["python_random_state"])
        if payload.get("numpy_random_state") is not None:
            np.random.set_state(payload["numpy_random_state"])
        if payload.get("torch_random_state") is not None:
            torch.set_rng_state(payload["torch_random_state"])
        if torch.cuda.is_available() and payload.get("cuda_random_state_all") is not None:
            torch.cuda.set_rng_state_all(payload["cuda_random_state_all"])
        print(f"[{model_name}] retomando na época {start_epoch}; última íntegra={start_epoch-1}")
        if payload.get("completed") and file_is_valid(best_path):
            best_payload = load_torch_payload(best_path, DEVICE)
            model.load_state_dict(best_payload["model_state_dict"])
            print(f"[{model_name}] treinamento já concluído; melhor época={best_epoch}")
            return model, pd.DataFrame(history), best_path

    completed = False
    for epoch in range(start_epoch, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, loader, optimizer, criterion, scaler, scheduler)
        val_metrics, _, _, _ = evaluate_on_arrays(model, val_X, val_y)
        current_lr = optimizer.param_groups[0]["lr"]
        row = {"model": model_name, "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "lr": current_lr, **{f"val_{key}": value for key, value in val_metrics.items()}}
        history.append(row)
        print(f"[{model_name}] epoch {epoch:04d}/{epochs} loss={train_loss:.4f} train_acc={train_acc:.3f} val_macro_f1={val_metrics['macro_f1']:.3f} lr={current_lr:.2e}")
        if val_metrics["macro_f1"] > best_f1:
            best_f1, best_epoch = float(val_metrics["macro_f1"]), epoch
            if PATIENCE is not None:
                patience_left = PATIENCE
            atomic_torch_save({
                "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_f1,
                "T": T, "D": D, "class_names": CLASS_NAMES, "mu": mu, "sigma": sigma,
            }, best_path)
        elif PATIENCE is not None:
            patience_left -= 1

        early_stop = PATIENCE is not None and patience_left <= 0
        completed = early_stop or epoch == epochs
        checkpoint_payload = {
            "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "epoch": epoch,
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "patience_left": patience_left,
            "completed": completed,
            "history": history,
            "python_random_state": random.getstate(),
            "numpy_random_state": np.random.get_state(),
            "torch_random_state": torch.get_rng_state(),
            "cuda_random_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        if epoch % CHECKPOINT_EVERY_EPOCHS == 0 or completed:
            atomic_torch_save(checkpoint_payload, latest_path)
        atomic_write_text(history_path, pd.DataFrame(history).to_csv(index=False))
        LEDGER.record_training_epoch(RUN_ID, model_name, epoch, row, checkpoint_path=latest_path)
        if early_stop:
            print(f"[{model_name}] early stopping na época {epoch}.")
            break

    if not file_is_valid(best_path):
        raise RuntimeError(f"Checkpoint de melhor modelo não foi produzido: {best_path}")
    best_payload = load_torch_payload(best_path, DEVICE)
    model.load_state_dict(best_payload["model_state_dict"])
    LEDGER.register_artifact(RUN_ID, f"train_{model_name}", latest_path, kind="latest_checkpoint")
    LEDGER.register_artifact(RUN_ID, f"train_{model_name}", best_path, kind="best_checkpoint")
    print(f"Melhor {model_name}: epoch={best_epoch}, val_macro_f1={best_f1:.4f}, tempo={(time.time()-start)/60:.1f} min")
    log_event(f"train_{model_name}", start_time=start, best_epoch=int(best_epoch), best_val_macro_f1=float(best_f1), checkpoint=str(best_path), latest_checkpoint=str(latest_path), params=int(count_parameters(model)), warmup_steps=int(warmup_steps) if warmup_steps is not None else None)
    return model, pd.DataFrame(history), best_path


## 17. Treinamento dos modelos

Treinamos os baselines, o EmoT principal e uma variante mitigada com reweighting por
classe e `Race`.


In [ ]:
# Etapa durável: training_all_models
_stage_required = list([CKPT_DIR / "best_FlatMLP.pt", CKPT_DIR / "best_TemporalCNN1D.pt", CKPT_DIR / "best_EmoT.pt", CKPT_DIR / "best_EmoT_Reweighted.pt"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'training_all_models',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'training_all_models' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\nmodels_to_train = {\n    "FlatMLP": FlatMLP(T=T, D=D, n_classes=N_CLASSES),\n    "TemporalCNN1D": TemporalCNN1D(T=T, D=D, n_classes=N_CLASSES),\n    "EmoT": EmoTransformer(D=D, n_classes=N_CLASSES, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dim_ff=DIM_FF, dropout=DROPOUT, mlp_head_size=MLP_HEAD_SIZE, max_len=T),\n    "EmoT_Reweighted": EmoTransformer(D=D, n_classes=N_CLASSES, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dim_ff=DIM_FF, dropout=DROPOUT, mlp_head_size=MLP_HEAD_SIZE, max_len=T),\n}\ntrained_models, histories, ckpt_paths = {}, [], {}\ntraining_all_start = time.time()\nfor name, model in models_to_train.items():\n    print("\\n" + "=" * 80)\n    print(f"Treinando {name} | parametros: {count_parameters(model):,}")\n    loader = fair_train_loader if name == "EmoT_Reweighted" else train_loader\n    trained, hist, ckpt = fit_model(name, model, loader, X_val, y_val, criterion_weighted)\n    trained_models[name] = trained\n    histories.append(hist)\n    ckpt_paths[name] = ckpt\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nhistory_df = pd.concat(histories, ignore_index=True)\natomic_write_text(TABLE_DIR / "training_history.csv", history_df.to_csv(index=False))\natomic_write_text(LOG_DIR / f"training_history_{RUN_ID}.csv", history_df.to_csv(index=False))\ndisplay(history_df.tail())\nlog_event("training_all_models", start_time=training_all_start, n_models=int(len(models_to_train)))\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\nsns.lineplot(data=history_df, x="epoch", y="train_loss", hue="model", ax=axes[0])\naxes[0].set_title("Loss de treino")\nsns.lineplot(data=history_df, x="epoch", y="val_macro_f1", hue="model", ax=axes[1])\naxes[1].set_title("Macro-F1 de validação")\nplt.tight_layout()\nfig.savefig(FIG_DIR / "03_training_curves.png", bbox_inches="tight")\nplt.show()\n', "<cremad-stage-training_all_models>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 18. Avaliação preditiva global

Esta e a avaliação clássica: acurácia, balanced accuracy, macro-F1 e relatório por classe.


In [ ]:

evaluation_start = time.time()
predictions, global_rows = {}, []
for name, model in trained_models.items():
    metrics, pred, prob, logits = evaluate_on_arrays(model, X_test, y_test)
    predictions[name] = {"pred": pred, "prob": prob, "logits": logits}
    ci = bootstrap_metric_ci(y_test, pred, lambda yt, yp: f1_score(yt, yp, average="macro", zero_division=0))
    global_rows.append({"model": name, **metrics, "macro_f1_ci_low": ci["ci_low"], "macro_f1_ci_high": ci["ci_high"], "n_test": len(y_test), "params": count_parameters(model)})
global_results = pd.DataFrame(global_rows).sort_values("macro_f1", ascending=False)
global_results.to_csv(TABLE_DIR / "global_test_metrics.csv", index=False)
global_results.to_csv(REPORT_DIR / "global_test_metrics.csv", index=False)
display(global_results)
print("Relatorio por classe do EmoT principal:")
print(classification_report(y_test, predictions["EmoT"]["pred"], target_names=CLASS_NAMES, zero_division=0))
log_event("global_predictive_evaluation", start_time=evaluation_start, n_models=int(len(trained_models)), n_test=int(len(y_test)))


## 19. Matrizes de confusão global e por Race

As matrizes por grupo atendem diretamente ao ajuste pós-qualificação: observar erros do
EmoT por grupos demográficos.


In [ ]:

def plot_confusion(y_true, y_pred, title, path=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
    cm_plot = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) if normalize else cm
    fmt = ".2f" if normalize else "d"
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_plot, annot=True, fmt=fmt, cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Verdadeiro")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if path:
        fig.savefig(path, bbox_inches="tight")
    plt.show()

emot_pred = predictions["EmoT"]["pred"]
confusion_start = time.time()
plot_confusion(y_test, emot_pred, "Matriz de confusao global - EmoT", FIG_DIR / "04_confusion_global_emot.png")

for race in meta_test["Race"].dropna().unique():
    mask = meta_test["Race"].values == race
    if mask.sum() < 30:
        print(f"Pulando Race={race}: n={mask.sum()} menor que 30")
        continue
    safe = re.sub(r"[^A-Za-z0-9]+", "_", str(race)).strip("_")
    plot_confusion(y_test[mask], emot_pred[mask], f"Matriz de confusao - EmoT - Race={race} (n={mask.sum()})", FIG_DIR / f"05_confusion_emot_race_{safe}.png")
log_event("confusion_matrices", start_time=confusion_start, n_race_groups=int(meta_test["Race"].nunique()))


## 20. Fairness preditivo

Calculamos métricas por grupo e gaps para `Race`, `Ethnicity`, `Sex` e faixa etária.
O foco interpretativo principal deve ser `Race`.


In [ ]:

fairness_start = time.time()
fairness_group_tables, fairness_summary_tables = [], []
protected_attributes = ["Race", "Ethnicity", "Sex", "age_bin"]
for model_name, pred_obj in predictions.items():
    for attr in protected_attributes:
        group_df, summary_df = summarize_fairness(y_test, pred_obj["pred"], pred_obj["prob"], meta_test[attr].values, attr, min_n=30)
        if len(group_df):
            group_df.insert(0, "model", model_name)
            group_df.insert(1, "protected_attribute", attr)
            fairness_group_tables.append(group_df)
        if len(summary_df):
            summary_df.insert(0, "model", model_name)
            perm = permutation_gap_test(y_test, pred_obj["pred"], meta_test[attr].values, min_n=30)
            summary_df["permutation_accuracy_gap"] = perm["observed_gap"]
            summary_df["permutation_p_value"] = perm["p_value"]
            fairness_summary_tables.append(summary_df)

fairness_by_group = pd.concat(fairness_group_tables, ignore_index=True)
fairness_summary = pd.concat(fairness_summary_tables, ignore_index=True)
fairness_by_group.to_csv(TABLE_DIR / "fairness_metrics_by_group.csv", index=False)
fairness_summary.to_csv(TABLE_DIR / "fairness_summary.csv", index=False)
fairness_by_group.to_csv(FAIRNESS_DIR / "fairness_metrics_by_group.csv", index=False)
fairness_summary.to_csv(FAIRNESS_DIR / "fairness_summary.csv", index=False)
print(f"Tabelas de fairness salvas em: {FAIRNESS_DIR}")

display(fairness_by_group[fairness_by_group["protected_attribute"] == "Race"].sort_values(["model", "group"]))
display(fairness_summary.sort_values(["protected_attribute", "performance_gap_accuracy"]))

race_summary = fairness_summary[fairness_summary["protected_attribute"] == "Race"].copy()
plot_df = race_summary.melt(
    id_vars="model",
    value_vars=["performance_gap_accuracy", "equal_opportunity_gap_mean", "equalized_odds_gap_mean", "calibration_ece_gap"],
    var_name="metric",
    value_name="gap",
)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=plot_df, x="metric", y="gap", hue="model", ax=ax)
ax.set_title("Gaps de fairness por Race")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
fig.savefig(FIG_DIR / "06_fairness_gaps_by_race.png", bbox_inches="tight")
plt.show()
log_event(
    "predictive_fairness_metrics",
    start_time=fairness_start,
    n_group_rows=int(len(fairness_by_group)),
    n_summary_rows=int(len(fairness_summary)),
)


## 20.1 Fairness interseccional

A revisão bibliográfica destacou que analisar `Race`, `Sex`, `Ethnicity` e idade separadamente
pode esconder grupos mais vulneráveis. Aqui criamos grupos interseccionais, como
`Race x Sex` e `Race x faixa etária`, e calculamos as mesmas métricas quando há tamanho
amostral mínimo.

Esta análise é exploratória: grupos muito pequenos são removidos para evitar conclusões
estatisticamente frágeis.


In [ ]:

intersectional_start = time.time()
INTERSECTIONAL_MIN_N = 20
intersection_specs = {
    "Race_x_Sex": ["Race", "Sex"],
    "Race_x_Ethnicity": ["Race", "Ethnicity"],
    "Race_x_age_bin": ["Race", "age_bin"],
    "Race_x_Sex_x_age_bin": ["Race", "Sex", "age_bin"],
}

meta_test_inter = meta_test.copy()
intersection_group_tables, intersection_summary_tables = [], []
for inter_name, cols in intersection_specs.items():
    meta_test_inter[inter_name] = meta_test_inter[cols].astype(str).agg(" | ".join, axis=1)
    group_counts = meta_test_inter[inter_name].value_counts()
    eligible_groups = set(group_counts[group_counts >= INTERSECTIONAL_MIN_N].index)
    eligible_mask = meta_test_inter[inter_name].isin(eligible_groups).values
    print(f"{inter_name}: {len(eligible_groups)} grupos com n >= {INTERSECTIONAL_MIN_N}")
    if not eligible_groups:
        continue
    for model_name, pred_obj in predictions.items():
        group_df, summary_df = summarize_fairness(
            y_test[eligible_mask],
            pred_obj["pred"][eligible_mask],
            pred_obj["prob"][eligible_mask],
            meta_test_inter.loc[eligible_mask, inter_name].values,
            inter_name,
            min_n=INTERSECTIONAL_MIN_N,
        )
        if len(group_df):
            group_df.insert(0, "model", model_name)
            group_df.insert(1, "intersectional_attribute", inter_name)
            intersection_group_tables.append(group_df)
        if len(summary_df):
            summary_df.insert(0, "model", model_name)
            summary_df.insert(1, "intersectional_attribute", inter_name)
            intersection_summary_tables.append(summary_df)

if intersection_group_tables:
    fairness_intersectional = pd.concat(intersection_group_tables, ignore_index=True)
    fairness_intersectional_summary = pd.concat(intersection_summary_tables, ignore_index=True)
    fairness_intersectional.to_csv(TABLE_DIR / "fairness_intersectional_metrics.csv", index=False)
    fairness_intersectional_summary.to_csv(TABLE_DIR / "fairness_intersectional_summary.csv", index=False)
    fairness_intersectional.to_csv(FAIRNESS_DIR / "fairness_intersectional_metrics.csv", index=False)
    fairness_intersectional_summary.to_csv(FAIRNESS_DIR / "fairness_intersectional_summary.csv", index=False)
    display(fairness_intersectional.sort_values(["model", "intersectional_attribute", "accuracy"]).head(30))
    display(fairness_intersectional_summary.sort_values(["model", "performance_gap_accuracy"], ascending=[True, False]))

    plot_inter = fairness_intersectional_summary[fairness_intersectional_summary["model"] == "EmoT"].copy()
    if len(plot_inter):
        fig, ax = plt.subplots(figsize=(11, 5))
        sns.barplot(data=plot_inter, x="intersectional_attribute", y="performance_gap_accuracy", ax=ax, color="#E45756")
        ax.set_title("Gap de acurácia interseccional - EmoT")
        ax.set_xlabel("")
        ax.set_ylabel("accuracy gap")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        fig.savefig(FIG_DIR / "06b_fairness_intersectional_gaps_emot.png", bbox_inches="tight")
        plt.show()
else:
    fairness_intersectional = pd.DataFrame()
    fairness_intersectional_summary = pd.DataFrame()
    print("Nenhum grupo interseccional atingiu o tamanho mínimo.")

log_event(
    "intersectional_fairness",
    start_time=intersectional_start,
    n_rows=int(len(fairness_intersectional)),
    min_n=int(INTERSECTIONAL_MIN_N),
)


## Validação cruzada definitiva: cinco folds por ator

Cada ator aparece no teste de exatamente um fold e nunca aparece simultaneamente em treino, validação ou teste. Os quatro modelos são treinados em todos os folds. Checkpoints por época e predições por fold permitem retomar do último ponto íntegro.


In [ ]:
# Etapa durável: cross_validation_5fold
_stage_required = list([TABLE_DIR / "cv_fold_metrics.csv", TABLE_DIR / "cv_summary_mean_std.csv", TABLE_DIR / "cv_pooled_predictions.csv", TABLE_DIR / "cv_group_emotion_metrics.csv", TABLE_DIR / "cv_confusion_pairs_by_group.csv"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'cross_validation_5fold',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'cross_validation_5fold' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\ncross_validation_start = time.time()\nCV_DIR = OUTPUT_DIR / "cross_validation"\nCV_MODEL_DIR = CV_DIR / "models"\nCV_LOG_DIR = CV_DIR / "logs"\nCV_FIG_DIR = CV_DIR / "figures"\nCV_TABLE_DIR = CV_DIR / "tables"\nfor directory in [CV_DIR, CV_MODEL_DIR, CV_LOG_DIR, CV_FIG_DIR, CV_TABLE_DIR]:\n    directory.mkdir(parents=True, exist_ok=True)\n\ncv_folds, cv_actor_assignments, cv_stratification = build_actor_folds(\n    meta_valid,\n    n_splits=CV_N_SPLITS,\n    seed=SEED,\n    validation_fraction=CV_VALIDATION_FRACTION,\n)\ncv_fold_audit = validate_actor_folds(meta_valid, cv_folds)\natomic_write_text(TABLE_DIR / "cv_actor_assignments.csv", cv_actor_assignments.to_csv(index=False))\natomic_write_json(REPORT_DIR / "cv_actor_split_audit.json", cv_fold_audit)\nvalid_sample_distribution, valid_emotion_distribution = sample_distribution_tables(\n    meta_valid,\n    group_attributes=PROTECTED_ATTRIBUTES_DEFINITIVE,\n    emotion_column="label_name",\n)\natomic_write_text(TABLE_DIR / "sample_distribution_valid_by_demographic.csv", valid_sample_distribution.to_csv(index=False))\natomic_write_text(TABLE_DIR / "sample_distribution_valid_emotion_by_demographic.csv", valid_emotion_distribution.to_csv(index=False))\ndisplay(cv_actor_assignments.groupby(["outer_test_fold", "Race", "Sex"], dropna=False).size().rename("n_actors").reset_index())\nprint(f"Estratificação externa: {cv_stratification} | auditoria: {cv_fold_audit}")\n\ndef instantiate_cv_model(model_name):\n    if model_name == "FlatMLP":\n        return FlatMLP(T=T, D=D, n_classes=N_CLASSES)\n    if model_name == "TemporalCNN1D":\n        return TemporalCNN1D(T=T, D=D, n_classes=N_CLASSES)\n    if model_name in {"EmoT", "EmoT_Reweighted"}:\n        return EmoTransformer(\n            D=D,\n            n_classes=N_CLASSES,\n            d_model=D_MODEL,\n            n_heads=N_HEADS,\n            n_layers=N_LAYERS,\n            dim_ff=DIM_FF,\n            dropout=DROPOUT,\n            mlp_head_size=MLP_HEAD_SIZE,\n            max_len=T,\n        )\n    raise ValueError(f"Modelo CV desconhecido: {model_name}")\n\ncv_records = []\ncv_prediction_frames = []\ncv_history_frames = []\ncv_balance_rows = []\noriginal_ckpt_dir, original_log_dir = CKPT_DIR, LOG_DIR\nmain_mu, main_sigma = mu.copy(), sigma.copy()\nforce_cv = "cross_validation_5fold" in FORCE_STAGES\n\ntry:\n    for fold in cv_folds:\n        fold_id = int(fold["fold"])\n        fold_train_idx = np.asarray(fold["train_idx"], dtype=int)\n        fold_val_idx = np.asarray(fold["val_idx"], dtype=int)\n        fold_test_idx = np.asarray(fold["test_idx"], dtype=int)\n        fold_dir = CV_DIR / f"fold_{fold_id:02d}"\n        fold_ckpt_dir = CV_MODEL_DIR / f"fold_{fold_id:02d}"\n        fold_log_dir = CV_LOG_DIR / f"fold_{fold_id:02d}"\n        for directory in [fold_dir, fold_ckpt_dir, fold_log_dir]:\n            directory.mkdir(parents=True, exist_ok=True)\n\n        X_fold_train_raw = X[fold_train_idx]\n        X_fold_val_raw = X[fold_val_idx]\n        X_fold_test_raw = X[fold_test_idx]\n        y_fold_train = y[fold_train_idx]\n        y_fold_val = y[fold_val_idx]\n        y_fold_test = y[fold_test_idx]\n        meta_fold_train = meta_valid.iloc[fold_train_idx].reset_index(drop=True)\n\n        fold_mu = X_fold_train_raw.reshape(-1, D).mean(axis=0).astype(np.float32)\n        fold_sigma = X_fold_train_raw.reshape(-1, D).std(axis=0).astype(np.float32)\n        fold_sigma[fold_sigma == 0] = 1.0\n        X_fold_train = ((X_fold_train_raw - fold_mu) / fold_sigma).astype(np.float32)\n        X_fold_val = ((X_fold_val_raw - fold_mu) / fold_sigma).astype(np.float32)\n        X_fold_test = ((X_fold_test_raw - fold_mu) / fold_sigma).astype(np.float32)\n        atomic_npz_save(\n            fold_dir / "normalization_and_indices.npz",\n            mu=fold_mu,\n            sigma=fold_sigma,\n            train_idx=fold_train_idx,\n            val_idx=fold_val_idx,\n            test_idx=fold_test_idx,\n        )\n\n        fold_class_weights, fold_sampler_plan = balance_strategy_table(meta_fold_train)\n        fold_class_weights.insert(0, "fold", fold_id)\n        fold_sampler_plan.insert(0, "fold", fold_id)\n        cv_balance_rows.extend(fold_sampler_plan.to_dict("records"))\n\n        # Os dois loaders compartilham o mesmo TensorDataset para evitar duas\n        # cópias integrais do fold na RAM do Colab.\n        fold_dataset = TensorDataset(\n            torch.from_numpy(X_fold_train),\n            torch.from_numpy(y_fold_train.astype(np.int64, copy=False)),\n        )\n        standard_loader = DataLoader(\n            fold_dataset,\n            batch_size=BATCH_SIZE,\n            shuffle=True,\n            num_workers=0,\n            pin_memory=torch.cuda.is_available(),\n        )\n        mitigated_loader = DataLoader(\n            fold_dataset,\n            batch_size=BATCH_SIZE,\n            shuffle=False,\n            sampler=make_group_label_sampler(meta_fold_train),\n            num_workers=0,\n            pin_memory=torch.cuda.is_available(),\n        )\n        fold_criterion = nn.CrossEntropyLoss(\n            weight=class_weights_from_y(y_fold_train),\n            label_smoothing=LABEL_SMOOTHING,\n        )\n\n        CKPT_DIR, LOG_DIR, mu, sigma = fold_ckpt_dir, fold_log_dir, fold_mu, fold_sigma\n        for model_name in CV_MODELS:\n            ledger_model_name = f"CV_F{fold_id:02d}_{model_name}"\n            prediction_cache = fold_dir / f"predictions_{model_name}.npz"\n            if file_is_valid(prediction_cache) and not force_cv:\n                with np.load(prediction_cache, allow_pickle=False) as cached:\n                    fold_pred = cached["y_pred"].astype(int)\n                    fold_prob = cached["y_prob"].astype(np.float32)\n                    fold_logits = cached["logits"].astype(np.float32)\n                    cached_test_idx = cached["test_idx"].astype(int)\n                    cached_y_true = cached["y_true"].astype(int)\n                if not np.array_equal(cached_test_idx, fold_test_idx) or not np.array_equal(cached_y_true, y_fold_test):\n                    raise RuntimeError(f"Cache CV incompatível: {prediction_cache}")\n                history_path = fold_log_dir / f"training_history_{ledger_model_name}.csv"\n                if file_is_valid(history_path):\n                    hist = pd.read_csv(history_path)\n                    hist["display_model"] = model_name\n                    hist["fold"] = fold_id\n                    cv_history_frames.append(hist)\n                print(f"Fold {fold_id}/{CV_N_SPLITS} | {model_name}: predição retomada do cache")\n            else:\n                if force_cv:\n                    FORCE_STAGES.add(f"train_{ledger_model_name}")\n                model = instantiate_cv_model(model_name)\n                loader = mitigated_loader if model_name == "EmoT_Reweighted" else standard_loader\n                trained, hist, _ = fit_model(\n                    ledger_model_name,\n                    model,\n                    loader,\n                    X_fold_val,\n                    y_fold_val,\n                    fold_criterion,\n                )\n                fold_pred, fold_prob, fold_logits = predict_model(trained, X_fold_test)\n                hist["display_model"] = model_name\n                hist["fold"] = fold_id\n                cv_history_frames.append(hist)\n                atomic_npz_save(\n                    prediction_cache,\n                    test_idx=fold_test_idx,\n                    y_true=y_fold_test,\n                    y_pred=fold_pred,\n                    y_prob=fold_prob,\n                    logits=fold_logits,\n                )\n                del trained, model\n                gc.collect()\n                if torch.cuda.is_available():\n                    torch.cuda.empty_cache()\n\n            cv_records.append(\n                {\n                    "model": model_name,\n                    "fold": fold_id,\n                    "y_true": y_fold_test.copy(),\n                    "y_pred": fold_pred.copy(),\n                }\n            )\n            prediction_frame = pd.DataFrame(\n                {\n                    "sample_idx": fold_test_idx,\n                    "fold": fold_id,\n                    "model": model_name,\n                    "y_true": y_fold_test,\n                    "y_pred": fold_pred,\n                }\n            )\n            for class_id, class_name in enumerate(CLASS_NAMES):\n                prediction_frame[f"prob_{class_name}"] = fold_prob[:, class_id]\n            cv_prediction_frames.append(prediction_frame)\n            LEDGER.register_artifact(RUN_ID, "cross_validation_5fold", prediction_cache, kind="cv_predictions")\n\n        del X_fold_train_raw, X_fold_val_raw, X_fold_test_raw\n        del X_fold_train, X_fold_val, X_fold_test\n        del fold_dataset, standard_loader, mitigated_loader, fold_criterion\n        gc.collect()\nfinally:\n    CKPT_DIR, LOG_DIR, mu, sigma = original_ckpt_dir, original_log_dir, main_mu, main_sigma\n\ncv_fold_metrics, cv_summary, cv_confusions = aggregate_cv_results(\n    cv_records,\n    labels=list(range(N_CLASSES)),\n)\ncv_pooled_predictions = pd.concat(cv_prediction_frames, ignore_index=True).sort_values(["model", "sample_idx"])\ncv_training_history = pd.concat(cv_history_frames, ignore_index=True) if cv_history_frames else pd.DataFrame()\ncv_balance_by_fold = pd.DataFrame(cv_balance_rows)\n\nexpected_rows = len(meta_valid) * len(CV_MODELS)\nif len(cv_pooled_predictions) != expected_rows:\n    raise RuntimeError(f"Predições CV incompletas: esperado={expected_rows}, obtido={len(cv_pooled_predictions)}")\ncoverage = cv_pooled_predictions.groupby("model")["sample_idx"].agg(["count", "nunique"])\nif not ((coverage["count"] == len(meta_valid)) & (coverage["nunique"] == len(meta_valid))).all():\n    raise RuntimeError(f"Cobertura CV inválida:\\n{coverage}")\n\natomic_write_text(TABLE_DIR / "cv_fold_metrics.csv", cv_fold_metrics.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_summary_mean_std.csv", cv_summary.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_pooled_predictions.csv", cv_pooled_predictions.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_training_history.csv", cv_training_history.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_balance_by_fold.csv", cv_balance_by_fold.to_csv(index=False))\n\nfor model_name, matrices in cv_confusions.items():\n    safe_model = safe_slug(model_name)\n    for matrix_name, matrix in matrices.items():\n        matrix_frame = pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES)\n        atomic_write_text(TABLE_DIR / f"cv_confusion_{matrix_name}_{safe_model}.csv", matrix_frame.to_csv())\n    annotations = np.empty_like(matrices["mean"], dtype=object)\n    for row_idx in range(N_CLASSES):\n        for col_idx in range(N_CLASSES):\n            annotations[row_idx, col_idx] = f"{matrices[\'mean\'][row_idx, col_idx]:.2f}\\n±{matrices[\'std\'][row_idx, col_idx]:.2f}"\n    fig, ax = plt.subplots(figsize=(9, 7))\n    sns.heatmap(\n        matrices["mean"],\n        annot=annotations,\n        fmt="",\n        cmap="Blues",\n        vmin=0,\n        vmax=1,\n        xticklabels=CLASS_NAMES,\n        yticklabels=CLASS_NAMES,\n        ax=ax,\n    )\n    ax.set_title(f"5-fold: matriz de confusão média ± DP — {model_name}")\n    ax.set_xlabel("Predito")\n    ax.set_ylabel("Verdadeiro")\n    plt.xticks(rotation=35, ha="right")\n    plt.tight_layout()\n    fig.savefig(FIG_DIR / f"cv_confusion_mean_{safe_model}.png", bbox_inches="tight")\n    fig.savefig(CV_FIG_DIR / f"cv_confusion_mean_{safe_model}.png", bbox_inches="tight")\n    plt.show()\n\ncv_group_metric_frames = []\ncv_group_metric_by_fold_frames = []\ncv_group_emotion_frames = []\ncv_confusion_pair_frames = []\nfor cv_model_name in CV_MODELS:\n    pooled_model = cv_pooled_predictions[cv_pooled_predictions["model"] == cv_model_name].sort_values("sample_idx").reset_index(drop=True)\n    pooled_model_meta = meta_valid.iloc[pooled_model["sample_idx"].to_numpy(dtype=int)].reset_index(drop=True)\n    pooled_model_y_true = pooled_model["y_true"].to_numpy(dtype=int)\n    pooled_model_y_pred = pooled_model["y_pred"].to_numpy(dtype=int)\n    pooled_model_prob = pooled_model[[f"prob_{name}" for name in CLASS_NAMES]].to_numpy(dtype=float)\n\n    for attribute in PROTECTED_ATTRIBUTES_DEFINITIVE:\n        group_table = metrics_by_group(\n            pooled_model_y_true,\n            pooled_model_y_pred,\n            pooled_model_prob,\n            pooled_model_meta[attribute].values,\n            min_n=CV_MIN_GROUP_N,\n        )\n        if len(group_table):\n            group_table.insert(0, "model", cv_model_name)\n            group_table.insert(1, "protected_attribute", attribute)\n            cv_group_metric_frames.append(group_table)\n\n    model_emotion_table = group_emotion_metrics(\n        pooled_model_meta,\n        pooled_model_y_true,\n        pooled_model_y_pred,\n        group_attributes=PROTECTED_ATTRIBUTES_DEFINITIVE,\n        class_names=CLASS_NAMES,\n        min_support=CV_MIN_EMOTION_SUPPORT,\n    )\n    model_emotion_table.insert(0, "model", cv_model_name)\n    cv_group_emotion_frames.append(model_emotion_table)\n\n    model_confusion_pairs = confusion_pairs_by_group(\n        pooled_model_meta,\n        pooled_model_y_true,\n        pooled_model_y_pred,\n        group_attributes=PROTECTED_ATTRIBUTES_DEFINITIVE,\n        class_names=CLASS_NAMES,\n        min_true_support=CV_MIN_EMOTION_SUPPORT,\n    )\n    if len(model_confusion_pairs):\n        model_confusion_pairs.insert(0, "model", cv_model_name)\n        cv_confusion_pair_frames.append(model_confusion_pairs)\n\n    for fold_id in range(1, CV_N_SPLITS + 1):\n        fold_predictions = pooled_model[pooled_model["fold"] == fold_id].sort_values("sample_idx").reset_index(drop=True)\n        fold_metadata = meta_valid.iloc[fold_predictions["sample_idx"].to_numpy(dtype=int)].reset_index(drop=True)\n        fold_y_true = fold_predictions["y_true"].to_numpy(dtype=int)\n        fold_y_pred = fold_predictions["y_pred"].to_numpy(dtype=int)\n        fold_prob = fold_predictions[[f"prob_{name}" for name in CLASS_NAMES]].to_numpy(dtype=float)\n        for attribute in PROTECTED_ATTRIBUTES_DEFINITIVE:\n            fold_group_table = metrics_by_group(\n                fold_y_true,\n                fold_y_pred,\n                fold_prob,\n                fold_metadata[attribute].values,\n                min_n=max(10, CV_MIN_GROUP_N // CV_N_SPLITS),\n            )\n            if len(fold_group_table):\n                fold_group_table.insert(0, "model", cv_model_name)\n                fold_group_table.insert(1, "fold", fold_id)\n                fold_group_table.insert(2, "protected_attribute", attribute)\n                cv_group_metric_by_fold_frames.append(fold_group_table)\n\ncv_group_metrics = pd.concat(cv_group_metric_frames, ignore_index=True)\ncv_group_metrics_by_fold = pd.concat(cv_group_metric_by_fold_frames, ignore_index=True)\ncv_group_emotion_metrics = pd.concat(cv_group_emotion_frames, ignore_index=True)\ncv_confusion_pairs = pd.concat(cv_confusion_pair_frames, ignore_index=True)\ncv_group_gap_summary = cv_group_metrics.groupby(["model", "protected_attribute"]).agg(\n    n_groups=("group", "nunique"),\n    worst_group_accuracy=("accuracy", "min"),\n    best_group_accuracy=("accuracy", "max"),\n    worst_group_macro_f1=("macro_f1", "min"),\n    best_group_macro_f1=("macro_f1", "max"),\n).reset_index()\ncv_group_gap_summary["accuracy_gap"] = cv_group_gap_summary["best_group_accuracy"] - cv_group_gap_summary["worst_group_accuracy"]\ncv_group_gap_summary["macro_f1_gap"] = cv_group_gap_summary["best_group_macro_f1"] - cv_group_gap_summary["worst_group_macro_f1"]\ncv_group_metrics_fold_summary = cv_group_metrics_by_fold.groupby(["model", "protected_attribute", "group"]).agg(\n    n_folds=("fold", "nunique"),\n    n_mean=("n", "mean"),\n    accuracy_mean=("accuracy", "mean"),\n    accuracy_std=("accuracy", "std"),\n    macro_f1_mean=("macro_f1", "mean"),\n    macro_f1_std=("macro_f1", "std"),\n    ece_mean=("ece", "mean"),\n    ece_std=("ece", "std"),\n).reset_index()\n\npooled_emot = cv_pooled_predictions[cv_pooled_predictions["model"] == "EmoT"].sort_values("sample_idx").reset_index(drop=True)\npooled_emot_meta = meta_valid.iloc[pooled_emot["sample_idx"].to_numpy(dtype=int)].reset_index(drop=True)\npooled_emot_y_true = pooled_emot["y_true"].to_numpy(dtype=int)\npooled_emot_y_pred = pooled_emot["y_pred"].to_numpy(dtype=int)\npooled_emot_prob = pooled_emot[[f"prob_{name}" for name in CLASS_NAMES]].to_numpy(dtype=float)\ncv_top_confusions = cv_confusion_pairs[cv_confusion_pairs["model"] == "EmoT"].groupby(["protected_attribute", "group"], group_keys=False).head(TOP_CONFUSIONS_PER_GROUP)\n\natomic_write_text(TABLE_DIR / "cv_group_metrics.csv", cv_group_metrics.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_group_metrics_by_fold.csv", cv_group_metrics_by_fold.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_group_metrics_fold_summary.csv", cv_group_metrics_fold_summary.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_group_gap_summary.csv", cv_group_gap_summary.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_group_emotion_metrics.csv", cv_group_emotion_metrics.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_confusion_pairs_by_group.csv", cv_confusion_pairs.to_csv(index=False))\natomic_write_text(TABLE_DIR / "cv_top_confusions_to_discuss.csv", cv_top_confusions.to_csv(index=False))\n\nfor attribute in PROTECTED_ATTRIBUTES_DEFINITIVE:\n    attribute_values = pooled_emot_meta[attribute].fillna("Unknown").astype(str).to_numpy()\n    for group in pd.unique(attribute_values):\n        group_mask = attribute_values == group\n        if int(group_mask.sum()) < CV_MIN_GROUP_N:\n            continue\n        safe_group = safe_slug(group)\n        cm = confusion_matrix(\n            pooled_emot_y_true[group_mask],\n            pooled_emot_y_pred[group_mask],\n            labels=list(range(N_CLASSES)),\n        ).astype(float)\n        cm = np.divide(cm, cm.sum(axis=1, keepdims=True), out=np.zeros_like(cm), where=cm.sum(axis=1, keepdims=True) > 0)\n        cm_frame = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)\n        atomic_write_text(TABLE_DIR / f"cv_confusion_EmoT_{safe_slug(attribute)}_{safe_group}.csv", cm_frame.to_csv())\n        fig, ax = plt.subplots(figsize=(8, 6))\n        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)\n        ax.set_title(f"EmoT 5-fold — {attribute}={group} (n={int(group_mask.sum())})")\n        ax.set_xlabel("Predito")\n        ax.set_ylabel("Verdadeiro")\n        plt.xticks(rotation=35, ha="right")\n        plt.tight_layout()\n        fig.savefig(FIG_DIR / f"cv_confusion_EmoT_{safe_slug(attribute)}_{safe_group}.png", bbox_inches="tight")\n        plt.close(fig)\n\nemot_comparison = cv_summary[cv_summary["model"].isin(["EmoT", "EmoT_Reweighted"])].copy()\nif set(emot_comparison["model"]) == {"EmoT", "EmoT_Reweighted"}:\n    baseline_row = emot_comparison.set_index("model").loc["EmoT"]\n    mitigated_row = emot_comparison.set_index("model").loc["EmoT_Reweighted"]\n    race_gaps = cv_group_gap_summary[cv_group_gap_summary["protected_attribute"] == "Race"].set_index("model")\n    race_accuracy_gap_delta = (\n        race_gaps.loc["EmoT_Reweighted", "accuracy_gap"] - race_gaps.loc["EmoT", "accuracy_gap"]\n        if {"EmoT", "EmoT_Reweighted"}.issubset(race_gaps.index)\n        else np.nan\n    )\n    race_macro_f1_gap_delta = (\n        race_gaps.loc["EmoT_Reweighted", "macro_f1_gap"] - race_gaps.loc["EmoT", "macro_f1_gap"]\n        if {"EmoT", "EmoT_Reweighted"}.issubset(race_gaps.index)\n        else np.nan\n    )\n    balance_mitigation_comparison = pd.DataFrame([\n        {\n            "comparison": "EmoT_Reweighted - EmoT",\n            "macro_f1_delta": mitigated_row["macro_f1_mean"] - baseline_row["macro_f1_mean"],\n            "balanced_accuracy_delta": mitigated_row["balanced_accuracy_mean"] - baseline_row["balanced_accuracy_mean"],\n            "accuracy_delta": mitigated_row["accuracy_mean"] - baseline_row["accuracy_mean"],\n            "race_accuracy_gap_delta": race_accuracy_gap_delta,\n            "race_macro_f1_gap_delta": race_macro_f1_gap_delta,\n            "interpretation_rule": "A mitigação é favorável somente se os gaps diminuírem sem perda global substantivamente inaceitável.",\n        }\n    ])\nelse:\n    balance_mitigation_comparison = pd.DataFrame()\natomic_write_text(TABLE_DIR / "balance_mitigation_cv_comparison.csv", balance_mitigation_comparison.to_csv(index=False))\n\nliterature_template = pd.DataFrame([\n    {\n        "study": "Esta dissertação — EmoT",\n        "year": time.strftime("%Y"),\n        "dataset": "CREMA-D",\n        "modality": "visual-only landmarks",\n        "identity_disjoint": True,\n        "evaluation_protocol": "5-fold actor-disjoint CV",\n        "metric_name": "macro_f1",\n        "metric_mean": float(cv_summary.set_index("model").loc["EmoT", "macro_f1_mean"]),\n        "metric_std": float(cv_summary.set_index("model").loc["EmoT", "macro_f1_std"]),\n        "source_doi_or_url": "resultado desta execução",\n        "directly_comparable": True,\n        "validation_status": "generated",\n    },\n    {\n        "study": "PREENCHER_APENAS_COM_FONTE_PRIMARIA_VALIDADA",\n        "year": np.nan,\n        "dataset": "CREMA-D",\n        "modality": "visual-only required",\n        "identity_disjoint": np.nan,\n        "evaluation_protocol": "documentar split/folds",\n        "metric_name": "documentar exatamente",\n        "metric_mean": np.nan,\n        "metric_std": np.nan,\n        "source_doi_or_url": "obrigatório",\n        "directly_comparable": False,\n        "validation_status": "pending documentary validation",\n    },\n])\natomic_write_text(TABLE_DIR / "literature_visual_only_comparison_template.csv", literature_template.to_csv(index=False))\n\nobservation_lines = [\n    "# Observações quantitativas para discussão",\n    "",\n    "As linhas abaixo são achados descritivos gerados automaticamente. Não implicam causalidade e devem ser interpretadas com suporte teórico e validação humana.",\n    "",\n]\nfor row in cv_top_confusions.itertuples(index=False):\n    observation_lines.append(\n        f"- {row.protected_attribute}={row.group}: {row.true_emotion} confundida com {row.predicted_emotion} "\n        f"em {100 * row.error_rate_within_true:.1f}% dos casos verdadeiros (n={row.count}/{row.true_emotion_support})."\n    )\natomic_write_text(REPORT_DIR / "cv_observations_to_discuss.md", "\\n".join(observation_lines) + "\\n")\n\ndisplay(cv_fold_metrics)\ndisplay(cv_summary)\ndisplay(cv_group_metrics)\ndisplay(cv_group_emotion_metrics.head(60))\ndisplay(cv_top_confusions.head(60))\ndisplay(balance_mitigation_comparison)\nlog_event(\n    "cross_validation_5fold",\n    start_time=cross_validation_start,\n    n_folds=int(CV_N_SPLITS),\n    n_models=int(len(CV_MODELS)),\n    n_training_jobs=int(CV_N_SPLITS * len(CV_MODELS)),\n    outer_stratification=cv_stratification,\n)\n', "<cremad-stage-cross_validation_5fold>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 21. Robustez por alto consenso facial

O experimento principal usa o label original. Aqui avaliamos apenas exemplos com
`face_agreement >= 0.70` para verificar se desempenho e gaps permanecem quando a
ambiguidade perceptual é menor.


In [ ]:

robustness_start = time.time()
high_mask = meta_test["face_agreement"].astype(float).values >= HIGH_CONSENSUS_THRESHOLD
print(f"Teste completo: {len(y_test)}")
print(f"Alto consenso facial: {high_mask.sum()} ({100*high_mask.mean():.1f}%)")
robust_rows = []
for model_name, obj in predictions.items():
    yt = y_test[high_mask]
    pred = obj["pred"][high_mask]
    prob = obj["prob"][high_mask]
    row = {"model": model_name, "subset": "high_face_consensus", "n": len(yt), **compute_basic_metrics(yt, pred)}
    _, fair_sum = summarize_fairness(yt, pred, prob, meta_test.loc[high_mask, "Race"].values, "Race", min_n=20)
    if len(fair_sum):
        row.update({f"race_{k}": v for k, v in fair_sum.iloc[0].to_dict().items() if k != "protected_attribute"})
    robust_rows.append(row)
for _, row in global_results.iterrows():
    robust_rows.append({"model": row["model"], "subset": "full_test", "n": int(row["n_test"]), "accuracy": row["accuracy"], "balanced_accuracy": row["balanced_accuracy"], "macro_f1": row["macro_f1"], "weighted_f1": row["weighted_f1"]})
robustness_df = pd.DataFrame(robust_rows)
robustness_df.to_csv(TABLE_DIR / "robustness_high_consensus.csv", index=False)
robustness_df.to_csv(FAIRNESS_DIR / "robustness_high_consensus.csv", index=False)
display(robustness_df.sort_values(["model", "subset"]))
log_event("high_consensus_robustness", start_time=robustness_start, n_high_consensus=int(high_mask.sum()), n_test=int(len(y_test)))


## 22. XAI - Attention Rollout temporal

O Attention Rollout acumula a atenção das camadas do EmoT. Extraímos a atenção do token
`CLS` para cada frame, produzindo uma curva temporal de importância.


In [ ]:
# Etapa durável: xai_attention_rollout
_stage_required = list([XAI_DIR / "xai_attention_rollout_emot.npz"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'xai_attention_rollout',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'xai_attention_rollout' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\ndef select_xai_indices(meta_df, max_samples=None, seed=SEED):\n    if max_samples is None or max_samples >= len(meta_df):\n        return np.arange(len(meta_df))\n    rng = np.random.default_rng(seed)\n    selected = []\n    grouped = meta_df.groupby(["Race", "label_code"], dropna=False)\n    per_group = max(1, max_samples // max(1, grouped.ngroups))\n    for _, idx in grouped.indices.items():\n        idx = np.array(list(idx))\n        selected.extend(rng.choice(idx, size=min(len(idx), per_group), replace=False).tolist())\n    if len(selected) < max_samples:\n        remaining = np.setdiff1d(np.arange(len(meta_df)), np.array(selected))\n        selected.extend(rng.choice(remaining, size=min(len(remaining), max_samples-len(selected)), replace=False).tolist())\n    return np.array(sorted(selected[:max_samples]))\n\ndef attention_rollout(attentions):\n    result = None\n    for attn in attentions:\n        attn_mean = attn.mean(dim=1)\n        eye = torch.eye(attn_mean.size(-1), device=attn_mean.device).unsqueeze(0).expand_as(attn_mean)\n        attn_res = 0.5 * attn_mean + 0.5 * eye\n        attn_res = attn_res / attn_res.sum(dim=-1, keepdim=True).clamp_min(1e-8)\n        result = attn_res if result is None else torch.bmm(attn_res, result)\n    return result\n\n@torch.no_grad()\ndef compute_attention_importance(model, X_np, batch_size=XAI_BATCH_SIZE):\n    model.eval()\n    curves, preds, probs = [], [], []\n    for start in tqdm(range(0, len(X_np), batch_size), desc="Attention Rollout"):\n        xb = torch.tensor(X_np[start:start+batch_size], dtype=torch.float32, device=DEVICE)\n        logits, attentions = model(xb, return_attention=True)\n        prob = F.softmax(logits, dim=1)\n        rollout = attention_rollout(attentions)\n        cls_to_frames = rollout[:, 0, 1:]\n        cls_to_frames = cls_to_frames / cls_to_frames.sum(dim=1, keepdim=True).clamp_min(1e-8)\n        curves.append(cls_to_frames.detach().cpu().numpy())\n        preds.append(prob.argmax(dim=1).detach().cpu().numpy())\n        probs.append(prob.detach().cpu().numpy())\n    return np.concatenate(curves), np.concatenate(preds), np.concatenate(probs)\n\nxai_idx = select_xai_indices(meta_test, MAX_XAI_SAMPLES)\nX_xai, y_xai = X_test[xai_idx], y_test[xai_idx]\nmeta_xai = meta_test.iloc[xai_idx].reset_index(drop=True)\nemot_model = trained_models["EmoT"].to(DEVICE)\nxai_attention_start = time.time()\nATTENTION_CACHE = XAI_DIR / "xai_attention_rollout_emot.npz"\nif file_is_valid(ATTENTION_CACHE) and "xai_attention_rollout" not in FORCE_STAGES:\n    attention_data = np.load(ATTENTION_CACHE, allow_pickle=False)\n    attention_curves = attention_data["attention_curves"]\n    xai_preds = attention_data["xai_preds"]\n    xai_probs = attention_data["xai_probs"]\n    print(f"Attention Rollout retomado do cache: {ATTENTION_CACHE}")\nelse:\n    attention_curves, xai_preds, xai_probs = compute_attention_importance(emot_model, X_xai)\n    atomic_npz_save(ATTENTION_CACHE, attention_curves=attention_curves, xai_preds=xai_preds, xai_probs=xai_probs, xai_idx=xai_idx, y_xai=y_xai)\nprint(f"Amostras XAI: {len(xai_idx)}")\nprint(f"attention_curves: {attention_curves.shape}")\natomic_write_text(XAI_DIR / "xai_sample_manifest.csv", meta_xai.to_csv(index=False))\nprint(f"Artefatos de Attention Rollout salvos em: {XAI_DIR}")\n\ndef normalized_entropy(p, axis=1):\n    p = p / np.maximum(p.sum(axis=axis, keepdims=True), 1e-12)\n    ent = -(p * np.log(np.maximum(p, 1e-12))).sum(axis=axis)\n    return ent / np.log(p.shape[axis])\n\nmeta_xai["attention_entropy"] = normalized_entropy(attention_curves, axis=1)\nfig, ax = plt.subplots(figsize=(11, 5))\nfor race, group in meta_xai.groupby("Race"):\n    if len(group) < 10:\n        continue\n    curve = attention_curves[group.index.values].mean(axis=0)\n    ax.plot(np.arange(T_FRAMES), curve, marker="o", label=f"{race} (n={len(group)})")\nax.set_title("Attention Rollout temporal médio por Race - EmoT")\nax.set_xlabel("frame amostrado")\nax.set_ylabel("importância normalizada")\nax.legend()\nplt.tight_layout()\nfig.savefig(FIG_DIR / "07_xai_temporal_attention_by_race.png", bbox_inches="tight")\nplt.show()\ndisplay(meta_xai.groupby("Race").agg(n=("file_stem", "count"), mean_entropy=("attention_entropy", "mean"), std_entropy=("attention_entropy", "std")).reset_index())\nlog_event("xai_attention_rollout", start_time=xai_attention_start, n_xai_samples=int(len(xai_idx)), attention_shape=str(attention_curves.shape))\n', "<cremad-stage-xai_attention_rollout>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 23. XAI - Gradient x Input por landmark

Gradient x Input estima a contribuicao das features de entrada para o logit explicado.
Agregamos por landmark e por regiao facial para interpretar **onde** o EmoT decidiu.


In [ ]:
# Etapa durável: xai_gradient_x_input
_stage_required = list([XAI_DIR / "xai_gradient_landmark_scores_emot.npz"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'xai_gradient_x_input',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'xai_gradient_x_input' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\ndef compute_grad_x_input(model, X_np, target_classes, batch_size=16):\n    model.eval()\n    all_attr = []\n    for start in tqdm(range(0, len(X_np), batch_size), desc="Gradient x Input"):\n        xb = torch.tensor(X_np[start:start+batch_size], dtype=torch.float32, device=DEVICE, requires_grad=True)\n        target = torch.tensor(target_classes[start:start+batch_size], dtype=torch.long, device=DEVICE)\n        logits = model(xb)\n        selected = logits[torch.arange(xb.size(0), device=DEVICE), target].sum()\n        model.zero_grad(set_to_none=True)\n        selected.backward()\n        all_attr.append((xb.grad * xb).abs().detach().cpu().numpy())\n        del xb, logits, selected\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    return np.concatenate(all_attr, axis=0)\n\ndef aggregate_attr_to_landmarks(attr):\n    attr_mean = attr.mean(axis=1)\n    n_blocks = len(FEATURE_BLOCKS)\n    scores = np.zeros((attr.shape[0], N_LANDMARKS), dtype=np.float64)\n    for b in range(n_blocks):\n        block = attr_mean[:, b * BASE_FEATURE_DIM : (b + 1) * BASE_FEATURE_DIM]\n        block = block.reshape(attr.shape[0], N_LANDMARKS, 2).mean(axis=2)\n        scores += block\n    return scores / np.maximum(scores.sum(axis=1, keepdims=True), 1e-12)\n\ndef aggregate_landmarks_to_regions(landmark_scores):\n    selected_to_pos = {lm_id: pos for pos, lm_id in enumerate(SELECTED_LANDMARK_IDS)}\n    rows = []\n    for region, ids in REGION_IDS.items():\n        positions = [selected_to_pos[i] for i in ids if i in selected_to_pos]\n        if positions:\n            values = landmark_scores[:, positions].sum(axis=1)\n            rows.append(pd.DataFrame({"sample_idx": np.arange(len(values)), "region": region, "importance": values}))\n    return pd.concat(rows, ignore_index=True)\n\ngrad_x_input_start = time.time()\nGRADIENT_CACHE = XAI_DIR / "xai_gradient_landmark_scores_emot.npz"\nif file_is_valid(GRADIENT_CACHE) and "xai_gradient_x_input" not in FORCE_STAGES:\n    gradient_data = np.load(GRADIENT_CACHE, allow_pickle=False)\n    grad_attr = gradient_data["grad_attr"]\n    landmark_scores = gradient_data["landmark_scores"]\n    print(f"Gradient x Input retomado do cache: {GRADIENT_CACHE}")\nelse:\n    grad_attr = compute_grad_x_input(emot_model, X_xai, xai_preds, batch_size=16)\n    landmark_scores = aggregate_attr_to_landmarks(grad_attr)\n    atomic_npz_save(GRADIENT_CACHE, grad_attr=grad_attr, landmark_scores=landmark_scores, selected_landmark_ids=np.asarray(SELECTED_LANDMARK_IDS, dtype=np.int64))\nregion_importance = aggregate_landmarks_to_regions(landmark_scores)\nregion_importance = region_importance.merge(meta_xai[["Race", "Ethnicity", "Sex", "label_name"]].reset_index(names="sample_idx"), on="sample_idx", how="left")\nregion_summary = region_importance.groupby(["Race", "region"]).agg(n=("importance", "count"), mean_importance=("importance", "mean"), std_importance=("importance", "std")).reset_index()\nregion_summary.to_csv(TABLE_DIR / "xai_region_importance_by_race.csv", index=False)\nregion_summary.to_csv(XAI_DIR / "xai_region_importance_by_race.csv", index=False)\ndisplay(region_summary)\n\nfig, ax = plt.subplots(figsize=(13, 6))\nsns.barplot(data=region_summary, x="region", y="mean_importance", hue="Race", ax=ax)\nax.set_title("Gradient x Input: importância media por regiao facial e Race")\nplt.xticks(rotation=30, ha="right")\nplt.tight_layout()\nfig.savefig(FIG_DIR / "08_xai_region_importance_by_race.png", bbox_inches="tight")\nplt.show()\nlog_event("xai_gradient_x_input", start_time=grad_x_input_start, n_xai_samples=int(len(X_xai)), landmark_scores_shape=str(landmark_scores.shape))\n', "<cremad-stage-xai_gradient_x_input>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 23.1 Integrated Gradients e importância estático-dinâmica

A revisão bibliográfica sugere comparar explicações por técnica e verificar se o modelo
depende mais de landmarks estáticos (`pos`) ou de dinâmica expressiva (`vel`, `acc`, `jerk`).

Como o modelo usa landmarks, não usamos Grad-CAM como técnica principal. Grad-CAM é mais
natural para CNNs sobre imagem. Aqui, a alternativa mais coerente é Integrated Gradients
diretamente sobre as features temporais de landmarks.


In [ ]:
# Etapa durável: xai_integrated_gradients
_stage_required = list([XAI_DIR / "xai_integrated_gradients_emot.npz"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'xai_integrated_gradients',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'xai_integrated_gradients' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\ndef compute_integrated_gradients(model, X_np, target_classes, steps=IG_STEPS, batch_size=16):\n    model.eval()\n    attrs = []\n    baseline_np = np.zeros_like(X_np, dtype=np.float32)\n    alphas = torch.linspace(0.0, 1.0, steps + 1, device=DEVICE)[1:]\n    for start in tqdm(range(0, len(X_np), batch_size), desc="Integrated Gradients"):\n        x = torch.tensor(X_np[start:start+batch_size], dtype=torch.float32, device=DEVICE)\n        baseline = torch.tensor(baseline_np[start:start+batch_size], dtype=torch.float32, device=DEVICE)\n        target = torch.tensor(target_classes[start:start+batch_size], dtype=torch.long, device=DEVICE)\n        total_grad = torch.zeros_like(x)\n        for alpha in alphas:\n            x_step = (baseline + alpha * (x - baseline)).detach().requires_grad_(True)\n            logits = model(x_step)\n            selected = logits[torch.arange(x_step.size(0), device=DEVICE), target].sum()\n            model.zero_grad(set_to_none=True)\n            selected.backward()\n            total_grad += x_step.grad.detach()\n        avg_grad = total_grad / len(alphas)\n        attr = ((x - baseline) * avg_grad).abs().detach().cpu().numpy()\n        attrs.append(attr)\n        del x, baseline, target, total_grad, avg_grad\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    return np.concatenate(attrs, axis=0)\n\ndef feature_block_importance(attr):\n    rows = []\n    for block_idx, block_name in enumerate(FEATURE_BLOCKS):\n        start_d = block_idx * BASE_FEATURE_DIM\n        end_d = (block_idx + 1) * BASE_FEATURE_DIM\n        values = attr[:, :, start_d:end_d].mean(axis=(1, 2))\n        rows.append(pd.DataFrame({"sample_idx": np.arange(len(values)), "feature_block": block_name, "importance": values}))\n    out = pd.concat(rows, ignore_index=True)\n    denom = out.groupby("sample_idx")["importance"].transform("sum").replace(0, np.nan)\n    out["importance_norm"] = out["importance"] / denom\n    return out.fillna(0)\n\nig_start = time.time()\nig_n = min(len(X_xai), MAX_IG_SAMPLES)\nig_indices = select_xai_indices(meta_xai, ig_n, seed=SEED) if ig_n < len(X_xai) else np.arange(len(X_xai))\nX_ig = X_xai[ig_indices]\ny_ig = y_xai[ig_indices]\nmeta_ig = meta_xai.iloc[ig_indices].reset_index(drop=True)\nxai_preds_ig = xai_preds[ig_indices]\n\nIG_CACHE = XAI_DIR / "xai_integrated_gradients_emot.npz"\nif file_is_valid(IG_CACHE) and "xai_integrated_gradients" not in FORCE_STAGES:\n    ig_data = np.load(IG_CACHE, allow_pickle=False)\n    ig_attr = ig_data["ig_attr"]\n    ig_landmark_scores = ig_data["ig_landmark_scores"]\n    ig_indices = ig_data["ig_indices"].astype(int)\n    X_ig, y_ig = X_xai[ig_indices], y_xai[ig_indices]\n    meta_ig = meta_xai.iloc[ig_indices].reset_index(drop=True)\n    xai_preds_ig = xai_preds[ig_indices]\n    print(f"Integrated Gradients retomado do cache: {IG_CACHE}")\nelse:\n    ig_attr = compute_integrated_gradients(emot_model, X_ig, xai_preds_ig, steps=IG_STEPS, batch_size=16)\n    ig_landmark_scores = aggregate_attr_to_landmarks(ig_attr)\n    atomic_npz_save(IG_CACHE, ig_attr=ig_attr, ig_landmark_scores=ig_landmark_scores, ig_indices=ig_indices, selected_landmark_ids=np.asarray(SELECTED_LANDMARK_IDS, dtype=np.int64))\nig_region_importance = aggregate_landmarks_to_regions(ig_landmark_scores)\nig_region_importance = ig_region_importance.merge(meta_ig[["Race", "Ethnicity", "Sex", "label_name"]].reset_index(names="sample_idx"), on="sample_idx", how="left")\nig_region_summary = ig_region_importance.groupby(["Race", "region"]).agg(n=("importance", "count"), mean_importance=("importance", "mean"), std_importance=("importance", "std")).reset_index()\nig_region_summary.to_csv(TABLE_DIR / "xai_integrated_gradients_region_importance_by_race.csv", index=False)\nig_region_summary.to_csv(XAI_DIR / "xai_integrated_gradients_region_importance_by_race.csv", index=False)\n\nblock_importance_gxi = feature_block_importance(grad_attr).merge(meta_xai[["Race", "Ethnicity", "Sex", "label_name"]].reset_index(names="sample_idx"), on="sample_idx", how="left")\nblock_importance_ig = feature_block_importance(ig_attr).merge(meta_ig[["Race", "Ethnicity", "Sex", "label_name"]].reset_index(names="sample_idx"), on="sample_idx", how="left")\nblock_summary_gxi = block_importance_gxi.groupby(["Race", "feature_block"]).agg(n=("importance_norm", "count"), mean_importance_norm=("importance_norm", "mean")).reset_index()\nblock_summary_ig = block_importance_ig.groupby(["Race", "feature_block"]).agg(n=("importance_norm", "count"), mean_importance_norm=("importance_norm", "mean")).reset_index()\nblock_summary_gxi["method"] = "Gradient_x_Input"\nblock_summary_ig["method"] = "Integrated_Gradients"\nblock_summary = pd.concat([block_summary_gxi, block_summary_ig], ignore_index=True)\nblock_summary.to_csv(TABLE_DIR / "xai_feature_block_importance_by_race.csv", index=False)\nblock_summary.to_csv(XAI_DIR / "xai_feature_block_importance_by_race.csv", index=False)\n\n\ndisplay(ig_region_summary.head(20))\ndisplay(block_summary.head(30))\n\nfig, ax = plt.subplots(figsize=(12, 5))\nsns.barplot(data=block_summary[block_summary["method"] == "Integrated_Gradients"], x="feature_block", y="mean_importance_norm", hue="Race", ax=ax)\nax.set_title("Integrated Gradients: importância estática vs dinâmica por Race")\nax.set_ylabel("importância normalizada média")\nplt.tight_layout()\nfig.savefig(FIG_DIR / "08b_xai_static_dynamic_importance_by_race.png", bbox_inches="tight")\nplt.show()\n\nlog_event("xai_integrated_gradients_static_dynamic", start_time=ig_start, n_ig_samples=int(len(X_ig)), ig_steps=int(IG_STEPS))\n', "<cremad-stage-xai_integrated_gradients>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## Verificação independente do Integrated Gradients com Captum

Uma amostra estratificada das explicações é recalculada pelo Captum. A execução salva similaridade cosseno e correlação de postos dos landmarks em relação à implementação manual, produzindo evidência numérica de consistência.


In [ ]:
# Etapa durável: captum_integrated_gradients_validation
_stage_required = list([XAI_DIR / "xai_captum_integrated_gradients_validation.npz", TABLE_DIR / "xai_captum_integrated_gradients_validation.csv"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'captum_integrated_gradients_validation',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'captum_integrated_gradients_validation' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\ncaptum_validation_start = time.time()\nfrom captum.attr import IntegratedGradients as CaptumIntegratedGradients\n\nCAPTUM_CACHE = XAI_DIR / "xai_captum_integrated_gradients_validation.npz"\ncaptum_n = min(int(CAPTUM_VERIFY_SAMPLES), len(X_ig))\nif file_is_valid(CAPTUM_CACHE) and "captum_integrated_gradients_validation" not in FORCE_STAGES:\n    with np.load(CAPTUM_CACHE, allow_pickle=False) as cached:\n        captum_attr = cached["captum_attr"].astype(np.float32)\n        captum_targets = cached["targets"].astype(int)\n    print(f"Validação Captum retomada: {CAPTUM_CACHE}")\nelse:\n    captum_engine = CaptumIntegratedGradients(emot_model)\n    captum_batches = []\n    captum_targets = xai_preds_ig[:captum_n].astype(int)\n    for start in tqdm(range(0, captum_n, CAPTUM_BATCH_SIZE), desc="Captum Integrated Gradients"):\n        stop = min(captum_n, start + CAPTUM_BATCH_SIZE)\n        inputs = torch.tensor(X_ig[start:stop], dtype=torch.float32, device=DEVICE, requires_grad=True)\n        baselines = torch.zeros_like(inputs)\n        targets = torch.tensor(captum_targets[start:stop], dtype=torch.long, device=DEVICE)\n        attribution = captum_engine.attribute(\n            inputs,\n            baselines=baselines,\n            target=targets,\n            n_steps=IG_STEPS,\n            internal_batch_size=max(1, min(CAPTUM_INTERNAL_BATCH_SIZE, len(inputs) * IG_STEPS)),\n        )\n        captum_batches.append(attribution.abs().detach().cpu().numpy())\n        del inputs, baselines, targets, attribution\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    captum_attr = np.concatenate(captum_batches, axis=0).astype(np.float32)\n    atomic_npz_save(CAPTUM_CACHE, captum_attr=captum_attr, targets=captum_targets)\n\nmanual_attr_comparison = ig_attr[:captum_n].astype(np.float32)\ncaptum_flat = captum_attr.reshape(captum_n, -1)\nmanual_flat = manual_attr_comparison.reshape(captum_n, -1)\ndot = (captum_flat * manual_flat).sum(axis=1)\ndenominator = np.linalg.norm(captum_flat, axis=1) * np.linalg.norm(manual_flat, axis=1)\ncosine_similarity = np.divide(dot, denominator, out=np.zeros_like(dot), where=denominator > 0)\n\ncaptum_landmarks = aggregate_attr_to_landmarks(captum_attr)\nmanual_landmarks = aggregate_attr_to_landmarks(manual_attr_comparison)\nrank_correlations = []\nfor sample_index in range(captum_n):\n    left = pd.Series(captum_landmarks[sample_index]).rank(method="average").to_numpy()\n    right = pd.Series(manual_landmarks[sample_index]).rank(method="average").to_numpy()\n    correlation = np.corrcoef(left, right)[0, 1]\n    rank_correlations.append(float(correlation) if np.isfinite(correlation) else np.nan)\n\ncaptum_validation = pd.DataFrame({\n    "sample_idx_within_ig": np.arange(captum_n),\n    "target_class": captum_targets,\n    "target_emotion": [CLASS_NAMES[value] for value in captum_targets],\n    "cosine_similarity_manual_vs_captum": cosine_similarity,\n    "spearman_landmark_rank_manual_vs_captum": rank_correlations,\n})\natomic_write_text(TABLE_DIR / "xai_captum_integrated_gradients_validation.csv", captum_validation.to_csv(index=False))\natomic_write_text(XAI_DIR / "xai_captum_integrated_gradients_validation.csv", captum_validation.to_csv(index=False))\n\ncaptum_report = f"""# Validação do Integrated Gradients com Captum\n\n- Amostras comparadas: {captum_n}\n- Passos de integração: {IG_STEPS}\n- Similaridade cosseno média entre implementação manual e Captum: {captum_validation[\'cosine_similarity_manual_vs_captum\'].mean():.4f}\n- Correlação de postos média dos landmarks: {captum_validation[\'spearman_landmark_rank_manual_vs_captum\'].mean():.4f}\n\nEsta verificação documenta correspondência numérica entre os dois motores. Diferenças residuais podem decorrer da regra de integração utilizada internamente pelo Captum.\n"""\natomic_write_text(REPORT_DIR / "xai_captum_validation.md", captum_report)\ndisplay(captum_validation.describe(include="all"))\nlog_event("captum_integrated_gradients_validation", start_time=captum_validation_start, n_samples=int(captum_n), ig_steps=int(IG_STEPS))\n', "<cremad-stage-captum_integrated_gradients_validation>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 24. Fairness explicativo: ADI e Temporal Bias Drift

Agora avaliamos disparidade nas explicações:

- **ADI:** distância entre distribuições médias de importância dos landmarks.
- **Temporal Bias Drift:** distância entre curvas temporais médias de atenção.


In [ ]:

def js_distance(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    p = p / np.maximum(p.sum(), 1e-12)
    q = q / np.maximum(q.sum(), 1e-12)
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * np.log(np.maximum(p, 1e-12) / np.maximum(m, 1e-12)))
    kl_qm = np.sum(q * np.log(np.maximum(q, 1e-12) / np.maximum(m, 1e-12)))
    return float(np.sqrt(0.5 * (kl_pm + kl_qm)))

def compute_explanatory_fairness(meta_df, attention, landmark_scores, group_col="Race", reference_group="Caucasian", min_n=10):
    rows = []
    groups = meta_df[group_col].values
    ref_mask = groups == reference_group if reference_group in set(groups) else np.ones(len(groups), dtype=bool)
    ref_attr = landmark_scores[ref_mask].mean(axis=0)
    ref_time = attention[ref_mask].mean(axis=0)
    for group in pd.Series(groups).dropna().unique():
        mask = groups == group
        if mask.sum() < min_n:
            continue
        rows.append({
            "group_col": group_col,
            "group": group,
            "n": int(mask.sum()),
            "ADI_js_distance_to_reference": js_distance(landmark_scores[mask].mean(axis=0), ref_attr),
            "Temporal_Bias_Drift_js_distance_to_reference": js_distance(attention[mask].mean(axis=0), ref_time),
            "attention_entropy_mean": float(normalized_entropy(attention[mask], axis=1).mean()),
        })
    return pd.DataFrame(rows)

xai_fairness_start = time.time()
xai_fairness_race = compute_explanatory_fairness(meta_xai, attention_curves, landmark_scores, "Race", "Caucasian")
xai_fairness_ethnicity = compute_explanatory_fairness(meta_xai, attention_curves, landmark_scores, "Ethnicity", "Not Hispanic")
xai_fairness = pd.concat([xai_fairness_race, xai_fairness_ethnicity], ignore_index=True)
xai_fairness.to_csv(TABLE_DIR / "xai_explanatory_fairness_adi_tbd.csv", index=False)
xai_fairness.to_csv(XAI_DIR / "xai_explanatory_fairness_adi_tbd.csv", index=False)
display(xai_fairness)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=xai_fairness_race, x="group", y="ADI_js_distance_to_reference", ax=axes[0], color="#4C78A8")
axes[0].set_title("ADI por Race")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=xai_fairness_race, x="group", y="Temporal_Bias_Drift_js_distance_to_reference", ax=axes[1], color="#F58518")
axes[1].set_title("Temporal Bias Drift por Race")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(FIG_DIR / "09_xai_adi_temporal_bias_drift_by_race.png", bbox_inches="tight")
plt.show()
log_event("xai_explanatory_fairness", start_time=xai_fairness_start, n_rows=int(len(xai_fairness)))


## 25. Top-k landmarks por grupo

O overlap de top-k landmarks verifica se os grupos compartilham os mesmos pontos faciais
mais importantes.


In [ ]:

def top_landmarks_for_group(meta_df, scores, group_col, group, k=TOP_K_LANDMARKS):
    mask = meta_df[group_col].values == group
    mean_scores = scores[mask].mean(axis=0)
    top_pos = np.argsort(mean_scores)[::-1][:k]
    return set([SELECTED_LANDMARK_IDS[i] for i in top_pos]), mean_scores

topk_start = time.time()
races = [r for r in meta_xai["Race"].dropna().unique() if (meta_xai["Race"] == r).sum() >= 10]
rows = []
for i, g1 in enumerate(races):
    set1, _ = top_landmarks_for_group(meta_xai, landmark_scores, "Race", g1)
    for g2 in races[i+1:]:
        set2, _ = top_landmarks_for_group(meta_xai, landmark_scores, "Race", g2)
        rows.append({"group_1": g1, "group_2": g2, "top_k": TOP_K_LANDMARKS, "jaccard_overlap": len(set1 & set2) / max(1, len(set1 | set2))})
topk_overlap = pd.DataFrame(rows)
topk_overlap.to_csv(TABLE_DIR / "xai_topk_landmark_overlap_by_race.csv", index=False)
topk_overlap.to_csv(XAI_DIR / "xai_topk_landmark_overlap_by_race.csv", index=False)
display(topk_overlap)

top_rows = []
for race in races:
    _, mean_scores = top_landmarks_for_group(meta_xai, landmark_scores, "Race", race)
    for rank, pos in enumerate(np.argsort(mean_scores)[::-1][:TOP_K_LANDMARKS], start=1):
        lm_id = SELECTED_LANDMARK_IDS[pos]
        top_rows.append({"Race": race, "rank": rank, "landmark_id": lm_id, "importance": float(mean_scores[pos]), "regions": ",".join(LANDMARK_TO_REGIONS.get(lm_id, ["other"]))})
top_landmarks_df = pd.DataFrame(top_rows)
top_landmarks_df.to_csv(TABLE_DIR / "xai_top_landmarks_by_race.csv", index=False)
top_landmarks_df.to_csv(XAI_DIR / "xai_top_landmarks_by_race.csv", index=False)
display(top_landmarks_df.head(30))
log_event("xai_topk_landmarks", start_time=topk_start, n_pairs=int(len(topk_overlap)), n_top_rows=int(len(top_landmarks_df)))


## 25.1 Oclusão por regiões faciais

A revisão propõe testar se a remoção de regiões como boca, olhos, sobrancelhas e região
inferior/superior da face degrada o desempenho de modo diferente entre grupos. Esta etapa
faz uma ablação direta nas features de landmarks: zeramos os landmarks de uma região e
medimos queda de probabilidade e acurácia.


In [ ]:
# Etapa durável: xai_region_occlusion
_stage_required = list([XAI_DIR / "xai_region_occlusion_samples.csv"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'xai_region_occlusion',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'xai_region_occlusion' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\ndef feature_indices_for_landmarks(landmark_ids):\n    selected_to_pos = {lm_id: pos for pos, lm_id in enumerate(SELECTED_LANDMARK_IDS)}\n    positions = [selected_to_pos[i] for i in landmark_ids if i in selected_to_pos]\n    feature_idx = []\n    for block_idx in range(len(FEATURE_BLOCKS)):\n        offset = block_idx * BASE_FEATURE_DIM\n        for pos in positions:\n            feature_idx.extend([offset + 2 * pos, offset + 2 * pos + 1])\n    return sorted(set(feature_idx))\n\nOCCLUSION_REGION_IDS = {\n    **REGION_IDS,\n    "upper_face": set(REGION_IDS["left_eye"]) | set(REGION_IDS["right_eye"]) | set(REGION_IDS["left_eyebrow"]) | set(REGION_IDS["right_eyebrow"]),\n    "lower_face": set(REGION_IDS["lips"]),\n    "expressive_core": set(REGION_IDS["lips"]) | set(REGION_IDS["left_eye"]) | set(REGION_IDS["right_eye"]) | set(REGION_IDS["left_eyebrow"]) | set(REGION_IDS["right_eyebrow"]),\n}\n\ndef region_occlusion_test(model, X_np, y_true, meta_df, target_classes, regions=OCCLUSION_REGION_IDS):\n    base_pred, base_prob, _ = predict_model(model, X_np, batch_size=REGION_OCCLUSION_BATCH_SIZE)\n    base_target_prob = base_prob[np.arange(len(X_np)), target_classes]\n    rows = []\n    for region, ids in tqdm(regions.items(), desc="Oclusão por região"):\n        feature_idx = feature_indices_for_landmarks(ids)\n        if not feature_idx:\n            continue\n        X_masked = X_np.copy()\n        X_masked[:, :, feature_idx] = 0.0\n        masked_pred, masked_prob, _ = predict_model(model, X_masked, batch_size=REGION_OCCLUSION_BATCH_SIZE)\n        masked_target_prob = masked_prob[np.arange(len(X_np)), target_classes]\n        for i in range(len(X_np)):\n            rows.append({\n                "sample_idx": i,\n                "region": region,\n                "n_landmarks": len(set(ids).intersection(SELECTED_LANDMARK_IDS)),\n                "base_pred": int(base_pred[i]),\n                "masked_pred": int(masked_pred[i]),\n                "true_label": int(y_true[i]),\n                "base_correct": bool(base_pred[i] == y_true[i]),\n                "masked_correct": bool(masked_pred[i] == y_true[i]),\n                "base_target_prob": float(base_target_prob[i]),\n                "masked_target_prob": float(masked_target_prob[i]),\n                "target_prob_drop": float(base_target_prob[i] - masked_target_prob[i]),\n            })\n    return pd.DataFrame(rows)\n\nocclusion_start = time.time()\nOCCLUSION_CACHE = XAI_DIR / "xai_region_occlusion_samples.csv"\nif file_is_valid(OCCLUSION_CACHE) and "xai_region_occlusion" not in FORCE_STAGES:\n    occlusion_df = pd.read_csv(OCCLUSION_CACHE)\n    print(f"Oclusão retomada do cache: {OCCLUSION_CACHE}")\nelse:\n    occlusion_df = region_occlusion_test(emot_model, X_xai, y_xai, meta_xai, xai_preds)\nif "Race" not in occlusion_df.columns:\n    occlusion_df = occlusion_df.merge(meta_xai[["Race", "Ethnicity", "Sex", "label_name"]].reset_index(names="sample_idx"), on="sample_idx", how="left")\nocclusion_summary = occlusion_df.groupby(["Race", "region"]).agg(\n    n=("sample_idx", "nunique"),\n    n_landmarks=("n_landmarks", "first"),\n    mean_target_prob_drop=("target_prob_drop", "mean"),\n    base_accuracy=("base_correct", "mean"),\n    masked_accuracy=("masked_correct", "mean"),\n).reset_index()\nocclusion_summary["accuracy_drop"] = occlusion_summary["base_accuracy"] - occlusion_summary["masked_accuracy"]\n\natomic_write_text(TABLE_DIR / "xai_region_occlusion_samples.csv", occlusion_df.to_csv(index=False))\natomic_write_text(TABLE_DIR / "xai_region_occlusion_by_race.csv", occlusion_summary.to_csv(index=False))\natomic_write_text(XAI_DIR / "xai_region_occlusion_samples.csv", occlusion_df.to_csv(index=False))\natomic_write_text(XAI_DIR / "xai_region_occlusion_by_race.csv", occlusion_summary.to_csv(index=False))\n\ndisplay(occlusion_summary.sort_values(["region", "accuracy_drop"], ascending=[True, False]).head(40))\n\nfig, ax = plt.subplots(figsize=(13, 6))\nsns.barplot(data=occlusion_summary, x="region", y="accuracy_drop", hue="Race", ax=ax)\nax.set_title("Oclusão por região facial: queda de acurácia por Race")\nax.set_ylabel("accuracy drop")\nplt.xticks(rotation=30, ha="right")\nplt.tight_layout()\nfig.savefig(FIG_DIR / "10a_xai_region_occlusion_accuracy_drop_by_race.png", bbox_inches="tight")\nplt.show()\n\nlog_event("xai_region_occlusion", start_time=occlusion_start, n_rows=int(len(occlusion_df)), n_regions=int(occlusion_summary["region"].nunique()))\n', "<cremad-stage-xai_region_occlusion>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 26. Deletion Test temporal

Removemos progressivamente os frames mais importantes segundo Attention Rollout. Se a
explicacao for fiel, a probabilidade da classe explicada deve cair rapidamente.


In [ ]:
# Etapa durável: xai_deletion_test
_stage_required = list([XAI_DIR / "xai_deletion_test_temporal.csv"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'xai_deletion_test',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'xai_deletion_test' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\n@torch.no_grad()\ndef deletion_test_temporal(model, X_np, frame_importance, target_classes, fractions=DELETION_FRACTIONS, batch_size=XAI_BATCH_SIZE):\n    _, prob_original, _ = predict_model(model, X_np, batch_size=batch_size)\n    base_prob = prob_original[np.arange(len(X_np)), target_classes]\n    rows = []\n    for frac in tqdm(fractions, desc="Deletion fractions"):\n        X_masked = X_np.copy()\n        k = int(round(frac * X_np.shape[1]))\n        if k > 0:\n            order = np.argsort(frame_importance, axis=1)[:, ::-1]\n            for i in range(len(X_masked)):\n                X_masked[i, order[i, :k], :] = 0.0\n        _, prob_masked, _ = predict_model(model, X_masked, batch_size=batch_size)\n        target_prob = prob_masked[np.arange(len(X_np)), target_classes]\n        for i in range(len(X_np)):\n            rows.append({"sample_idx": i, "fraction_deleted": frac, "k_frames_deleted": k, "base_prob": float(base_prob[i]), "masked_prob": float(target_prob[i]), "prob_drop": float(base_prob[i] - target_prob[i])})\n    return pd.DataFrame(rows)\n\ndeletion_start = time.time()\nDELETION_CACHE = XAI_DIR / "xai_deletion_test_temporal.csv"\nif file_is_valid(DELETION_CACHE) and "xai_deletion_test" not in FORCE_STAGES:\n    deletion_df = pd.read_csv(DELETION_CACHE)\n    print(f"Deletion Test retomado do cache: {DELETION_CACHE}")\nelse:\n    deletion_df = deletion_test_temporal(emot_model, X_xai, attention_curves, xai_preds)\nif "Race" not in deletion_df.columns:\n    deletion_df = deletion_df.merge(meta_xai[["Race", "Ethnicity", "Sex", "label_name"]].reset_index(names="sample_idx"), on="sample_idx", how="left")\natomic_write_text(TABLE_DIR / "xai_deletion_test_temporal.csv", deletion_df.to_csv(index=False))\natomic_write_text(XAI_DIR / "xai_deletion_test_temporal.csv", deletion_df.to_csv(index=False))\ndisplay(deletion_df.head())\n\ndef auc_deletion_for_group(df):\n    curve = df.groupby("fraction_deleted")["masked_prob"].mean().reset_index().sort_values("fraction_deleted")\n    return float(np.trapz(curve["masked_prob"].values, curve["fraction_deleted"].values))\n\nauc_rows = []\nfor race, g in deletion_df.groupby("Race"):\n    if g["sample_idx"].nunique() >= 10:\n        auc_rows.append({"Race": race, "n_samples": g["sample_idx"].nunique(), "deletion_auc_masked_prob": auc_deletion_for_group(g), "mean_aopc_drop": g.groupby("sample_idx")["prob_drop"].mean().mean()})\ndeletion_auc_by_race = pd.DataFrame(auc_rows)\natomic_write_text(TABLE_DIR / "xai_deletion_auc_by_race.csv", deletion_auc_by_race.to_csv(index=False))\natomic_write_text(XAI_DIR / "xai_deletion_auc_by_race.csv", deletion_auc_by_race.to_csv(index=False))\ndisplay(deletion_auc_by_race)\n\ndelete_summary = deletion_df.groupby(["Race", "fraction_deleted"]).agg(mean_masked_prob=("masked_prob", "mean"), n=("masked_prob", "count")).reset_index()\nfig, ax = plt.subplots(figsize=(10, 5))\nsns.lineplot(data=delete_summary, x="fraction_deleted", y="mean_masked_prob", hue="Race", marker="o", ax=ax)\nax.set_title("Deletion Test temporal por Race")\nax.set_xlabel("fração de frames removidos")\nax.set_ylabel("probabilidade media da classe explicada")\nplt.tight_layout()\nfig.savefig(FIG_DIR / "10_xai_deletion_curves_by_race.png", bbox_inches="tight")\nplt.show()\nlog_event("xai_deletion_test", start_time=deletion_start, n_rows=int(len(deletion_df)), n_auc_rows=int(len(deletion_auc_by_race)))\n', "<cremad-stage-xai_deletion_test>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


## 26.1 Correlação exploratória entre XAI e fairness

A revisão bibliográfica sugere testar se indicadores de explicabilidade ajudam a antecipar
ou explicar disparidades. Aqui agregamos, por `Race`, métricas preditivas e métricas XAI.

Importante: com poucos grupos demográficos, esta análise é **exploratória**. Correlação não
prova causalidade; ela apenas orienta interpretação e hipóteses para o texto.


In [ ]:

correlation_start = time.time()

emot_race_metrics = fairness_by_group[
    (fairness_by_group["model"] == "EmoT") & (fairness_by_group["protected_attribute"] == "Race")
].copy()
emot_race_metrics = emot_race_metrics.rename(columns={"group": "Race"})

attention_group = meta_xai.groupby("Race").agg(
    xai_attention_entropy_mean=("attention_entropy", "mean"),
    xai_attention_entropy_std=("attention_entropy", "std"),
    xai_n=("file_stem", "count"),
).reset_index()

adi_group = xai_fairness_race.rename(columns={
    "group": "Race",
    "ADI_js_distance_to_reference": "xai_ADI",
    "Temporal_Bias_Drift_js_distance_to_reference": "xai_TBD",
    "attention_entropy_mean": "xai_attention_entropy_group_mean",
})

deletion_group = deletion_auc_by_race.rename(columns={
    "deletion_auc_masked_prob": "xai_deletion_auc",
    "mean_aopc_drop": "xai_mean_aopc_drop",
})

lower_occ = occlusion_summary[occlusion_summary["region"] == "lower_face"][
    ["Race", "accuracy_drop", "mean_target_prob_drop"]
].rename(columns={
    "accuracy_drop": "xai_lower_face_accuracy_drop",
    "mean_target_prob_drop": "xai_lower_face_prob_drop",
})

block_ig = block_summary[block_summary["method"] == "Integrated_Gradients"].pivot_table(
    index="Race",
    columns="feature_block",
    values="mean_importance_norm",
    aggfunc="mean",
).reset_index()
for col in FEATURE_BLOCKS:
    if col not in block_ig.columns:
        block_ig[col] = 0.0
dynamic_cols = [c for c in ["vel", "acc", "jerk"] if c in block_ig.columns]
block_ig["xai_static_pos_importance"] = block_ig.get("pos", 0.0)
block_ig["xai_dynamic_importance"] = block_ig[dynamic_cols].sum(axis=1) if dynamic_cols else 0.0
block_ig = block_ig[["Race", "xai_static_pos_importance", "xai_dynamic_importance"]]

xai_fairness_group_profile = emot_race_metrics.merge(attention_group, on="Race", how="left")
xai_fairness_group_profile = xai_fairness_group_profile.merge(adi_group[["Race", "xai_ADI", "xai_TBD"]], on="Race", how="left")
xai_fairness_group_profile = xai_fairness_group_profile.merge(deletion_group[["Race", "xai_deletion_auc", "xai_mean_aopc_drop"]], on="Race", how="left")
xai_fairness_group_profile = xai_fairness_group_profile.merge(lower_occ, on="Race", how="left")
xai_fairness_group_profile = xai_fairness_group_profile.merge(block_ig, on="Race", how="left")
xai_fairness_group_profile.to_csv(TABLE_DIR / "xai_fairness_group_profile_by_race.csv", index=False)
xai_fairness_group_profile.to_csv(XAI_DIR / "xai_fairness_group_profile_by_race.csv", index=False)
display(xai_fairness_group_profile)

fairness_metrics_for_corr = [c for c in ["accuracy", "macro_f1", "ece"] if c in xai_fairness_group_profile.columns]
xai_metrics_for_corr = [
    "xai_attention_entropy_mean",
    "xai_ADI",
    "xai_TBD",
    "xai_deletion_auc",
    "xai_mean_aopc_drop",
    "xai_lower_face_accuracy_drop",
    "xai_lower_face_prob_drop",
    "xai_static_pos_importance",
    "xai_dynamic_importance",
]
xai_metrics_for_corr = [c for c in xai_metrics_for_corr if c in xai_fairness_group_profile.columns]

try:
    from scipy.stats import pearsonr, spearmanr
except Exception:
    pearsonr = spearmanr = None

corr_rows = []
for fair_col in fairness_metrics_for_corr:
    for xai_col in xai_metrics_for_corr:
        tmp = xai_fairness_group_profile[[fair_col, xai_col]].dropna()
        row = {"fairness_metric": fair_col, "xai_metric": xai_col, "n_groups": int(len(tmp))}
        if len(tmp) >= 3 and tmp[fair_col].nunique() > 1 and tmp[xai_col].nunique() > 1:
            if pearsonr is not None:
                pr = pearsonr(tmp[fair_col], tmp[xai_col])
                sr = spearmanr(tmp[fair_col], tmp[xai_col])
                row.update({
                    "pearson_r": float(pr.statistic),
                    "pearson_p": float(pr.pvalue),
                    "spearman_rho": float(sr.statistic),
                    "spearman_p": float(sr.pvalue),
                })
            else:
                row.update({
                    "pearson_r": float(tmp[fair_col].corr(tmp[xai_col], method="pearson")),
                    "pearson_p": np.nan,
                    "spearman_rho": float(tmp[fair_col].corr(tmp[xai_col], method="spearman")),
                    "spearman_p": np.nan,
                })
        else:
            row.update({"pearson_r": np.nan, "pearson_p": np.nan, "spearman_rho": np.nan, "spearman_p": np.nan})
        corr_rows.append(row)

xai_fairness_correlation = pd.DataFrame(corr_rows)
xai_fairness_correlation.to_csv(TABLE_DIR / "xai_fairness_correlation_by_race.csv", index=False)
xai_fairness_correlation.to_csv(XAI_DIR / "xai_fairness_correlation_by_race.csv", index=False)
display(xai_fairness_correlation.sort_values("spearman_rho", key=lambda s: s.abs(), ascending=False).head(20))

if "ece" in xai_fairness_group_profile.columns and "xai_attention_entropy_mean" in xai_fairness_group_profile.columns:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(data=xai_fairness_group_profile, x="xai_attention_entropy_mean", y="ece", hue="Race", s=90, ax=ax)
    for _, row in xai_fairness_group_profile.iterrows():
        ax.text(row["xai_attention_entropy_mean"], row["ece"], str(row["Race"]), fontsize=8)
    ax.set_title("Correlação exploratória: entropia de atenção vs ECE")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "10b_xai_fairness_correlation_entropy_ece.png", bbox_inches="tight")
    plt.show()

log_event("xai_fairness_correlation", start_time=correlation_start, n_rows=int(len(xai_fairness_correlation)))


## 27. Robustez XAI em alto consenso facial

Repetimos ADI e Temporal Bias Drift no subconjunto `face_agreement >= 0.70`.


In [ ]:

xai_robustness_start = time.time()
xai_high_mask = meta_xai["face_agreement"].astype(float).values >= HIGH_CONSENSUS_THRESHOLD
print(f"Amostras XAI totais: {len(meta_xai)}")
print(f"Amostras XAI de alto consenso: {xai_high_mask.sum()}")
if xai_high_mask.sum() >= 30:
    xai_fair_high = compute_explanatory_fairness(
        meta_xai.loc[xai_high_mask].reset_index(drop=True),
        attention_curves[xai_high_mask],
        landmark_scores[xai_high_mask],
        "Race",
        "Caucasian",
        min_n=5,
    )
    xai_fair_high["subset"] = "high_face_consensus"
    xai_fair_full = xai_fairness_race.copy()
    xai_fair_full["subset"] = "full_xai_test"
    xai_robustness = pd.concat([xai_fair_full, xai_fair_high], ignore_index=True)
    xai_robustness.to_csv(TABLE_DIR / "xai_robustness_high_consensus.csv", index=False)
    xai_robustness.to_csv(XAI_DIR / "xai_robustness_high_consensus.csv", index=False)
    display(xai_robustness)
else:
    print("Subconjunto pequeno demais para robustez XAI.")
log_event("xai_high_consensus_robustness", start_time=xai_robustness_start, n_high_consensus=int(xai_high_mask.sum()), n_xai=int(len(meta_xai)))


## 28. Síntese automática

Esta célula grava um resumo curto para apoiar a escrita da dissertação/artigo.


In [ ]:

best_model_row = global_results.iloc[0].to_dict()
emot_global = global_results[global_results["model"] == "EmoT"].iloc[0].to_dict()
emot_race_fair = fairness_summary[(fairness_summary["model"] == "EmoT") & (fairness_summary["protected_attribute"] == "Race")]

lines = [
    f"Melhor modelo por macro-F1: {best_model_row['model']} (macro-F1={best_model_row['macro_f1']:.3f}).",
    f"EmoT: accuracy={emot_global['accuracy']:.3f}, balanced_accuracy={emot_global['balanced_accuracy']:.3f}, macro-F1={emot_global['macro_f1']:.3f}.",
]
if len(emot_race_fair):
    row = emot_race_fair.iloc[0]
    lines.append(
        f"Fairness preditivo do EmoT por Race: worst-group accuracy={row['worst_group_accuracy']:.3f}, "
        f"performance gap={row['performance_gap_accuracy']:.3f}, equal opportunity gap médio={row['equal_opportunity_gap_mean']:.3f}, "
        f"p_perm={row['permutation_p_value']:.3f}."
    )
if len(xai_fairness_race):
    lines.append(
        f"Fairness explicativo por Race: ADI maximo={xai_fairness_race['ADI_js_distance_to_reference'].max():.3f}, "
        f"Temporal Bias Drift maximo={xai_fairness_race['Temporal_Bias_Drift_js_distance_to_reference'].max():.3f}."
    )

summary_text = "\n".join(f"- {line}" for line in lines)
print(summary_text)
summary_path = REPORT_DIR / "summary_results.md"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("# Síntese dos resultados\n\n")
    f.write(summary_text + "\n")
print(f"Síntese salva em: {summary_path}")


## Síntese e índice dos artefatos definitivos

Este índice aponta os resultados que serão usados na redação posterior. A redação da dissertação e qualquer mudança no Overleaf continuam condicionadas à validação do usuário.


In [ ]:

definitive_summary_lines = [
    "# Índice da execução definitiva",
    "",
    f"- Run ID: `{RUN_ID}`",
    f"- Base válida: {len(meta_valid):,} vídeos; execução integral: {FULL_RUN}.",
    f"- Protocolo principal de robustez: {CV_N_SPLITS}-fold por ator, estratificação `{cv_stratification}`.",
    f"- Melhor modelo no CV por macro-F1 médio: {cv_summary.iloc[0]['model']} ({cv_summary.iloc[0]['macro_f1_mean']:.4f} ± {cv_summary.iloc[0]['macro_f1_std']:.4f}).",
    "- Modalidade: exclusivamente visual; nenhuma feature de áudio foi utilizada.",
    "- Fairness: Race, Ethnicity, Sex, age_bin e análises interseccionais.",
    "- XAI: Attention Rollout, Gradient x Input, Integrated Gradients, Captum, oclusão regional e Deletion Test.",
    "- O arquivo `cv_observations_to_discuss.md` contém achados descritivos; eles não devem ser apresentados como causalidade.",
    "- O arquivo `literature_visual_only_comparison_template.csv` exige validação documental de cada comparador externo.",
    "- Nenhum conteúdo foi escrito no Overleaf.",
]
atomic_write_text(REPORT_DIR / "definitive_execution_index.md", "\n".join(definitive_summary_lines) + "\n")
print("\n".join(definitive_summary_lines))


## 29. Logs de tempo, GPU e processamento

Esta célula consolida a rastreabilidade computacional: quanto tempo cada etapa levou,
quanto de GPU/VRAM estava sendo usado e quais arquivos de log foram salvos.


In [ ]:

show_gpu_snapshot("fim_experimento")

resource_log = pd.DataFrame(RUN_EVENTS)
resource_log_path = LOG_DIR / f"resource_usage_log_{RUN_ID}.csv"
resource_log_latest = LOG_DIR / "resource_usage_log_latest.csv"
resource_log.to_csv(resource_log_path, index=False)
resource_log.to_csv(resource_log_latest, index=False)

timing_summary = resource_log.dropna(subset=["duration_sec"]).copy() if "duration_sec" in resource_log.columns else pd.DataFrame()
if len(timing_summary):
    timing_summary = timing_summary.sort_values("duration_sec", ascending=False)
    timing_summary.to_csv(TABLE_DIR / "processing_time_by_stage.csv", index=False)
    timing_summary.to_csv(LOG_DIR / "processing_time_by_stage.csv", index=False)
    display_cols = [
        "stage",
        "status",
        "duration_sec",
        "duration_min",
        "gpu_name",
        "nvidia_gpu_utilization_percent",
        "nvidia_memory_used_mb",
        "gpu_memory_allocated_mb",
        "gpu_memory_reserved_mb",
    ]
    display_cols = [c for c in display_cols if c in timing_summary.columns]
    display(timing_summary[display_cols].head(30))

    fig, ax = plt.subplots(figsize=(12, max(5, 0.35 * min(25, len(timing_summary)))))
    plot_time = timing_summary.head(25).sort_values("duration_sec")
    sns.barplot(data=plot_time, x="duration_sec", y="stage", ax=ax, color="#4C78A8")
    ax.set_title("Tempo de processamento por etapa")
    ax.set_xlabel("segundos")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "11_processing_time_by_stage.png", bbox_inches="tight")
    plt.show()
else:
    print("Nenhuma etapa com duração registrada.")

gpu_cols = ["nvidia_gpu_utilization_percent", "nvidia_memory_used_mb", "nvidia_power_draw_w", "nvidia_temperature_c"]
available_gpu_cols = [c for c in gpu_cols if c in resource_log.columns and resource_log[c].notna().any()]
if available_gpu_cols:
    gpu_timeline = resource_log[["stage"] + available_gpu_cols].copy()
    gpu_timeline.to_csv(LOG_DIR / "gpu_usage_timeline.csv", index=False)
    display(gpu_timeline.tail(20))

    fig, axes = plt.subplots(len(available_gpu_cols), 1, figsize=(13, 2.8 * len(available_gpu_cols)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, available_gpu_cols):
        ax.plot(np.arange(len(resource_log)), resource_log[col], marker="o")
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("evento registrado")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "12_gpu_usage_timeline.png", bbox_inches="tight")
    plt.show()
else:
    print("Sem métricas detalhadas do nvidia-smi. O log ainda contém memória PyTorch quando CUDA está disponível.")

print(f"Log de recursos salvo em: {resource_log_path}")
print(f"Resumo de tempos salvo em: {TABLE_DIR / 'processing_time_by_stage.csv'}")


## 30. Artefatos gerados

Principais saídas:

- `outputs/tables/global_test_metrics.csv`
- `outputs/tables/fairness_metrics_by_group.csv`
- `outputs/tables/fairness_summary.csv`
- `outputs/tables/fairness_intersectional_metrics.csv`
- `outputs/tables/robustness_high_consensus.csv`
- `outputs/tables/xai_explanatory_fairness_adi_tbd.csv`
- `outputs/tables/xai_deletion_test_temporal.csv`
- `outputs/tables/xai_integrated_gradients_region_importance_by_race.csv`
- `outputs/tables/xai_feature_block_importance_by_race.csv`
- `outputs/tables/xai_region_occlusion_by_race.csv`
- `outputs/tables/xai_fairness_correlation_by_race.csv`
- `outputs/tables/visual_sample_manifest.csv`
- `outputs/tables/processing_time_by_stage.csv`
- `outputs/figures/*.png`
- `outputs/figures/02_visual_sample_frames_by_race.png`
- `outputs/figures/02_visual_preprocessing_pipeline.png`
- `outputs/figures/06b_fairness_intersectional_gaps_emot.png`
- `outputs/figures/08b_xai_static_dynamic_importance_by_race.png`
- `outputs/figures/10a_xai_region_occlusion_accuracy_drop_by_race.png`
- `outputs/figures/10b_xai_fairness_correlation_entropy_ece.png`
- `outputs/figures/11_processing_time_by_stage.png`
- `outputs/figures/12_gpu_usage_timeline.png`
- `outputs/models/best_*.pt`
- `outputs/fairness/*.csv`
- `outputs/xai/*.csv` e `outputs/xai/*.npz`
- `outputs/logs/*.json` e `outputs/logs/*.csv`
- `outputs/reports/summary_results.md`

Leitura recomendada para a dissertação:

1. estatistica descritiva do CREMA-D;
2. qualidade de processamento facial por grupo;
3. comparação dos baselines e EmoT;
4. fairness preditivo por `Race`;
5. XAI temporal e por landmarks;
6. fairness explicativo;
7. robustez com alto consenso facial.


## 31. Download de todos os artefatos

Esta célula compacta os artefatos gerados e inicia o download no Colab.

Por padrão, o `.zip` inclui:

- `outputs/`: tabelas, figuras, modelos, logs, relatórios, fairness e XAI;
- `cache/`: tensores processados, landmarks extraídos, split/normalização e modelo `.task` do MediaPipe, quando existir.

Os vídeos brutos ficam fora por padrão, porque podem deixar o arquivo muito grande e podem ser baixados novamente do CREMA-D. Se quiser incluir também os vídeos, altere `INCLUDE_RAW_VIDEOS_IN_ZIP = True`.


In [ ]:
# Etapa durável: package_artifacts
_stage_required = list([ZIP_DIR / f"cremad_fairness_xai_artifacts_{RUN_ID}.zip"])
with tracked_stage(
    LEDGER,
    RUN_ID,
    'package_artifacts',
    required_artifacts=_stage_required,
    metadata={"resume_mode": RESUME_MODE, "forced": 'package_artifacts' in FORCE_STAGES},
    log_dir=LOG_DIR,
    progress_dir=PROGRESS_DIR,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    resource_probe=current_gpu_stats,
):
    exec(compile('\nimport sqlite3\nimport zipfile\n\nINCLUDE_OUTPUTS_IN_ZIP = True\nINCLUDE_CACHE_IN_ZIP = True\nINCLUDE_RAW_VIDEOS_IN_ZIP = False\n\n# Marca provisoriamente como concluída para que o snapshot registre o desfecho.\n# Se o empacotamento falhar, tracked_stage reverte o run para "failed".\nLEDGER.mark_run(RUN_ID, "completed")\nstate_snapshot = REPORT_DIR / "experiment_state_snapshot.sqlite"\nstate_snapshot_tmp = state_snapshot.with_name(state_snapshot.name + f".tmp-{os.getpid()}")\nwith contextlib.closing(sqlite3.connect(STATE_DB)) as source_db, contextlib.closing(sqlite3.connect(state_snapshot_tmp)) as target_db:\n    source_db.backup(target_db)\n    target_db.commit()\nos.replace(state_snapshot_tmp, state_snapshot)\n\nartifact_zip = ZIP_DIR / f"cremad_fairness_xai_artifacts_{RUN_ID}.zip"\nmanifest_path = ZIP_DIR / f"download_manifest_{RUN_ID}.csv"\n\ndef should_skip_artifact(path):\n    path = Path(path)\n    if path == LOG_DIR / "stage_logs" / "package_artifacts.log":\n        return True\n    try:\n        path.resolve().relative_to(ZIP_DIR.resolve())\n        return True\n    except ValueError:\n        return path.name.endswith(".tmp") or ".tmp-" in path.name\n\nartifact_sources = []\nif INCLUDE_OUTPUTS_IN_ZIP and OUTPUT_DIR.exists():\n    artifact_sources.append(("outputs", OUTPUT_DIR))\nif INCLUDE_CACHE_IN_ZIP and CACHE_DIR.exists():\n    artifact_sources.append(("cache", CACHE_DIR))\nif INCLUDE_RAW_VIDEOS_IN_ZIP and VIDEO_DIR.exists():\n    artifact_sources.append(("raw/videos_flv", VIDEO_DIR))\n\nmanifest_rows = []\nfor source_name, source_dir in artifact_sources:\n    for path in sorted(source_dir.rglob("*")):\n        if path.is_file() and not should_skip_artifact(path):\n            manifest_rows.append({\n                "source": source_name,\n                "relative_path": str(Path(source_name) / path.relative_to(source_dir)),\n                "bytes": int(path.stat().st_size),\n                "sha256": sha256_file(path),\n                "absolute_path": str(path),\n            })\n\ndownload_manifest = pd.DataFrame(manifest_rows)\natomic_write_text(manifest_path, download_manifest.to_csv(index=False))\natomic_write_text(TABLE_DIR / "download_manifest_latest.csv", download_manifest.to_csv(index=False))\n\nzip_tmp = artifact_zip.with_name(artifact_zip.name + f".tmp-{os.getpid()}")\ntry:\n    with zipfile.ZipFile(zip_tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:\n        archive.write(manifest_path, arcname="outputs/download_manifest.csv")\n        for row in manifest_rows:\n            archive.write(row["absolute_path"], arcname=row["relative_path"])\n    os.replace(zip_tmp, artifact_zip)\nfinally:\n    if zip_tmp.exists():\n        zip_tmp.unlink()\n\nLEDGER.register_artifact(RUN_ID, "package_artifacts", manifest_path, kind="manifest")\nLEDGER.register_artifact(RUN_ID, "package_artifacts", state_snapshot, kind="sqlite_snapshot")\nLEDGER.register_artifact(RUN_ID, "package_artifacts", artifact_zip, kind="zip")\nprint(f"Arquivo criado: {artifact_zip}")\nprint(f"Tamanho: {artifact_zip.stat().st_size / (1024 ** 2):.1f} MB")\nprint(f"Arquivos incluídos: {len(download_manifest):,}")\ndisplay(download_manifest.sort_values("bytes", ascending=False).head(20))\n\nif IN_COLAB and AUTO_DOWNLOAD_FINAL_ZIP:\n    from google.colab import files\n    files.download(str(artifact_zip))\nelse:\n    print("Download automático desativado; o ZIP permanece persistido no Google Drive.")\n', "<cremad-stage-package_artifacts>", "exec"), globals())
print_stage_dashboard(LEDGER, RUN_ID)


In [ ]:
LEDGER.mark_run(RUN_ID, 'completed')
atomic_write_json(ACTIVE_RUN_JSON, {
    'run_id': RUN_ID, 'config_hash': CONFIG_HASH, 'status': 'completed',
    'updated_at': utc_now(),
})
atomic_write_json(STATE_DIR / f'completed_{RUN_ID}.json', {
    'run_id': RUN_ID, 'status': 'completed', 'completed_at': utc_now(),
    'run_dir': str(RUN_DIR), 'artifact_zip': str(artifact_zip),
})
save_progress_snapshot(LEDGER, RUN_ID, PROGRESS_DIR, 'run', 'completed')
print_stage_dashboard(LEDGER, RUN_ID)
print(f'EXECUÇÃO CONCLUÍDA E PERSISTIDA: {RUN_ID}')
print(f'Artefatos: {RUN_DIR}')
